In [1]:
# Import libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Set random seed untuk reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")

Libraries imported successfully!
NumPy version: 2.2.6
PyTorch version: 2.9.1+cpu


---

# SHEET 1: Raw Data & Configuration

Tahapan ini berisi:
1. Konfigurasi model TimeMixer
2. Generate synthetic data sederhana
3. Setup untuk perhitungan manual

---

## 1.1 Model Configuration

In [2]:
# ============================================
# MODEL CONFIGURATION PARAMETERS
# ============================================

# Data dimensions
n_timesteps = 12      # Total data points available
n_features = 2        # Number of features/channels (Feature_1, Feature_2)
batch_size = 1        # Satu sample untuk manual calculation

# Sequence configuration
seq_len = 8          # Input sequence length (historical window)
label_len = 2        # Decoder context length
pred_len = 4         # Prediction length (future to predict)

# Model architecture
d_model = 4          # Hidden dimension (embedding size)
d_ff = 8             # Feed-forward dimension (2x d_model)
e_layers = 1         # Number of PDM blocks
dropout = 0.0        # No dropout untuk manual calculation

# Multi-scale configuration
down_sampling_layers = 1     # Number of downsampling layers
down_sampling_window = 2     # Downsampling factor
down_sampling_method = 'avg' # avg pooling

# Decomposition configuration
decomp_method = 'moving_avg' # moving_avg decomposition
moving_avg = 3               # Moving average window size

# Channel configuration
channel_independence = False # Multivariate (semua channel diproses bersama)

# Time features
n_time_features = 2  # Sederhana: hour_sin, hour_cos

# Create config dictionary
config = {
    'n_timesteps': n_timesteps,
    'n_features': n_features,
    'batch_size': batch_size,
    'seq_len': seq_len,
    'label_len': label_len,
    'pred_len': pred_len,
    'd_model': d_model,
    'd_ff': d_ff,
    'e_layers': e_layers,
    'dropout': dropout,
    'down_sampling_layers': down_sampling_layers,
    'down_sampling_window': down_sampling_window,
    'down_sampling_method': down_sampling_method,
    'decomp_method': decomp_method,
    'moving_avg': moving_avg,
    'channel_independence': channel_independence,
    'n_time_features': n_time_features
}

print("="*60)
print("MODEL CONFIGURATION")
print("="*60)
for key, value in config.items():
    print(f"{key:25s}: {value}")
print("="*60)

MODEL CONFIGURATION
n_timesteps              : 12
n_features               : 2
batch_size               : 1
seq_len                  : 8
label_len                : 2
pred_len                 : 4
d_model                  : 4
d_ff                     : 8
e_layers                 : 1
dropout                  : 0.0
down_sampling_layers     : 1
down_sampling_window     : 2
down_sampling_method     : avg
decomp_method            : moving_avg
moving_avg               : 3
channel_independence     : False
n_time_features          : 2


## 1.2 Generate Synthetic Raw Data

Data synthetic sederhana dengan 2 features untuk 12 timesteps.
Nilai dibuat agak terstruktur agar mudah di-track perhitungannya.

In [3]:
# ============================================
# GENERATE SYNTHETIC RAW DATA
# ============================================

# Generate simple synthetic data dengan pola yang jelas
# Feature 1: Linear trend + small noise
# Feature 2: Sinusoidal pattern + small noise

timesteps = np.arange(n_timesteps)

# Feature 1: Linear trend
feature_1 = 10 + 2 * timesteps + np.array([0.1, -0.2, 0.15, -0.1, 0.2, -0.15, 0.1, 0.0, -0.1, 0.2, -0.1, 0.15])

# Feature 2: Sinusoidal pattern
feature_2 = 20 + 5 * np.sin(2 * np.pi * timesteps / 6) + np.array([0.2, -0.1, 0.0, 0.1, -0.2, 0.15, -0.1, 0.2, 0.0, -0.15, 0.1, -0.05])

# Combine into raw data matrix
raw_data = np.column_stack([feature_1, feature_2])

print("Raw data shape:", raw_data.shape)  # (12, 2)
print("\n" + "="*60)
print("RAW DATA GENERATED")
print("="*60)

Raw data shape: (12, 2)

RAW DATA GENERATED


## 1.3 Display Raw Data for Excel

Format untuk copy-paste ke Excel Sheet 1

In [4]:
# ============================================
# EXCEL SHEET 1: RAW DATA
# ============================================

# Create DataFrame untuk raw data
df_raw = pd.DataFrame(
    raw_data,
    columns=['Feature_1', 'Feature_2'],
    index=[f't_{i}' for i in range(n_timesteps)]
)
df_raw.index.name = 'Timestep'

print("\n📊 COPY THIS TO EXCEL - Sheet 1: Raw Data")
print("="*60)
print(df_raw.to_string())
print("="*60)

# Display dengan format yang lebih detail
print("\n📋 DETAILED VIEW (with more decimals)")
print("="*60)
pd.set_option('display.float_format', '{:.6f}'.format)
print(df_raw)
pd.reset_option('display.float_format')
print("="*60)


📊 COPY THIS TO EXCEL - Sheet 1: Raw Data
          Feature_1  Feature_2
Timestep                      
t_0           10.10  20.200000
t_1           11.80  24.230127
t_2           14.15  24.330127
t_3           15.90  20.100000
t_4           18.20  15.469873
t_5           19.85  15.819873
t_6           22.10  19.900000
t_7           24.00  24.530127
t_8           25.90  24.330127
t_9           28.20  19.850000
t_10          29.90  15.769873
t_11          32.15  15.619873

📋 DETAILED VIEW (with more decimals)
          Feature_1  Feature_2
Timestep                      
t_0       10.100000  20.200000
t_1       11.800000  24.230127
t_2       14.150000  24.330127
t_3       15.900000  20.100000
t_4       18.200000  15.469873
t_5       19.850000  15.819873
t_6       22.100000  19.900000
t_7       24.000000  24.530127
t_8       25.900000  24.330127
t_9       28.200000  19.850000
t_10      29.900000  15.769873
t_11      32.150000  15.619873


## 1.4 Configuration Summary Table for Excel

In [5]:
# ============================================
# EXCEL SHEET 1: CONFIGURATION TABLE
# ============================================

# Create configuration DataFrame
df_config = pd.DataFrame([
    ['Data Configuration', '', ''],
    ['Total Timesteps', n_timesteps, 'Total data points available'],
    ['Number of Features', n_features, 'Number of variables/channels'],
    ['Batch Size', batch_size, 'Samples per batch'],
    ['', '', ''],
    ['Sequence Configuration', '', ''],
    ['Input Length (seq_len)', seq_len, 'Historical window size'],
    ['Label Length (label_len)', label_len, 'Decoder context length'],
    ['Prediction Length (pred_len)', pred_len, 'Future prediction window'],
    ['', '', ''],
    ['Model Architecture', '', ''],
    ['Hidden Dimension (d_model)', d_model, 'Embedding dimension'],
    ['Feed-forward Dimension (d_ff)', d_ff, 'FFN hidden size'],
    ['Number of Layers (e_layers)', e_layers, 'PDM blocks count'],
    ['', '', ''],
    ['Multi-Scale Configuration', '', ''],
    ['Downsampling Layers', down_sampling_layers, 'Number of scales'],
    ['Downsampling Window', down_sampling_window, 'Downsampling factor'],
    ['Downsampling Method', down_sampling_method, 'Pooling method'],
    ['', '', ''],
    ['Decomposition Configuration', '', ''],
    ['Decomposition Method', decomp_method, 'Series decomposition'],
    ['Moving Average Window', moving_avg, 'MA window size'],
    ['', '', ''],
    ['Other Configuration', '', ''],
    ['Channel Independence', channel_independence, 'Multivariate mode'],
    ['Time Features Count', n_time_features, 'Temporal encoding dims'],
], columns=['Parameter', 'Value', 'Description'])

print("\n📊 COPY THIS TO EXCEL - Sheet 1: Configuration")
print("="*80)
print(df_config.to_string(index=False))
print("="*80)


📊 COPY THIS TO EXCEL - Sheet 1: Configuration
                    Parameter      Value                  Description
           Data Configuration                                        
              Total Timesteps         12  Total data points available
           Number of Features          2 Number of variables/channels
                   Batch Size          1            Samples per batch
                                                                     
       Sequence Configuration                                        
       Input Length (seq_len)          8       Historical window size
     Label Length (label_len)          2       Decoder context length
 Prediction Length (pred_len)          4     Future prediction window
                                                                     
           Model Architecture                                        
   Hidden Dimension (d_model)          4          Embedding dimension
Feed-forward Dimension (d_ff)          8   

## 1.5 Data Shapes Overview

Ringkasan dimensi data di setiap tahapan (untuk referensi)

In [6]:
# ============================================
# DATA SHAPES OVERVIEW
# ============================================

# Calculate expected shapes at each stage
shapes_info = [
    ['Stage', 'Data Name', 'Shape', 'Description'],
    ['', '', '', ''],
    ['Raw Data', 'raw_data', f'({n_timesteps}, {n_features})', 'Original time series'],
    ['', '', '', ''],
    ['Input Batch', 'batch_x', f'({batch_size}, {seq_len}, {n_features})', 'Encoder input'],
    ['Input Batch', 'batch_x_mark', f'({batch_size}, {seq_len}, {n_time_features})', 'Time features input'],
    ['Target Batch', 'batch_y', f'({batch_size}, {label_len + pred_len}, {n_features})', 'Decoder target'],
    ['Target Batch', 'batch_y_mark', f'({batch_size}, {label_len + pred_len}, {n_time_features})', 'Time features target'],
    ['', '', '', ''],
    ['Multi-Scale', 'x_scale_0 (original)', f'({batch_size}, {seq_len}, {n_features})', 'Original scale'],
    ['Multi-Scale', 'x_scale_1 (down)', f'({batch_size}, {seq_len//down_sampling_window}, {n_features})', 'Downsampled scale'],
    ['', '', '', ''],
    ['After Embedding', 'enc_out_scale_0', f'({batch_size}, {seq_len}, {d_model})', 'Embedded original'],
    ['After Embedding', 'enc_out_scale_1', f'({batch_size}, {seq_len//down_sampling_window}, {d_model})', 'Embedded downsampled'],
    ['', '', '', ''],
    ['Final Output', 'prediction', f'({batch_size}, {pred_len}, {n_features})', 'Model prediction'],
]

df_shapes = pd.DataFrame(shapes_info[1:], columns=shapes_info[0])

print("\n📊 DATA SHAPES AT EACH STAGE")
print("="*80)
print(df_shapes.to_string(index=False))
print("="*80)


📊 DATA SHAPES AT EACH STAGE
          Stage            Data Name     Shape          Description
                                                                   
       Raw Data             raw_data   (12, 2) Original time series
                                                                   
    Input Batch              batch_x (1, 8, 2)        Encoder input
    Input Batch         batch_x_mark (1, 8, 2)  Time features input
   Target Batch              batch_y (1, 6, 2)       Decoder target
   Target Batch         batch_y_mark (1, 6, 2) Time features target
                                                                   
    Multi-Scale x_scale_0 (original) (1, 8, 2)       Original scale
    Multi-Scale     x_scale_1 (down) (1, 4, 2)    Downsampled scale
                                                                   
After Embedding      enc_out_scale_0 (1, 8, 4)    Embedded original
After Embedding      enc_out_scale_1 (1, 4, 4) Embedded downsampled
                   

## 1.6 Summary for Sheet 1

**Status: ✅ Sheet 1 Complete**

### What we have now:
1. ✅ Model configuration parameters
2. ✅ Synthetic raw data (12 timesteps × 2 features)
3. ✅ Data ready for Excel export

### Next Steps:
- **Sheet 2**: Data Normalization (StandardScaler)
- **Sheet 3**: Time Features Extraction
- **Sheet 4**: Multi-Scale Downsampling
- ... and so on

---

**⏸️ PAUSE HERE - Waiting for confirmation before proceeding to Sheet 2**

In [7]:
# Save variables for next sheets
print("\n✅ Sheet 1 Variables Saved:")
print(f"   - config: {len(config)} parameters")
print(f"   - raw_data: {raw_data.shape}")
print(f"   - df_raw: {df_raw.shape}")
print("\n🎯 Ready for Sheet 2: Data Normalization")


✅ Sheet 1 Variables Saved:
   - config: 17 parameters
   - raw_data: (12, 2)
   - df_raw: (12, 2)

🎯 Ready for Sheet 2: Data Normalization


---

# SHEET 2: Data Normalization (StandardScaler)

Tahapan ini berisi:
1. Split data menjadi Train/Test set
2. Perhitungan Mean & Standard Deviation dari training data
3. Normalisasi semua data menggunakan statistics dari training
4. Formula Excel untuk setiap perhitungan

**Formula Normalisasi:**
$$z = \frac{x - \mu}{\sigma}$$

Di mana:
- $z$ = normalized value
- $x$ = raw value
- $\mu$ = mean (dari training data)
- $\sigma$ = standard deviation (dari training data)

---

## 2.1 Split Data: Train vs Test

Untuk normalisasi yang benar, kita harus:
1. Hitung mean & std **hanya dari training data**
2. Gunakan statistics tersebut untuk normalize train, val, dan test

**Splitting Strategy:**
- Train: timestep 0-7 (8 timesteps) → untuk menghitung mean & std
- Test: timestep 8-11 (4 timesteps) → hanya untuk validasi

Ini mengikuti prinsip: **"Test data tidak boleh mempengaruhi normalization statistics"**

In [8]:
# ============================================
# SPLIT DATA INTO TRAIN AND TEST
# ============================================

# Train: first 8 timesteps (t_0 to t_7)
# Test: last 4 timesteps (t_8 to t_11)

train_border = 8  # First 8 timesteps for training

train_data = raw_data[:train_border]  # [0:8] → shape (8, 2)
test_data = raw_data[train_border:]   # [8:12] → shape (4, 2)

print("="*60)
print("DATA SPLIT")
print("="*60)
print(f"Total timesteps: {n_timesteps}")
print(f"Train timesteps: {train_border} (t_0 to t_{train_border-1})")
print(f"Test timesteps:  {n_timesteps - train_border} (t_{train_border} to t_{n_timesteps-1})")
print(f"\nTrain data shape: {train_data.shape}")
print(f"Test data shape:  {test_data.shape}")
print("="*60)

# Display train data
df_train = pd.DataFrame(
    train_data,
    columns=['Feature_1', 'Feature_2'],
    index=[f't_{i}' for i in range(train_border)]
)
df_train.index.name = 'Timestep'

print("\n📊 TRAINING DATA (used for calculating mean & std)")
print("="*60)
print(df_train.to_string())
print("="*60)

DATA SPLIT
Total timesteps: 12
Train timesteps: 8 (t_0 to t_7)
Test timesteps:  4 (t_8 to t_11)

Train data shape: (8, 2)
Test data shape:  (4, 2)

📊 TRAINING DATA (used for calculating mean & std)
          Feature_1  Feature_2
Timestep                      
t_0           10.10  20.200000
t_1           11.80  24.230127
t_2           14.15  24.330127
t_3           15.90  20.100000
t_4           18.20  15.469873
t_5           19.85  15.819873
t_6           22.10  19.900000
t_7           24.00  24.530127


## 2.2 Calculate Mean and Standard Deviation

Hitung statistik dari training data saja (per feature/column).

**Formula:**
- Mean: $\mu = \frac{1}{n}\sum_{i=1}^{n} x_i$
- Std: $\sigma = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (x_i - \mu)^2}$

In [9]:
# ============================================
# CALCULATE MEAN AND STD FROM TRAINING DATA
# ============================================

# Calculate per feature (column-wise)
train_mean = train_data.mean(axis=0)  # Mean of each column
train_std = train_data.std(axis=0, ddof=0)  # Population std (ddof=0)

print("="*60)
print("NORMALIZATION STATISTICS (from Training Data)")
print("="*60)
print(f"Feature_1 Mean: {train_mean[0]:.6f}")
print(f"Feature_1 Std:  {train_std[0]:.6f}")
print(f"\nFeature_2 Mean: {train_mean[1]:.6f}")
print(f"Feature_2 Std:  {train_std[1]:.6f}")
print("="*60)

# Create statistics table for Excel
df_stats = pd.DataFrame({
    'Feature': ['Feature_1', 'Feature_2'],
    'Mean (μ)': train_mean,
    'Std (σ)': train_std
})

print("\n📊 COPY THIS TO EXCEL - Sheet 2: Normalization Statistics")
print("="*60)
print(df_stats.to_string(index=False))
print("="*60)

NORMALIZATION STATISTICS (from Training Data)
Feature_1 Mean: 17.012500
Feature_1 Std:  4.590122

Feature_2 Mean: 20.572516
Feature_2 Std:  3.402193

📊 COPY THIS TO EXCEL - Sheet 2: Normalization Statistics
  Feature  Mean (μ)  Std (σ)
Feature_1 17.012500 4.590122
Feature_2 20.572516 3.402193


## 2.3 Manual Calculation Example (Feature_1 Mean)

Mari kita hitung manual mean untuk Feature_1 sebagai contoh:

**Data Feature_1 (Training):** [10.10, 11.80, 14.15, 15.90, 18.20, 19.85, 22.10, 24.00]

**Step-by-step:**
1. Sum = 10.10 + 11.80 + 14.15 + 15.90 + 18.20 + 19.85 + 22.10 + 24.00
2. Count = 8
3. Mean = Sum / Count

In [10]:
# ============================================
# MANUAL CALCULATION EXAMPLE - MEAN
# ============================================

print("="*60)
print("MANUAL CALCULATION: Feature_1 Mean")
print("="*60)

# Feature_1 training values
f1_train = train_data[:, 0]
print(f"Training values: {f1_train}")
print()

# Step-by-step calculation
print("Step-by-step calculation:")
sum_f1 = 0
for i, val in enumerate(f1_train):
    sum_f1 += val
    print(f"  Step {i+1}: sum = {sum_f1:.2f} (added {val:.2f})")

print(f"\nFinal sum: {sum_f1:.6f}")
print(f"Count: {len(f1_train)}")
mean_f1_manual = sum_f1 / len(f1_train)
print(f"Mean = {sum_f1:.6f} / {len(f1_train)} = {mean_f1_manual:.6f}")
print(f"\nVerification with NumPy: {train_mean[0]:.6f}")
print(f"Match: {np.isclose(mean_f1_manual, train_mean[0])}")
print("="*60)

# Excel formula
print("\n📝 EXCEL FORMULA for Mean:")
print("="*60)
print("If Feature_1 training data is in cells B2:B9,")
print("Mean formula: =AVERAGE(B2:B9)")
print("Or manual: =(B2+B3+B4+B5+B6+B7+B8+B9)/8")
print("="*60)

MANUAL CALCULATION: Feature_1 Mean
Training values: [10.1  11.8  14.15 15.9  18.2  19.85 22.1  24.  ]

Step-by-step calculation:
  Step 1: sum = 10.10 (added 10.10)
  Step 2: sum = 21.90 (added 11.80)
  Step 3: sum = 36.05 (added 14.15)
  Step 4: sum = 51.95 (added 15.90)
  Step 5: sum = 70.15 (added 18.20)
  Step 6: sum = 90.00 (added 19.85)
  Step 7: sum = 112.10 (added 22.10)
  Step 8: sum = 136.10 (added 24.00)

Final sum: 136.100000
Count: 8
Mean = 136.100000 / 8 = 17.012500

Verification with NumPy: 17.012500
Match: True

📝 EXCEL FORMULA for Mean:
If Feature_1 training data is in cells B2:B9,
Mean formula: =AVERAGE(B2:B9)
Or manual: =(B2+B3+B4+B5+B6+B7+B8+B9)/8


## 2.4 Manual Calculation Example (Feature_1 Std)

**Formula:**
$$\sigma = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (x_i - \mu)^2}$$

**Steps:**
1. Calculate deviation from mean: $(x_i - \mu)$
2. Square each deviation: $(x_i - \mu)^2$
3. Sum all squared deviations
4. Divide by count (n)
5. Take square root

In [11]:
# ============================================
# MANUAL CALCULATION EXAMPLE - STD
# ============================================

print("="*60)
print("MANUAL CALCULATION: Feature_1 Standard Deviation")
print("="*60)

# Use previously calculated mean
mean_f1 = train_mean[0]
print(f"Mean (μ): {mean_f1:.6f}\n")

# Step-by-step calculation
print("Step-by-step calculation:")
print(f"{'i':<4} {'x_i':<10} {'(x_i - μ)':<15} {'(x_i - μ)²':<15}")
print("-"*60)

squared_deviations = []
for i, val in enumerate(f1_train):
    deviation = val - mean_f1
    squared_dev = deviation ** 2
    squared_deviations.append(squared_dev)
    print(f"{i:<4} {val:<10.2f} {deviation:<15.6f} {squared_dev:<15.6f}")

print("-"*60)
sum_squared_dev = sum(squared_deviations)
variance = sum_squared_dev / len(f1_train)
std_f1_manual = np.sqrt(variance)

print(f"\nSum of squared deviations: {sum_squared_dev:.6f}")
print(f"Variance (σ²) = {sum_squared_dev:.6f} / {len(f1_train)} = {variance:.6f}")
print(f"Std (σ) = √{variance:.6f} = {std_f1_manual:.6f}")
print(f"\nVerification with NumPy: {train_std[0]:.6f}")
print(f"Match: {np.isclose(std_f1_manual, train_std[0])}")
print("="*60)

# Excel formula
print("\n📝 EXCEL FORMULA for Std:")
print("="*60)
print("If Feature_1 training data is in cells B2:B9,")
print("Std formula: =STDEV.P(B2:B9)  [Population std]")
print("Or manual: =SQRT(SUMXMY2(B2:B9, mean_cell)/8)")
print("="*60)

MANUAL CALCULATION: Feature_1 Standard Deviation
Mean (μ): 17.012500

Step-by-step calculation:
i    x_i        (x_i - μ)       (x_i - μ)²     
------------------------------------------------------------
0    10.10      -6.912500       47.782656      
1    11.80      -5.212500       27.170156      
2    14.15      -2.862500       8.193906       
3    15.90      -1.112500       1.237656       
4    18.20      1.187500        1.410156       
5    19.85      2.837500        8.051406       
6    22.10      5.087500        25.882656      
7    24.00      6.987500        48.825156      
------------------------------------------------------------

Sum of squared deviations: 168.553750
Variance (σ²) = 168.553750 / 8 = 21.069219
Std (σ) = √21.069219 = 4.590122

Verification with NumPy: 4.590122
Match: True

📝 EXCEL FORMULA for Std:
If Feature_1 training data is in cells B2:B9,
Std formula: =STDEV.P(B2:B9)  [Population std]
Or manual: =SQRT(SUMXMY2(B2:B9, mean_cell)/8)


## 2.5 Normalize ALL Data

Sekarang normalize semua timesteps (train + test) menggunakan statistics dari training data.

**Formula:** $z = \frac{x - \mu}{\sigma}$

Di mana $\mu$ dan $\sigma$ berasal dari training data saja.

In [12]:
# ============================================
# NORMALIZE ALL DATA (TRAIN + TEST)
# ============================================

# Apply normalization: z = (x - mean) / std
normalized_data = (raw_data - train_mean) / train_std

print("="*60)
print("NORMALIZED DATA (using training statistics)")
print("="*60)
print(f"Shape: {normalized_data.shape}")
print("="*60)

# Create DataFrame for normalized data
df_normalized = pd.DataFrame(
    normalized_data,
    columns=['Feature_1_norm', 'Feature_2_norm'],
    index=[f't_{i}' for i in range(n_timesteps)]
)
df_normalized.index.name = 'Timestep'

print("\n📊 COPY THIS TO EXCEL - Sheet 2: Normalized Data")
print("="*60)
pd.set_option('display.float_format', '{:.6f}'.format)
print(df_normalized.to_string())
pd.reset_option('display.float_format')
print("="*60)

# Verify: training data should have mean≈0, std≈1
train_normalized = normalized_data[:train_border]
print(f"\n✅ VERIFICATION (Training portion):")
print(f"   Mean of normalized Feature_1: {train_normalized[:, 0].mean():.10f} (should be ≈0)")
print(f"   Std of normalized Feature_1:  {train_normalized[:, 0].std(ddof=0):.10f} (should be ≈1)")
print(f"   Mean of normalized Feature_2: {train_normalized[:, 1].mean():.10f} (should be ≈0)")
print(f"   Std of normalized Feature_2:  {train_normalized[:, 1].std(ddof=0):.10f} (should be ≈1)")

NORMALIZED DATA (using training statistics)
Shape: (12, 2)

📊 COPY THIS TO EXCEL - Sheet 2: Normalized Data
          Feature_1_norm  Feature_2_norm
Timestep                                
t_0            -1.505951       -0.109493
t_1            -1.135591        1.075075
t_2            -0.623622        1.104467
t_3            -0.242368       -0.138886
t_4             0.258708       -1.499810
t_5             0.618175       -1.396935
t_6             1.108358       -0.197671
t_7             1.522291        1.163253
t_8             1.936223        1.104467
t_9             2.437299       -0.212368
t_10            2.807660       -1.411631
t_11            3.297843       -1.455721

✅ VERIFICATION (Training portion):
   Mean of normalized Feature_1: 0.0000000000 (should be ≈0)
   Std of normalized Feature_1:  1.0000000000 (should be ≈1)
   Mean of normalized Feature_2: -0.0000000000 (should be ≈0)
   Std of normalized Feature_2:  1.0000000000 (should be ≈1)


## 2.6 Detailed Example: Normalize One Value

Mari kita hitung manual normalisasi untuk satu nilai sebagai contoh:

**Example:** Normalize Feature_1 di timestep t_0 (value = 10.10)

In [13]:
# ============================================
# DETAILED NORMALIZATION EXAMPLE
# ============================================

print("="*60)
print("NORMALIZATION EXAMPLE: Feature_1 at t_0")
print("="*60)

# Values
x_value = raw_data[0, 0]  # Feature_1 at t_0
mean_value = train_mean[0]
std_value = train_std[0]

print(f"Raw value (x):     {x_value:.6f}")
print(f"Training mean (μ): {mean_value:.6f}")
print(f"Training std (σ):  {std_value:.6f}")
print()

# Step-by-step
print("Step-by-step calculation:")
print(f"1. Subtract mean: x - μ = {x_value:.6f} - {mean_value:.6f} = {x_value - mean_value:.6f}")
print(f"2. Divide by std: (x - μ) / σ = {x_value - mean_value:.6f} / {std_value:.6f} = {(x_value - mean_value) / std_value:.6f}")
print()

z_manual = (x_value - mean_value) / std_value
z_from_array = normalized_data[0, 0]

print(f"Result (z): {z_manual:.6f}")
print(f"Verification from array: {z_from_array:.6f}")
print(f"Match: {np.isclose(z_manual, z_from_array)}")
print("="*60)

# Excel formula
print("\n📝 EXCEL FORMULA for Normalization:")
print("="*60)
print("If raw value is in cell B2,")
print("   mean is in cell $F$1,")
print("   std is in cell $F$2,")
print("")
print("Formula: =(B2-$F$1)/$F$2")
print("")
print("The $ signs make the reference absolute (fixed when copying)")
print("="*60)

NORMALIZATION EXAMPLE: Feature_1 at t_0
Raw value (x):     10.100000
Training mean (μ): 17.012500
Training std (σ):  4.590122

Step-by-step calculation:
1. Subtract mean: x - μ = 10.100000 - 17.012500 = -6.912500
2. Divide by std: (x - μ) / σ = -6.912500 / 4.590122 = -1.505951

Result (z): -1.505951
Verification from array: -1.505951
Match: True

📝 EXCEL FORMULA for Normalization:
If raw value is in cell B2,
   mean is in cell $F$1,
   std is in cell $F$2,

Formula: =(B2-$F$1)/$F$2

The $ signs make the reference absolute (fixed when copying)


## 2.7 Side-by-Side Comparison Table

Tabel perbandingan Raw vs Normalized untuk semua data.

In [14]:
# ============================================
# SIDE-BY-SIDE COMPARISON TABLE
# ============================================

# Create comprehensive comparison table
df_comparison = pd.DataFrame({
    'Timestep': [f't_{i}' for i in range(n_timesteps)],
    'F1_Raw': raw_data[:, 0],
    'F1_Normalized': normalized_data[:, 0],
    'F2_Raw': raw_data[:, 1],
    'F2_Normalized': normalized_data[:, 1],
})

# Add a column to mark train vs test
df_comparison['Split'] = ['TRAIN'] * train_border + ['TEST'] * (n_timesteps - train_border)

print("="*80)
print("📊 COPY THIS TO EXCEL - Sheet 2: Complete Comparison Table")
print("="*80)
pd.set_option('display.float_format', '{:.6f}'.format)
print(df_comparison.to_string(index=False))
pd.reset_option('display.float_format')
print("="*80)

📊 COPY THIS TO EXCEL - Sheet 2: Complete Comparison Table
Timestep    F1_Raw  F1_Normalized    F2_Raw  F2_Normalized Split
     t_0 10.100000      -1.505951 20.200000      -0.109493 TRAIN
     t_1 11.800000      -1.135591 24.230127       1.075075 TRAIN
     t_2 14.150000      -0.623622 24.330127       1.104467 TRAIN
     t_3 15.900000      -0.242368 20.100000      -0.138886 TRAIN
     t_4 18.200000       0.258708 15.469873      -1.499810 TRAIN
     t_5 19.850000       0.618175 15.819873      -1.396935 TRAIN
     t_6 22.100000       1.108358 19.900000      -0.197671 TRAIN
     t_7 24.000000       1.522291 24.530127       1.163253 TRAIN
     t_8 25.900000       1.936223 24.330127       1.104467  TEST
     t_9 28.200000       2.437299 19.850000      -0.212368  TEST
    t_10 29.900000       2.807660 15.769873      -1.411631  TEST
    t_11 32.150000       3.297843 15.619873      -1.455721  TEST


## 2.8 Summary for Sheet 2

**Status: ✅ Sheet 2 Complete**

### What we have now:
1. ✅ Data split (Train: 8 timesteps, Test: 4 timesteps)
2. ✅ Mean & Std calculated from training data only
3. ✅ All data normalized using training statistics
4. ✅ Manual calculation examples with step-by-step breakdown
5. ✅ Excel formulas provided

### Key Statistics:
- **Feature_1:** Mean = {:.4f}, Std = {:.4f}
- **Feature_2:** Mean = {:.4f}, Std = {:.4f}

### Excel Sheet 2 Contents:
1. Training statistics (mean, std)
2. Raw data vs Normalized data comparison
3. Manual calculation examples
4. Excel formulas for each step

### Next Steps:
- **Sheet 3**: Time Features Extraction (hour_sin, hour_cos encoding)
- **Sheet 4**: Multi-Scale Downsampling
- **Sheet 5**: Embedding Layer
- ... and so on

---

**⏸️ PAUSE HERE - Waiting for confirmation before proceeding to Sheet 3**

In [15]:
# Save variables for next sheets
print("="*60)
print("✅ Sheet 2 Variables Saved:")
print("="*60)
print(f"   - train_data: {train_data.shape}")
print(f"   - test_data: {test_data.shape}")
print(f"   - train_mean: {train_mean}")
print(f"   - train_std: {train_std}")
print(f"   - normalized_data: {normalized_data.shape}")
print(f"   - df_normalized: {df_normalized.shape}")
print("="*60)
print("\n🎯 Ready for Sheet 3: Time Features Extraction")
print("="*60)

✅ Sheet 2 Variables Saved:
   - train_data: (8, 2)
   - test_data: (4, 2)
   - train_mean: [17.0125     20.57251588]
   - train_std: [4.59012187 3.40219309]
   - normalized_data: (12, 2)
   - df_normalized: (12, 2)

🎯 Ready for Sheet 3: Time Features Extraction


---

# SHEET 3: Time Features Extraction

Tahapan ini berisi:
1. Generate timestamp untuk setiap timestep
2. Ekstraksi fitur temporal (hour, day, month, dll)
3. Encoding dengan sine-cosine untuk fitur siklik
4. Persiapan time features untuk input model

**Why Time Features?**
- TimeMixer menggunakan temporal information untuk memahami pola musiman
- Encoding sine-cosine membuat fitur siklik (24 jam = kembali ke 0)

**Time Features kita:**
- `hour_sin` dan `hour_cos` - Encoding untuk jam dalam sehari (sederhana untuk manual calculation)

---

## 3.1 Generate Timestamps

Untuk dataset kita (12 timesteps), kita akan simulasikan timestamps dengan interval 1 jam.

**Asumsi:**
- Start: 2024-01-01 00:00:00
- Interval: 1 jam
- Total: 12 timesteps (0 sampai 11 jam)

In [16]:
# ============================================
# GENERATE TIMESTAMPS
# ============================================

from datetime import datetime, timedelta

# Start timestamp
start_time = datetime(2024, 1, 1, 0, 0, 0)  # 2024-01-01 00:00:00

# Generate timestamps with 1-hour interval
timestamps = [start_time + timedelta(hours=i) for i in range(n_timesteps)]

# Extract hour information (0-23)
hours = [ts.hour for ts in timestamps]

print("="*80)
print("TIMESTAMPS GENERATED")
print("="*80)
print(f"Start: {timestamps[0]}")
print(f"End:   {timestamps[-1]}")
print(f"Interval: 1 hour")
print(f"Total timesteps: {len(timestamps)}")
print("="*80)

# Create DataFrame for display
df_timestamps = pd.DataFrame({
    'Timestep': [f't_{i}' for i in range(n_timesteps)],
    'Datetime': timestamps,
    'Hour': hours
})

print("\n📊 COPY THIS TO EXCEL - Sheet 3: Timestamps")
print("="*80)
print(df_timestamps.to_string(index=False))
print("="*80)

TIMESTAMPS GENERATED
Start: 2024-01-01 00:00:00
End:   2024-01-01 11:00:00
Interval: 1 hour
Total timesteps: 12

📊 COPY THIS TO EXCEL - Sheet 3: Timestamps
Timestep            Datetime  Hour
     t_0 2024-01-01 00:00:00     0
     t_1 2024-01-01 01:00:00     1
     t_2 2024-01-01 02:00:00     2
     t_3 2024-01-01 03:00:00     3
     t_4 2024-01-01 04:00:00     4
     t_5 2024-01-01 05:00:00     5
     t_6 2024-01-01 06:00:00     6
     t_7 2024-01-01 07:00:00     7
     t_8 2024-01-01 08:00:00     8
     t_9 2024-01-01 09:00:00     9
    t_10 2024-01-01 10:00:00    10
    t_11 2024-01-01 11:00:00    11


## 3.2 Sine-Cosine Encoding for Hour

Untuk membuat fitur siklik yang smooth, kita encode hour menggunakan sine dan cosine.

**Formula:**
$$\text{hour\_sin} = \sin\left(\frac{2\pi \times \text{hour}}{24}\right)$$
$$\text{hour\_cos} = \cos\left(\frac{2\pi \times \text{hour}}{24}\right)$$

**Why sine-cosine?**
- Hour 23 dan hour 0 harus dekat (bukan jauh seperti angka biasa)
- Sine-cosine membuat lingkaran: (sin, cos) di hour 0 ≈ hour 24
- Model dapat belajar pola siklik dengan lebih baik

In [17]:
# ============================================
# SINE-COSINE ENCODING FOR HOUR
# ============================================

# Calculate sine and cosine encoding
hour_sin = np.sin(2 * np.pi * np.array(hours) / 24)
hour_cos = np.cos(2 * np.pi * np.array(hours) / 24)

# Combine into time features array
time_features = np.column_stack([hour_sin, hour_cos])  # Shape: (12, 2)

print("="*80)
print("TIME FEATURES ENCODED")
print("="*80)
print(f"Shape: {time_features.shape}")
print(f"Features: ['hour_sin', 'hour_cos']")
print("="*80)

# Create DataFrame for display
df_time_features = pd.DataFrame({
    'Timestep': [f't_{i}' for i in range(n_timesteps)],
    'Hour': hours,
    'hour_sin': hour_sin,
    'hour_cos': hour_cos
})

print("\n📊 COPY THIS TO EXCEL - Sheet 3: Time Features")
print("="*80)
pd.set_option('display.float_format', '{:.6f}'.format)
print(df_time_features.to_string(index=False))
pd.reset_option('display.float_format')
print("="*80)

TIME FEATURES ENCODED
Shape: (12, 2)
Features: ['hour_sin', 'hour_cos']

📊 COPY THIS TO EXCEL - Sheet 3: Time Features
Timestep  Hour  hour_sin  hour_cos
     t_0     0  0.000000  1.000000
     t_1     1  0.258819  0.965926
     t_2     2  0.500000  0.866025
     t_3     3  0.707107  0.707107
     t_4     4  0.866025  0.500000
     t_5     5  0.965926  0.258819
     t_6     6  1.000000  0.000000
     t_7     7  0.965926 -0.258819
     t_8     8  0.866025 -0.500000
     t_9     9  0.707107 -0.707107
    t_10    10  0.500000 -0.866025
    t_11    11  0.258819 -0.965926


## 3.3 Manual Calculation Example (Hour = 0)

Mari kita hitung manual sine-cosine encoding untuk hour = 0 sebagai contoh.

**Given:** hour = 0

In [18]:
# ============================================
# MANUAL CALCULATION EXAMPLE - SINE-COSINE ENCODING
# ============================================

print("="*80)
print("MANUAL CALCULATION: Sine-Cosine Encoding for hour = 0")
print("="*80)

hour_example = 0
print(f"Hour: {hour_example}")
print()

# Step-by-step calculation
print("Step-by-step calculation:")
print()

# Step 1: Calculate angle in radians
angle_rad = 2 * np.pi * hour_example / 24
print(f"1. Calculate angle:")
print(f"   angle = 2π × hour / 24")
print(f"   angle = 2π × {hour_example} / 24")
print(f"   angle = {angle_rad:.6f} radians")
print(f"   (atau {np.degrees(angle_rad):.2f} degrees)")
print()

# Step 2: Calculate sine
sin_value = np.sin(angle_rad)
print(f"2. Calculate sine:")
print(f"   hour_sin = sin({angle_rad:.6f})")
print(f"   hour_sin = {sin_value:.6f}")
print()

# Step 3: Calculate cosine
cos_value = np.cos(angle_rad)
print(f"3. Calculate cosine:")
print(f"   hour_cos = cos({angle_rad:.6f})")
print(f"   hour_cos = {cos_value:.6f}")
print()

# Verification
print("Verification from array:")
print(f"   hour_sin[0] = {hour_sin[0]:.6f}")
print(f"   hour_cos[0] = {hour_cos[0]:.6f}")
print(f"   Match: {np.isclose(sin_value, hour_sin[0]) and np.isclose(cos_value, hour_cos[0])}")
print("="*80)

# Excel formula
print("\n📝 EXCEL FORMULA for Sine-Cosine Encoding:")
print("="*80)
print("If hour is in cell B2,")
print("")
print("hour_sin formula: =SIN(2*PI()*B2/24)")
print("hour_cos formula: =COS(2*PI()*B2/24)")
print("")
print("Note: Excel's PI() function returns π (3.14159...)")
print("="*80)

MANUAL CALCULATION: Sine-Cosine Encoding for hour = 0
Hour: 0

Step-by-step calculation:

1. Calculate angle:
   angle = 2π × hour / 24
   angle = 2π × 0 / 24
   angle = 0.000000 radians
   (atau 0.00 degrees)

2. Calculate sine:
   hour_sin = sin(0.000000)
   hour_sin = 0.000000

3. Calculate cosine:
   hour_cos = cos(0.000000)
   hour_cos = 1.000000

Verification from array:
   hour_sin[0] = 0.000000
   hour_cos[0] = 1.000000
   Match: True

📝 EXCEL FORMULA for Sine-Cosine Encoding:
If hour is in cell B2,

hour_sin formula: =SIN(2*PI()*B2/24)
hour_cos formula: =COS(2*PI()*B2/24)

Note: Excel's PI() function returns π (3.14159...)


## 3.4 More Examples (Hour = 3, 6, 9)

Mari lihat beberapa contoh lagi untuk memahami pola siklik.

In [19]:
# ============================================
# MORE EXAMPLES - DIFFERENT HOURS
# ============================================

print("="*80)
print("SINE-COSINE ENCODING FOR DIFFERENT HOURS")
print("="*80)
print(f"{'Hour':<6} {'Angle (rad)':<15} {'Angle (deg)':<15} {'hour_sin':<12} {'hour_cos':<12}")
print("-"*80)

example_hours = [0, 3, 6, 9]
for h in example_hours:
    angle = 2 * np.pi * h / 24
    sin_val = np.sin(angle)
    cos_val = np.cos(angle)
    print(f"{h:<6} {angle:<15.6f} {np.degrees(angle):<15.2f} {sin_val:<12.6f} {cos_val:<12.6f}")

print("="*80)

# Pattern explanation
print("\n💡 PATTERN EXPLANATION:")
print("="*80)
print("Hour 0:  angle = 0°      → sin = 0.000,  cos = 1.000  (start of circle)")
print("Hour 6:  angle = 90°     → sin = 1.000,  cos = 0.000  (top of circle)")
print("Hour 12: angle = 180°    → sin = 0.000,  cos = -1.000 (opposite side)")
print("Hour 18: angle = 270°    → sin = -1.000, cos = 0.000  (bottom of circle)")
print("Hour 24: angle = 360°    → sin = 0.000,  cos = 1.000  (back to start!)")
print()
print("This creates a smooth circular representation!")
print("="*80)

SINE-COSINE ENCODING FOR DIFFERENT HOURS
Hour   Angle (rad)     Angle (deg)     hour_sin     hour_cos    
--------------------------------------------------------------------------------
0      0.000000        0.00            0.000000     1.000000    
3      0.785398        45.00           0.707107     0.707107    
6      1.570796        90.00           1.000000     0.000000    
9      2.356194        135.00          0.707107     -0.707107   

💡 PATTERN EXPLANATION:
Hour 0:  angle = 0°      → sin = 0.000,  cos = 1.000  (start of circle)
Hour 6:  angle = 90°     → sin = 1.000,  cos = 0.000  (top of circle)
Hour 12: angle = 180°    → sin = 0.000,  cos = -1.000 (opposite side)
Hour 18: angle = 270°    → sin = -1.000, cos = 0.000  (bottom of circle)
Hour 24: angle = 360°    → sin = 0.000,  cos = 1.000  (back to start!)

This creates a smooth circular representation!


## 3.5 Visualization Helper (Optional)

Untuk lebih memahami, berikut koordinat (sin, cos) yang membentuk lingkaran.

In [20]:
# ============================================
# CIRCULAR REPRESENTATION
# ============================================

print("="*80)
print("CIRCULAR REPRESENTATION (for Excel scatter plot)")
print("="*80)
print(f"{'Hour':<6} {'X (hour_sin)':<15} {'Y (hour_cos)':<15}")
print("-"*80)

for i, h in enumerate(hours):
    print(f"{h:<6} {hour_sin[i]:<15.6f} {hour_cos[i]:<15.6f}")

print("="*80)
print("\n💡 TIP FOR EXCEL:")
print("Plot hour_sin (X-axis) vs hour_cos (Y-axis) in a scatter chart")
print("You'll see the points form a circle!")
print("This shows the cyclic nature of time features.")
print("="*80)

CIRCULAR REPRESENTATION (for Excel scatter plot)
Hour   X (hour_sin)    Y (hour_cos)   
--------------------------------------------------------------------------------
0      0.000000        1.000000       
1      0.258819        0.965926       
2      0.500000        0.866025       
3      0.707107        0.707107       
4      0.866025        0.500000       
5      0.965926        0.258819       
6      1.000000        0.000000       
7      0.965926        -0.258819      
8      0.866025        -0.500000      
9      0.707107        -0.707107      
10     0.500000        -0.866025      
11     0.258819        -0.965926      

💡 TIP FOR EXCEL:
Plot hour_sin (X-axis) vs hour_cos (Y-axis) in a scatter chart
You'll see the points form a circle!
This shows the cyclic nature of time features.


## 3.6 Complete Time Features Table

Tabel lengkap dengan semua informasi untuk Excel.

In [21]:
# ============================================
# COMPLETE TIME FEATURES TABLE
# ============================================

# Create comprehensive table with all info
df_time_complete = pd.DataFrame({
    'Timestep': [f't_{i}' for i in range(n_timesteps)],
    'Datetime': timestamps,
    'Hour': hours,
    'Angle_Radians': 2 * np.pi * np.array(hours) / 24,
    'Angle_Degrees': np.degrees(2 * np.pi * np.array(hours) / 24),
    'hour_sin': hour_sin,
    'hour_cos': hour_cos
})

print("="*80)
print("📊 COPY THIS TO EXCEL - Sheet 3: Complete Time Features")
print("="*80)
pd.set_option('display.float_format', '{:.6f}'.format)
print(df_time_complete.to_string(index=False))
pd.reset_option('display.float_format')
print("="*80)

# Summary statistics
print("\n📊 SUMMARY STATISTICS:")
print("="*80)
print(f"hour_sin range: [{hour_sin.min():.6f}, {hour_sin.max():.6f}]")
print(f"hour_cos range: [{hour_cos.min():.6f}, {hour_cos.max():.6f}]")
print(f"hour_sin mean:  {hour_sin.mean():.6f}")
print(f"hour_cos mean:  {hour_cos.mean():.6f}")
print("="*80)

📊 COPY THIS TO EXCEL - Sheet 3: Complete Time Features
Timestep            Datetime  Hour  Angle_Radians  Angle_Degrees  hour_sin  hour_cos
     t_0 2024-01-01 00:00:00     0       0.000000       0.000000  0.000000  1.000000
     t_1 2024-01-01 01:00:00     1       0.261799      15.000000  0.258819  0.965926
     t_2 2024-01-01 02:00:00     2       0.523599      30.000000  0.500000  0.866025
     t_3 2024-01-01 03:00:00     3       0.785398      45.000000  0.707107  0.707107
     t_4 2024-01-01 04:00:00     4       1.047198      60.000000  0.866025  0.500000
     t_5 2024-01-01 05:00:00     5       1.308997      75.000000  0.965926  0.258819
     t_6 2024-01-01 06:00:00     6       1.570796      90.000000  1.000000  0.000000
     t_7 2024-01-01 07:00:00     7       1.832596     105.000000  0.965926 -0.258819
     t_8 2024-01-01 08:00:00     8       2.094395     120.000000  0.866025 -0.500000
     t_9 2024-01-01 09:00:00     9       2.356194     135.000000  0.707107 -0.707107
    t_10 2

## 3.7 Summary for Sheet 3

**Status: ✅ Sheet 3 Complete**

### What we have now:
1. ✅ Timestamps generated (12 hourly timestamps)
2. ✅ Hour extracted (0 to 11)
3. ✅ Sine-cosine encoding applied
4. ✅ Time features array created: shape (12, 2)
5. ✅ Manual calculation examples
6. ✅ Excel formulas provided

### Key Outputs:
- **time_features**: Array of shape (12, 2) containing [hour_sin, hour_cos]
- **Range**: Both features are in [-1, 1]

### Excel Sheet 3 Contents:
1. Timestamps table
2. Time features with sine-cosine encoding
3. Manual calculation steps
4. Circular representation coordinates
5. Excel formulas

### Next Steps:
- **Sheet 4**: Multi-Scale Downsampling (create multiple temporal resolutions)
- **Sheet 5**: Embedding Layer (Linear projection)
- **Sheet 6**: Series Decomposition (Season + Trend)
- ... and so on

---

**⏸️ PAUSE HERE - Waiting for confirmation before proceeding to Sheet 4**

In [22]:
# Save variables for next sheets
print("="*60)
print("✅ Sheet 3 Variables Saved:")
print("="*60)
print(f"   - timestamps: {len(timestamps)} datetime objects")
print(f"   - hours: {hours}")
print(f"   - time_features: {time_features.shape}")
print(f"   - hour_sin: {hour_sin.shape}")
print(f"   - hour_cos: {hour_cos.shape}")
print("="*60)
print("\n🎯 Ready for Sheet 4: Multi-Scale Downsampling")
print("="*60)

✅ Sheet 3 Variables Saved:
   - timestamps: 12 datetime objects
   - hours: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
   - time_features: (12, 2)
   - hour_sin: (12,)
   - hour_cos: (12,)

🎯 Ready for Sheet 4: Multi-Scale Downsampling


---

# SHEET 4: Multi-Scale Downsampling

Tahapan ini berisi:
1. Ekstraksi sequence untuk input model (seq_len = 8)
2. Downsampling untuk membuat multiple temporal resolutions
3. Aplikasi downsampling ke data dan time features
4. Persiapan input batch

**Why Multi-Scale?**
- TimeMixer menggunakan multiple temporal resolutions untuk menangkap pola di berbagai skala waktu
- Scale 1 (Original): Detail granular (8 timesteps)
- Scale 2 (Downsampled): Pola lebih luas (4 timesteps)

**Downsampling Method:** Average Pooling dengan window = 2

---

## 4.1 Extract Input Sequence (seq_len = 8)

Dari normalized data (12 timesteps), kita ambil 8 timesteps pertama sebagai input sequence.

**Mapping:**
- Input (x): t_0 sampai t_7 (8 timesteps)
- Target (y): t_2 sampai t_11 (6 timesteps: 2 label + 4 prediction)

In [23]:
# ============================================
# EXTRACT INPUT SEQUENCE
# ============================================

# Extract input sequence (first seq_len timesteps)
seq_x = normalized_data[:seq_len]  # Shape: (8, 2)
seq_x_mark = time_features[:seq_len]  # Shape: (8, 2)

# Extract target sequence (for reference - not used in downsampling)
# From index (seq_len - label_len) to (seq_len - label_len + label_len + pred_len)
target_start = seq_len - label_len  # 8 - 2 = 6
target_end = target_start + label_len + pred_len  # 6 + 2 + 4 = 12
seq_y = normalized_data[target_start:target_end]  # Shape: (6, 2)
seq_y_mark = time_features[target_start:target_end]  # Shape: (6, 2)

print("="*80)
print("INPUT & TARGET SEQUENCES EXTRACTED")
print("="*80)
print(f"Input sequence (seq_x):      shape {seq_x.shape} - timesteps t_0 to t_{seq_len-1}")
print(f"Input time features:         shape {seq_x_mark.shape}")
print(f"Target sequence (seq_y):     shape {seq_y.shape} - timesteps t_{target_start} to t_{target_end-1}")
print(f"Target time features:        shape {seq_y_mark.shape}")
print("="*80)

# Add batch dimension (batch_size = 1)
batch_x = seq_x[np.newaxis, ...]  # Shape: (1, 8, 2)
batch_x_mark = seq_x_mark[np.newaxis, ...]  # Shape: (1, 8, 2)
batch_y = seq_y[np.newaxis, ...]  # Shape: (1, 6, 2)
batch_y_mark = seq_y_mark[np.newaxis, ...]  # Shape: (1, 6, 2)

print(f"\nWith batch dimension:")
print(f"batch_x:      {batch_x.shape}")
print(f"batch_x_mark: {batch_x_mark.shape}")
print(f"batch_y:      {batch_y.shape}")
print(f"batch_y_mark: {batch_y_mark.shape}")
print("="*80)

INPUT & TARGET SEQUENCES EXTRACTED
Input sequence (seq_x):      shape (8, 2) - timesteps t_0 to t_7
Input time features:         shape (8, 2)
Target sequence (seq_y):     shape (6, 2) - timesteps t_6 to t_11
Target time features:        shape (6, 2)

With batch dimension:
batch_x:      (1, 8, 2)
batch_x_mark: (1, 8, 2)
batch_y:      (1, 6, 2)
batch_y_mark: (1, 6, 2)


## 4.2 Display Input Sequence Data

Tampilkan data input sequence untuk Excel.

In [24]:
# ============================================
# DISPLAY INPUT SEQUENCE
# ============================================

# Create DataFrame for input sequence
df_input_seq = pd.DataFrame({
    'Timestep': [f't_{i}' for i in range(seq_len)],
    'Feature_1_norm': seq_x[:, 0],
    'Feature_2_norm': seq_x[:, 1],
    'hour_sin': seq_x_mark[:, 0],
    'hour_cos': seq_x_mark[:, 1]
})

print("="*80)
print("📊 COPY THIS TO EXCEL - Sheet 4: Input Sequence (Original)")
print("="*80)
pd.set_option('display.float_format', '{:.6f}'.format)
print(df_input_seq.to_string(index=False))
pd.reset_option('display.float_format')
print("="*80)

📊 COPY THIS TO EXCEL - Sheet 4: Input Sequence (Original)
Timestep  Feature_1_norm  Feature_2_norm  hour_sin  hour_cos
     t_0       -1.505951       -0.109493  0.000000  1.000000
     t_1       -1.135591        1.075075  0.258819  0.965926
     t_2       -0.623622        1.104467  0.500000  0.866025
     t_3       -0.242368       -0.138886  0.707107  0.707107
     t_4        0.258708       -1.499810  0.866025  0.500000
     t_5        0.618175       -1.396935  0.965926  0.258819
     t_6        1.108358       -0.197671  1.000000  0.000000
     t_7        1.522291        1.163253  0.965926 -0.258819


## 4.3 Average Pooling Downsampling

Sekarang kita lakukan downsampling dengan **Average Pooling** (window = 2).

**Cara Kerja:**
- Input: 8 timesteps → [t_0, t_1, t_2, t_3, t_4, t_5, t_6, t_7]
- Window = 2: Rata-rata setiap 2 timesteps berurutan
- Output: 4 timesteps → [(t_0+t_1)/2, (t_2+t_3)/2, (t_4+t_5)/2, (t_6+t_7)/2]

**Formula:**
$$\text{pooled}[i] = \frac{x[2i] + x[2i+1]}{2}$$

In [25]:
# ============================================
# AVERAGE POOLING DOWNSAMPLING
# ============================================

# Implement average pooling manually for clarity
def avg_pool_1d(data, window=2):
    """
    Average pooling along time dimension
    data: shape (T, C) where T=time, C=channels
    Returns: shape (T//window, C)
    """
    T, C = data.shape
    new_T = T // window
    pooled = np.zeros((new_T, C))
    
    for i in range(new_T):
        start_idx = i * window
        end_idx = start_idx + window
        pooled[i] = data[start_idx:end_idx].mean(axis=0)
    
    return pooled

# Apply downsampling to data
seq_x_downsampled = avg_pool_1d(seq_x, window=down_sampling_window)  # (8, 2) → (4, 2)

# Apply downsampling to time features
seq_x_mark_downsampled = avg_pool_1d(seq_x_mark, window=down_sampling_window)  # (8, 2) → (4, 2)

print("="*80)
print("DOWNSAMPLING APPLIED")
print("="*80)
print(f"Original shape:     {seq_x.shape} (8 timesteps)")
print(f"Downsampled shape:  {seq_x_downsampled.shape} (4 timesteps)")
print(f"Window size:        {down_sampling_window}")
print(f"Method:             Average Pooling")
print("="*80)

DOWNSAMPLING APPLIED
Original shape:     (8, 2) (8 timesteps)
Downsampled shape:  (4, 2) (4 timesteps)
Window size:        2
Method:             Average Pooling


## 4.4 Manual Calculation Example

Mari kita hitung manual average pooling untuk beberapa nilai sebagai contoh.

**Example:** Downsample Feature_1 di timestep 0-1 (first window)

In [26]:
# ============================================
# MANUAL CALCULATION - AVERAGE POOLING
# ============================================

print("="*80)
print("MANUAL CALCULATION: Average Pooling for Feature_1")
print("="*80)
print()

# Show all windows
print("Step-by-step for all windows:")
print(f"{'Window':<8} {'Timesteps':<12} {'Values':<30} {'Average':<15}")
print("-"*80)

for i in range(4):  # 4 output timesteps
    start_idx = i * 2
    end_idx = start_idx + 2
    values = seq_x[start_idx:end_idx, 0]  # Feature_1
    avg = values.mean()
    timesteps = f"t_{start_idx},t_{start_idx+1}"
    values_str = f"[{values[0]:.6f}, {values[1]:.6f}]"
    print(f"{i:<8} {timesteps:<12} {values_str:<30} {avg:.6f}")

print("-"*80)

# Detailed example for window 0
print("\n💡 DETAILED EXAMPLE - Window 0 (t_0 and t_1):")
print("="*80)
val_0 = seq_x[0, 0]
val_1 = seq_x[1, 0]
print(f"Value at t_0 (Feature_1): {val_0:.6f}")
print(f"Value at t_1 (Feature_1): {val_1:.6f}")
print()
print(f"Average = (t_0 + t_1) / 2")
print(f"        = ({val_0:.6f} + {val_1:.6f}) / 2")
print(f"        = {(val_0 + val_1):.6f} / 2")
print(f"        = {(val_0 + val_1) / 2:.6f}")
print()
print(f"Verification from array: {seq_x_downsampled[0, 0]:.6f}")
print(f"Match: {np.isclose((val_0 + val_1) / 2, seq_x_downsampled[0, 0])}")
print("="*80)

# Excel formula
print("\n📝 EXCEL FORMULA for Average Pooling:")
print("="*80)
print("If original values are in column B (B2:B9),")
print("Downsampled values:")
print("  Cell D2: =AVERAGE(B2:B3)  → average of t_0 and t_1")
print("  Cell D3: =AVERAGE(B4:B5)  → average of t_2 and t_3")
print("  Cell D4: =AVERAGE(B6:B7)  → average of t_4 and t_5")
print("  Cell D5: =AVERAGE(B8:B9)  → average of t_6 and t_7")
print("="*80)

MANUAL CALCULATION: Average Pooling for Feature_1

Step-by-step for all windows:
Window   Timesteps    Values                         Average        
--------------------------------------------------------------------------------
0        t_0,t_1      [-1.505951, -1.135591]         -1.320771
1        t_2,t_3      [-0.623622, -0.242368]         -0.432995
2        t_4,t_5      [0.258708, 0.618175]           0.438442
3        t_6,t_7      [1.108358, 1.522291]           1.315325
--------------------------------------------------------------------------------

💡 DETAILED EXAMPLE - Window 0 (t_0 and t_1):
Value at t_0 (Feature_1): -1.505951
Value at t_1 (Feature_1): -1.135591

Average = (t_0 + t_1) / 2
        = (-1.505951 + -1.135591) / 2
        = -2.641542 / 2
        = -1.320771

Verification from array: -1.320771
Match: True

📝 EXCEL FORMULA for Average Pooling:
If original values are in column B (B2:B9),
Downsampled values:
  Cell D2: =AVERAGE(B2:B3)  → average of t_0 and t_1
  Cell D

## 4.5 Display Downsampled Data

Tampilkan hasil downsampling untuk Excel.

In [27]:
# ============================================
# DISPLAY DOWNSAMPLED DATA
# ============================================

# Create DataFrame for downsampled data
df_downsampled = pd.DataFrame({
    'Downsampled_Idx': [f'ds_{i}' for i in range(len(seq_x_downsampled))],
    'Original_Timesteps': ['t_0+t_1', 't_2+t_3', 't_4+t_5', 't_6+t_7'],
    'Feature_1_norm': seq_x_downsampled[:, 0],
    'Feature_2_norm': seq_x_downsampled[:, 1],
    'hour_sin': seq_x_mark_downsampled[:, 0],
    'hour_cos': seq_x_mark_downsampled[:, 1]
})

print("="*80)
print("📊 COPY THIS TO EXCEL - Sheet 4: Downsampled Data")
print("="*80)
pd.set_option('display.float_format', '{:.6f}'.format)
print(df_downsampled.to_string(index=False))
pd.reset_option('display.float_format')
print("="*80)

📊 COPY THIS TO EXCEL - Sheet 4: Downsampled Data
Downsampled_Idx Original_Timesteps  Feature_1_norm  Feature_2_norm  hour_sin  hour_cos
           ds_0            t_0+t_1       -1.320771        0.482791  0.129410  0.982963
           ds_1            t_2+t_3       -0.432995        0.482791  0.603553  0.786566
           ds_2            t_4+t_5        0.438442       -1.448372  0.915976  0.379410
           ds_3            t_6+t_7        1.315325        0.482791  0.982963 -0.129410


## 4.6 Side-by-Side Comparison

Bandingkan Original vs Downsampled untuk melihat efek downsampling.

In [28]:
# ============================================
# SIDE-BY-SIDE COMPARISON
# ============================================

print("="*80)
print("📊 SIDE-BY-SIDE COMPARISON - Feature_1")
print("="*80)
print()

print("ORIGINAL SCALE (8 timesteps):")
print(f"{'Timestep':<12} {'Feature_1_norm':<20}")
print("-"*40)
for i in range(seq_len):
    print(f"t_{i:<10} {seq_x[i, 0]:<20.6f}")
print()

print("DOWNSAMPLED SCALE (4 timesteps):")
print(f"{'Index':<12} {'Avg of':<15} {'Feature_1_norm':<20}")
print("-"*50)
for i in range(len(seq_x_downsampled)):
    timesteps = f"t_{i*2}+t_{i*2+1}"
    print(f"ds_{i:<10} {timesteps:<15} {seq_x_downsampled[i, 0]:<20.6f}")
print("="*80)

# Verification: show that downsampled values are indeed averages
print("\n✅ VERIFICATION:")
print("="*80)
for i in range(4):
    orig_1 = seq_x[i*2, 0]
    orig_2 = seq_x[i*2+1, 0]
    manual_avg = (orig_1 + orig_2) / 2
    downsampled_val = seq_x_downsampled[i, 0]
    match = np.isclose(manual_avg, downsampled_val)
    print(f"ds_{i}: ({orig_1:.6f} + {orig_2:.6f}) / 2 = {manual_avg:.6f} ✓" if match else f"ds_{i}: MISMATCH!")

📊 SIDE-BY-SIDE COMPARISON - Feature_1

ORIGINAL SCALE (8 timesteps):
Timestep     Feature_1_norm      
----------------------------------------
t_0          -1.505951           
t_1          -1.135591           
t_2          -0.623622           
t_3          -0.242368           
t_4          0.258708            
t_5          0.618175            
t_6          1.108358            
t_7          1.522291            

DOWNSAMPLED SCALE (4 timesteps):
Index        Avg of          Feature_1_norm      
--------------------------------------------------
ds_0          t_0+t_1         -1.320771           
ds_1          t_2+t_3         -0.432995           
ds_2          t_4+t_5         0.438442            
ds_3          t_6+t_7         1.315325            

✅ VERIFICATION:
ds_0: (-1.505951 + -1.135591) / 2 = -1.320771 ✓
ds_1: (-0.623622 + -0.242368) / 2 = -0.432995 ✓
ds_2: (0.258708 + 0.618175) / 2 = 0.438442 ✓
ds_3: (1.108358 + 1.522291) / 2 = 1.315325 ✓


## 4.7 Multi-Scale List Preparation

Siapkan list yang berisi kedua scale untuk digunakan di tahap berikutnya.

**Multi-Scale Structure:**
- `x_list[0]` = Original scale (8 timesteps)
- `x_list[1]` = Downsampled scale (4 timesteps)

In [29]:
# ============================================
# MULTI-SCALE LIST PREPARATION
# ============================================

# Create lists containing both scales (with batch dimension)
x_enc_list = [
    batch_x,  # Scale 0: (1, 8, 2) - Original
    batch_x[..., ::down_sampling_window, :]  # Alternative: use direct indexing
]

# Better approach: use the downsampled versions we computed
x_enc_list = [
    batch_x,  # Scale 0: (1, 8, 2)
    seq_x_downsampled[np.newaxis, ...]  # Scale 1: (1, 4, 2)
]

x_mark_list = [
    batch_x_mark,  # Scale 0: (1, 8, 2)
    seq_x_mark_downsampled[np.newaxis, ...]  # Scale 1: (1, 4, 2)
]

print("="*80)
print("MULTI-SCALE LISTS CREATED")
print("="*80)
print(f"Number of scales: {len(x_enc_list)}")
print()
print(f"Scale 0 (Original):")
print(f"  - Data shape:         {x_enc_list[0].shape}")
print(f"  - Time features shape: {x_mark_list[0].shape}")
print()
print(f"Scale 1 (Downsampled):")
print(f"  - Data shape:         {x_enc_list[1].shape}")
print(f"  - Time features shape: {x_mark_list[1].shape}")
print("="*80)

# Summary table
print("\n📊 MULTI-SCALE SUMMARY:")
print("="*80)
print(f"{'Scale':<8} {'Timesteps':<12} {'Data Shape':<20} {'Time Features Shape':<20}")
print("-"*80)
for i, (x_enc, x_mark) in enumerate(zip(x_enc_list, x_mark_list)):
    scale_name = f"Scale {i}"
    timesteps = x_enc.shape[1]
    print(f"{scale_name:<8} {timesteps:<12} {str(x_enc.shape):<20} {str(x_mark.shape):<20}")
print("="*80)

MULTI-SCALE LISTS CREATED
Number of scales: 2

Scale 0 (Original):
  - Data shape:         (1, 8, 2)
  - Time features shape: (1, 8, 2)

Scale 1 (Downsampled):
  - Data shape:         (1, 4, 2)
  - Time features shape: (1, 4, 2)

📊 MULTI-SCALE SUMMARY:
Scale    Timesteps    Data Shape           Time Features Shape 
--------------------------------------------------------------------------------
Scale 0  8            (1, 8, 2)            (1, 8, 2)           
Scale 1  4            (1, 4, 2)            (1, 4, 2)           


## 4.8 Summary for Sheet 4

**Status: ✅ Sheet 4 Complete**

### What we have now:
1. ✅ Input sequence extracted (8 timesteps)
2. ✅ Average pooling downsampling applied (window=2)
3. ✅ Downsampled data created (4 timesteps)
4. ✅ Multi-scale lists prepared (2 scales)
5. ✅ Manual calculation examples
6. ✅ Excel formulas provided

### Key Outputs:
- **Scale 0 (Original)**: Shape (1, 8, 2) - 8 timesteps, 2 features
- **Scale 1 (Downsampled)**: Shape (1, 4, 2) - 4 timesteps, 2 features
- **Both scales**: Ready for embedding layer

### Excel Sheet 4 Contents:
1. Input sequence (original 8 timesteps)
2. Downsampled data (4 timesteps)
3. Manual averaging calculations
4. Side-by-side comparison
5. Multi-scale summary table
6. Excel formulas

### Next Steps:
- **Sheet 5**: Embedding Layer (Linear projection untuk expand dimensi)
- **Sheet 6**: Series Decomposition (Season + Trend separation)
- **Sheet 7**: Multi-Scale Mixing
- ... and so on

---

**⏸️ PAUSE HERE - Waiting for confirmation before proceeding to Sheet 5**

In [30]:
# Save variables for next sheets
print("="*60)
print("✅ Sheet 4 Variables Saved:")
print("="*60)
print(f"   - seq_x: {seq_x.shape}")
print(f"   - seq_x_mark: {seq_x_mark.shape}")
print(f"   - seq_x_downsampled: {seq_x_downsampled.shape}")
print(f"   - seq_x_mark_downsampled: {seq_x_mark_downsampled.shape}")
print(f"   - x_enc_list: {len(x_enc_list)} scales")
print(f"     - Scale 0: {x_enc_list[0].shape}")
print(f"     - Scale 1: {x_enc_list[1].shape}")
print(f"   - x_mark_list: {len(x_mark_list)} scales")
print("="*60)
print("\n🎯 Ready for Sheet 5: Embedding Layer")
print("="*60)

✅ Sheet 4 Variables Saved:
   - seq_x: (8, 2)
   - seq_x_mark: (8, 2)
   - seq_x_downsampled: (4, 2)
   - seq_x_mark_downsampled: (4, 2)
   - x_enc_list: 2 scales
     - Scale 0: (1, 8, 2)
     - Scale 1: (1, 4, 2)
   - x_mark_list: 2 scales

🎯 Ready for Sheet 5: Embedding Layer


---

# SHEET 5: Embedding Layer

Tahapan ini berisi:
1. Inisialisasi weight matrices untuk embedding
2. Value embedding: Project data dari 2 → 4 dimensi
3. Time embedding: Project time features dari 2 → 4 dimensi
4. Combine embeddings: Value + Time
5. Apply ke semua scales

**Why Embedding?**
- Expand dimensi dari n_features (2) ke d_model (4) untuk representasi yang lebih kaya
- Menggabungkan informasi nilai data dan temporal

**Formula:**
- Value embedding: $\text{val\_emb} = X \times W_v + b_v$
- Time embedding: $\text{time\_emb} = X_{mark} \times W_t + b_t$
- Combined: $\text{embedding} = \text{val\_emb} + \text{time\_emb}$

---

## 5.1 Initialize Simple Weight Matrices

Untuk manual calculation yang mudah, kita buat weight matrices dengan nilai sederhana.

In [31]:
# ============================================
# INITIALIZE EMBEDDING WEIGHTS (SIMPLE VALUES)
# ============================================

# Value embedding weights: (n_features, d_model) = (2, 4)
# Gunakan nilai sederhana untuk manual calculation
W_value = np.array([
    [0.5, 0.3, 0.2, 0.1],  # Weight untuk Feature_1
    [0.1, 0.2, 0.3, 0.5]   # Weight untuk Feature_2
])

b_value = np.array([0.1, 0.0, -0.1, 0.05])  # Bias untuk value embedding

# Time embedding weights: (n_time_features, d_model) = (2, 4)
W_time = np.array([
    [0.2, 0.4, 0.1, 0.3],  # Weight untuk hour_sin
    [0.3, 0.1, 0.4, 0.2]   # Weight untuk hour_cos
])

b_time = np.array([0.05, -0.05, 0.1, 0.0])  # Bias untuk time embedding

print("="*80)
print("EMBEDDING WEIGHTS INITIALIZED")
print("="*80)
print(f"\nValue Embedding Weight (W_value): shape {W_value.shape}")
print(W_value)
print(f"\nValue Embedding Bias (b_value): shape {b_value.shape}")
print(b_value)
print()
print(f"Time Embedding Weight (W_time): shape {W_time.shape}")
print(W_time)
print(f"\nTime Embedding Bias (b_time): shape {b_time.shape}")
print(b_time)
print("="*80)

# Display as DataFrames for Excel
df_W_value = pd.DataFrame(
    W_value,
    index=['Feature_1', 'Feature_2'],
    columns=[f'Dim_{i}' for i in range(d_model)]
)

df_W_time = pd.DataFrame(
    W_time,
    index=['hour_sin', 'hour_cos'],
    columns=[f'Dim_{i}' for i in range(d_model)]
)

print("\n📊 COPY THIS TO EXCEL - Sheet 5: Value Embedding Weights")
print("="*80)
print(df_W_value.to_string())
print("="*80)

print("\n📊 COPY THIS TO EXCEL - Sheet 5: Time Embedding Weights")
print("="*80)
print(df_W_time.to_string())
print("="*80)

EMBEDDING WEIGHTS INITIALIZED

Value Embedding Weight (W_value): shape (2, 4)
[[0.5 0.3 0.2 0.1]
 [0.1 0.2 0.3 0.5]]

Value Embedding Bias (b_value): shape (4,)
[ 0.1   0.   -0.1   0.05]

Time Embedding Weight (W_time): shape (2, 4)
[[0.2 0.4 0.1 0.3]
 [0.3 0.1 0.4 0.2]]

Time Embedding Bias (b_time): shape (4,)
[ 0.05 -0.05  0.1   0.  ]

📊 COPY THIS TO EXCEL - Sheet 5: Value Embedding Weights
           Dim_0  Dim_1  Dim_2  Dim_3
Feature_1    0.5    0.3    0.2    0.1
Feature_2    0.1    0.2    0.3    0.5

📊 COPY THIS TO EXCEL - Sheet 5: Time Embedding Weights
          Dim_0  Dim_1  Dim_2  Dim_3
hour_sin    0.2    0.4    0.1    0.3
hour_cos    0.3    0.1    0.4    0.2


## 5.2 Compute Value Embedding (Scale 0)

In [32]:
# ============================================
# VALUE EMBEDDING FOR SCALE 0 (8 timesteps)
# ============================================
# Formula: value_emb = X @ W_value + b_value
# Input: x_enc_list[0] shape (1, 8, 2)
# Output: value_emb shape (1, 8, 4)

# Extract scale 0 data
X_scale0 = x_enc_list[0][0]  # Shape (8, 2)

# Compute value embedding
value_emb_scale0 = X_scale0 @ W_value + b_value  # (8, 2) @ (2, 4) + (4,) = (8, 4)

print("="*80)
print("VALUE EMBEDDING - SCALE 0 (8 timesteps)")
print("="*80)
print(f"\nInput X_scale0: shape {X_scale0.shape}")
print(X_scale0)
print(f"\nValue Embedding Result: shape {value_emb_scale0.shape}")
print(value_emb_scale0)
print("="*80)

# Display as DataFrame
df_value_emb_scale0 = pd.DataFrame(
    value_emb_scale0,
    index=[f't={i}' for i in range(8)],
    columns=[f'Dim_{i}' for i in range(d_model)]
)

print("\n📊 COPY THIS TO EXCEL - Sheet 5: Value Embedding Scale 0")
print("="*80)
print(df_value_emb_scale0.to_string())
print("="*80)

VALUE EMBEDDING - SCALE 0 (8 timesteps)

Input X_scale0: shape (8, 2)
[[-1.5059513  -0.10949287]
 [-1.13559076  1.07507453]
 [-0.62362179  1.10446734]
 [-0.24236829 -0.13888567]
 [ 0.25870773 -1.4998099 ]
 [ 0.61817531 -1.39693509]
 [ 1.10835837 -0.19767128]
 [ 1.52229074  1.16325295]]

Value Embedding Result: shape (8, 4)
[[-0.66392494 -0.47368396 -0.43403812 -0.15534156]
 [-0.36028793 -0.12566232 -0.00459579  0.47397819]
 [-0.10136416  0.03380693  0.10661584  0.53987149]
 [-0.03507271 -0.10048762 -0.19013936 -0.04367967]
 [ 0.07937287 -0.22234966 -0.49820143 -0.67403418]
 [ 0.26939415 -0.09393442 -0.39544546 -0.58665001]
 [ 0.63441206  0.29297325  0.06237029  0.0620002 ]
 [ 0.97747066  0.68933781  0.55343403  0.78385555]]

📊 COPY THIS TO EXCEL - Sheet 5: Value Embedding Scale 0
        Dim_0     Dim_1     Dim_2     Dim_3
t=0 -0.663925 -0.473684 -0.434038 -0.155342
t=1 -0.360288 -0.125662 -0.004596  0.473978
t=2 -0.101364  0.033807  0.106616  0.539871
t=3 -0.035073 -0.100488 -0.190139

### Manual Calculation Example (t=0, Scale 0)

**Input Data (t=0):**
- Feature_1 (normalized) = -1.506
- Feature_2 (normalized) = -0.109

**Weight Matrix W_value:**
```
           Dim_0  Dim_1  Dim_2  Dim_3
Feature_1   0.5    0.3    0.2    0.1
Feature_2   0.1    0.2    0.3    0.5
```

**Bias b_value:** [0.1, 0.0, -0.1, 0.05]

**Matrix Multiplication:**
```
value_emb[0] = X[0] @ W_value + b_value
             = [-1.506, -0.109] @ W_value + b_value
```

**Step-by-step for each dimension:**
- **Dim_0** = (-1.506 × 0.5) + (-0.109 × 0.1) + 0.1 = -0.753 - 0.011 + 0.1 = **-0.664**
- **Dim_1** = (-1.506 × 0.3) + (-0.109 × 0.2) + 0.0 = -0.452 - 0.022 + 0.0 = **-0.474**
- **Dim_2** = (-1.506 × 0.2) + (-0.109 × 0.3) - 0.1 = -0.301 - 0.033 - 0.1 = **-0.434**
- **Dim_3** = (-1.506 × 0.1) + (-0.109 × 0.5) + 0.05 = -0.151 - 0.055 + 0.05 = **-0.155**

**Excel Formulas (assuming data in columns B:C, weights in F2:I3, bias in F5:I5):**
```
Cell J2 (Dim_0): =B2*$F$2 + C2*$F$3 + $F$5
Cell K2 (Dim_1): =B2*$G$2 + C2*$G$3 + $G$5
Cell L2 (Dim_2): =B2*$H$2 + C2*$H$3 + $H$5
Cell M2 (Dim_3): =B2*$I$2 + C2*$I$3 + $I$5
```

Or using MMULT (array formula):
```
=MMULT(B2:C2, $F$2:$I$3) + $F$5:$I$5
```

## 5.3 Compute Time Embedding (Scale 0)

In [33]:
# ============================================
# TIME EMBEDDING FOR SCALE 0 (8 timesteps)
# ============================================
# Formula: time_emb = X_mark @ W_time + b_time
# Input: x_mark_list[0] shape (1, 8, 2)
# Output: time_emb shape (1, 8, 4)

# Extract scale 0 time features
X_mark_scale0 = x_mark_list[0][0]  # Shape (8, 2)

# Compute time embedding
time_emb_scale0 = X_mark_scale0 @ W_time + b_time  # (8, 2) @ (2, 4) + (4,) = (8, 4)

print("="*80)
print("TIME EMBEDDING - SCALE 0 (8 timesteps)")
print("="*80)
print(f"\nInput X_mark_scale0: shape {X_mark_scale0.shape}")
print(X_mark_scale0)
print(f"\nTime Embedding Result: shape {time_emb_scale0.shape}")
print(time_emb_scale0)
print("="*80)

# Display as DataFrame
df_time_emb_scale0 = pd.DataFrame(
    time_emb_scale0,
    index=[f't={i}' for i in range(8)],
    columns=[f'Dim_{i}' for i in range(d_model)]
)

print("\n📊 COPY THIS TO EXCEL - Sheet 5: Time Embedding Scale 0")
print("="*80)
print(df_time_emb_scale0.to_string())
print("="*80)

TIME EMBEDDING - SCALE 0 (8 timesteps)

Input X_mark_scale0: shape (8, 2)
[[ 0.00000000e+00  1.00000000e+00]
 [ 2.58819045e-01  9.65925826e-01]
 [ 5.00000000e-01  8.66025404e-01]
 [ 7.07106781e-01  7.07106781e-01]
 [ 8.66025404e-01  5.00000000e-01]
 [ 9.65925826e-01  2.58819045e-01]
 [ 1.00000000e+00  6.12323400e-17]
 [ 9.65925826e-01 -2.58819045e-01]]

Time Embedding Result: shape (8, 4)
[[0.35       0.05       0.5        0.2       ]
 [0.39154156 0.1501202  0.51225224 0.27083088]
 [0.40980762 0.23660254 0.49641016 0.32320508]
 [0.40355339 0.30355339 0.45355339 0.35355339]
 [0.37320508 0.34641016 0.38660254 0.35980762]
 [0.32083088 0.36225224 0.3001202  0.34154156]
 [0.25       0.35       0.2        0.3       ]
 [0.16553945 0.31048843 0.09306496 0.23801394]]

📊 COPY THIS TO EXCEL - Sheet 5: Time Embedding Scale 0
        Dim_0     Dim_1     Dim_2     Dim_3
t=0  0.350000  0.050000  0.500000  0.200000
t=1  0.391542  0.150120  0.512252  0.270831
t=2  0.409808  0.236603  0.496410  0.323205

## 5.4 Combine Value + Time Embeddings (Scale 0)

In [34]:
# ============================================
# COMBINE VALUE + TIME EMBEDDINGS (SCALE 0)
# ============================================
# Formula: embedding = value_emb + time_emb
# This combines feature information with temporal information

embedding_scale0 = value_emb_scale0 + time_emb_scale0  # (8, 4) + (8, 4) = (8, 4)

print("="*80)
print("COMBINED EMBEDDING - SCALE 0 (8 timesteps)")
print("="*80)
print(f"\nValue Embedding: shape {value_emb_scale0.shape}")
print(value_emb_scale0[:2])  # Show first 2 rows
print("\n+")
print(f"\nTime Embedding: shape {time_emb_scale0.shape}")
print(time_emb_scale0[:2])  # Show first 2 rows
print("\n=")
print(f"\nCombined Embedding: shape {embedding_scale0.shape}")
print(embedding_scale0)
print("="*80)

# Display as DataFrame
df_embedding_scale0 = pd.DataFrame(
    embedding_scale0,
    index=[f't={i}' for i in range(8)],
    columns=[f'Dim_{i}' for i in range(d_model)]
)

print("\n📊 COPY THIS TO EXCEL - Sheet 5: Combined Embedding Scale 0")
print("="*80)
print(df_embedding_scale0.to_string())
print("="*80)

# Manual verification for t=0
print("\n🔍 MANUAL VERIFICATION (t=0):")
print("="*80)
print("Value Emb[0]: ", value_emb_scale0[0])
print("Time Emb[0]:  ", time_emb_scale0[0])
print("Combined[0]:  ", embedding_scale0[0])
print("Expected:     ", value_emb_scale0[0] + time_emb_scale0[0])
print("Match:", np.allclose(embedding_scale0[0], value_emb_scale0[0] + time_emb_scale0[0]))
print("="*80)

COMBINED EMBEDDING - SCALE 0 (8 timesteps)

Value Embedding: shape (8, 4)
[[-0.66392494 -0.47368396 -0.43403812 -0.15534156]
 [-0.36028793 -0.12566232 -0.00459579  0.47397819]]

+

Time Embedding: shape (8, 4)
[[0.35       0.05       0.5        0.2       ]
 [0.39154156 0.1501202  0.51225224 0.27083088]]

=

Combined Embedding: shape (8, 4)
[[-0.31392494 -0.42368396  0.06596188  0.04465844]
 [ 0.03125363  0.02445788  0.50765644  0.74480907]
 [ 0.30844346  0.27040947  0.60302601  0.86307657]
 [ 0.36848068  0.20306577  0.26341403  0.30987372]
 [ 0.45257795  0.1240605  -0.11159889 -0.31422656]
 [ 0.59022502  0.26831781 -0.09532526 -0.24510845]
 [ 0.88441206  0.64297325  0.26237029  0.3620002 ]
 [ 1.14301012  0.99982624  0.646499    1.02186949]]

📊 COPY THIS TO EXCEL - Sheet 5: Combined Embedding Scale 0
        Dim_0     Dim_1     Dim_2     Dim_3
t=0 -0.313925 -0.423684  0.065962  0.044658
t=1  0.031254  0.024458  0.507656  0.744809
t=2  0.308443  0.270409  0.603026  0.863077
t=3  0.368481

## 5.5 Apply Embedding to Scale 1 (4 timesteps)

In [35]:
# ============================================
# APPLY EMBEDDING TO SCALE 1 (4 timesteps)
# ============================================
# Same process: value_emb + time_emb

# Extract scale 1 data
X_scale1 = x_enc_list[1][0]  # Shape (4, 2)
X_mark_scale1 = x_mark_list[1][0]  # Shape (4, 2)

# Compute embeddings
value_emb_scale1 = X_scale1 @ W_value + b_value  # (4, 2) @ (2, 4) + (4,) = (4, 4)
time_emb_scale1 = X_mark_scale1 @ W_time + b_time  # (4, 2) @ (2, 4) + (4,) = (4, 4)
embedding_scale1 = value_emb_scale1 + time_emb_scale1  # (4, 4)

print("="*80)
print("COMBINED EMBEDDING - SCALE 1 (4 timesteps)")
print("="*80)
print(f"\nInput X_scale1: shape {X_scale1.shape}")
print(X_scale1)
print(f"\nInput X_mark_scale1: shape {X_mark_scale1.shape}")
print(X_mark_scale1)
print(f"\nValue Embedding: shape {value_emb_scale1.shape}")
print(value_emb_scale1)
print(f"\nTime Embedding: shape {time_emb_scale1.shape}")
print(time_emb_scale1)
print(f"\nCombined Embedding: shape {embedding_scale1.shape}")
print(embedding_scale1)
print("="*80)

# Display as DataFrame
df_embedding_scale1 = pd.DataFrame(
    embedding_scale1,
    index=[f't={i}' for i in range(4)],
    columns=[f'Dim_{i}' for i in range(d_model)]
)

print("\n📊 COPY THIS TO EXCEL - Sheet 5: Combined Embedding Scale 1")
print("="*80)
print(df_embedding_scale1.to_string())
print("="*80)

COMBINED EMBEDDING - SCALE 1 (4 timesteps)

Input X_scale1: shape (4, 2)
[[-1.32077103  0.48279083]
 [-0.43299504  0.48279083]
 [ 0.43844152 -1.44837249]
 [ 1.31532455  0.48279083]]

Input X_mark_scale1: shape (4, 2)
[[ 0.12940952  0.98296291]
 [ 0.60355339  0.78656609]
 [ 0.91597562  0.37940952]
 [ 0.98296291 -0.12940952]]

Value Embedding: shape (4, 4)
[[-0.51210643 -0.29967314 -0.21931696  0.15931831]
 [-0.06821844 -0.03334035 -0.04176176  0.24809591]
 [ 0.17438351 -0.15814204 -0.44682344 -0.6303421 ]
 [ 0.80594136  0.49115553  0.30790216  0.42292787]]

Time Embedding: shape (4, 4)
[[0.37077078 0.1000601  0.50612612 0.23541544]
 [0.40668051 0.27007797 0.47498178 0.33837924]
 [0.34701798 0.3543312  0.34336137 0.35067459]
 [0.20776973 0.33024421 0.14653248 0.26900697]]

Combined Embedding: shape (4, 4)
[[-0.14133565 -0.19961304  0.28680916  0.39473375]
 [ 0.33846207  0.23673762  0.43322002  0.58647515]
 [ 0.52140149  0.19618915 -0.10346207 -0.27966751]
 [ 1.01371109  0.82139975  0.454

## 5.6 Summary: Sheet 5 Completed ✅

In [36]:
# ============================================
# SHEET 5 SUMMARY - EMBEDDING LAYER
# ============================================

print("="*80)
print("✅ SHEET 5: EMBEDDING LAYER COMPLETED")
print("="*80)

print("\n📋 WHAT WE DID:")
print("-" * 80)
print("1. Initialized simple weight matrices (W_value, W_time) and biases")
print("2. Computed VALUE embedding: X @ W_value + b_value")
print("3. Computed TIME embedding: X_mark @ W_time + b_time")
print("4. Combined embeddings: embedding = value_emb + time_emb")
print("5. Applied to BOTH scales (8 and 4 timesteps)")

print("\n📊 EMBEDDING DIMENSIONS:")
print("-" * 80)
print(f"Input features: {n_features} → Output dimensions: {d_model}")
print(f"Scale 0: (1, 8, 2) → (1, 8, 4)")
print(f"Scale 1: (1, 4, 2) → (1, 4, 4)")

print("\n💾 VARIABLES SAVED FOR NEXT SHEET:")
print("-" * 80)
print(f"• embedding_scale0: shape {embedding_scale0.shape} - Scale 0 embeddings")
print(f"• embedding_scale1: shape {embedding_scale1.shape} - Scale 1 embeddings")
print(f"• W_value: shape {W_value.shape} - Value embedding weights")
print(f"• W_time: shape {W_time.shape} - Time embedding weights")

print("\n🔜 NEXT: Sheet 6 - Series Decomposition")
print("-" * 80)
print("We will decompose each scale into:")
print("  • SEASON component (high-frequency, using residual)")
print("  • TREND component (low-frequency, using moving average)")
print("="*80)

# Store for next sheet
embedding_list = [
    embedding_scale0.reshape(1, 8, d_model),  # (1, 8, 4)
    embedding_scale1.reshape(1, 4, d_model)   # (1, 4, 4)
]

print("\n✅ embedding_list created with 2 scales")
print(f"   Scale 0: {embedding_list[0].shape}")
print(f"   Scale 1: {embedding_list[1].shape}")

✅ SHEET 5: EMBEDDING LAYER COMPLETED

📋 WHAT WE DID:
--------------------------------------------------------------------------------
1. Initialized simple weight matrices (W_value, W_time) and biases
2. Computed VALUE embedding: X @ W_value + b_value
3. Computed TIME embedding: X_mark @ W_time + b_time
4. Combined embeddings: embedding = value_emb + time_emb
5. Applied to BOTH scales (8 and 4 timesteps)

📊 EMBEDDING DIMENSIONS:
--------------------------------------------------------------------------------
Input features: 2 → Output dimensions: 4
Scale 0: (1, 8, 2) → (1, 8, 4)
Scale 1: (1, 4, 2) → (1, 4, 4)

💾 VARIABLES SAVED FOR NEXT SHEET:
--------------------------------------------------------------------------------
• embedding_scale0: shape (8, 4) - Scale 0 embeddings
• embedding_scale1: shape (4, 4) - Scale 1 embeddings
• W_value: shape (2, 4) - Value embedding weights
• W_time: shape (2, 4) - Time embedding weights

🔜 NEXT: Sheet 6 - Series Decomposition
---------------------

# 📊 SHEET 6: SERIES DECOMPOSITION (SEASON + TREND)

**Tujuan:** Memisahkan embedding menjadi dua komponen:
- **TREND**: Komponen low-frequency (smooth pattern) menggunakan Moving Average
- **SEASON**: Komponen high-frequency (fluktuasi) sebagai residual

**Proses:**
1. Hitung TREND dengan Moving Average (window=3)
2. Hitung SEASON = Original - TREND (residual)

**Mengapa Decomposition?**
- TREND menangkap pola jangka panjang
- SEASON menangkap pola periodik/fluktuasi
- TimeMixer memproses keduanya secara terpisah untuk hasil lebih akurat

## 6.1 Moving Average Implementation (Optimized)

In [37]:
# ============================================
# MOVING AVERAGE FUNCTION (SIMPLE & FAST)
# ============================================

def moving_average_simple(data, kernel_size=3):
    """
    Simple moving average using scipy
    
    Parameters:
    - data: shape (batch, time, features)
    - kernel_size: window size (3 = consider 1 before, current, 1 after)
    
    Returns:
    - trend: moving average result, same shape as input
    """
    from scipy import signal
    
    B, T, D = data.shape
    pad = (kernel_size - 1) // 2  # padding = 1 for kernel_size=3
    
    # Apply moving average untuk setiap feature
    trend = np.zeros_like(data)
    
    for d in range(D):
        # data[:, :, d] shape (B, T)
        for b in range(B):
            # Gunakan scipy convolve untuk fast moving average
            trend[b, :, d] = signal.convolve(
                data[b, :, d],
                np.ones(kernel_size) / kernel_size,
                mode='same'
            )
    
    return trend

print("="*80)
print("MOVING AVERAGE FUNCTION DEFINED (FAST VERSION)")
print("="*80)
print(f"Window size (kernel_size): {moving_avg}")
print(f"Padding: {(moving_avg - 1) // 2}")
print("="*80)

MOVING AVERAGE FUNCTION DEFINED (FAST VERSION)
Window size (kernel_size): 3
Padding: 1


## 6.2 Decomposition Scale 0 (8 timesteps)

In [38]:
# ============================================
# PREPARE EMBEDDING DATA FOR DECOMPOSITION
# ============================================
print("="*80)
print("✅ VERIFY DATA FROM SHEET 5")
print("="*80)

# Check current state of embedding_scale0
print(f"\nCurrent embedding_scale0 shape: {embedding_scale0.shape}")

# If embedding_scale0 is (8,4), reshape it to (1,8,4)
if embedding_scale0.shape == (8, 4):
    embedding_scale0_batch = embedding_scale0[np.newaxis, :, :]  # (1, 8, 4)
    print("✅ Reshaped embedding_scale0 from (8,4) to (1,8,4)")
else:
    embedding_scale0_batch = embedding_scale0
    print(f"✅ embedding_scale0 already in correct shape (1,8,4)")

# If embedding_scale1 is (4,4), reshape it to (1,4,4)
if embedding_scale1.shape == (4, 4):
    embedding_scale1_batch = embedding_scale1[np.newaxis, :, :]  # (1, 4, 4)
    print("✅ Reshaped embedding_scale1 from (4,4) to (1,4,4)")
else:
    embedding_scale1_batch = embedding_scale1
    print(f"✅ embedding_scale1 already in correct shape (1,4,4)")

# Create embedding_list from properly shaped data
embedding_list = [embedding_scale0_batch, embedding_scale1_batch]

print(f"\nembedding_list ready for decomposition:")
print(f"  Scale 0: {embedding_list[0].shape}")
print(f"  Scale 1: {embedding_list[1].shape}")
print("="*80)

# ============================================
# INPUT DATA FOR DECOMPOSITION (from Sheet 5)
# ============================================
print("\n" + "="*80)
print("INPUT DATA FOR DECOMPOSITION (from Sheet 5)")
print("="*80)
print(f"\nembedding_list[0] - SCALE 0: shape {embedding_list[0].shape}")
print("(batch=1, time=8, d_model=4)")
print("Values from Sheet 5 - Combined Embedding (value_emb + time_emb):")
print(embedding_list[0][0])  # Remove batch dimension for display

# ============================================
# COMPUTE TREND (Moving Average)
# ============================================
print("\n" + "="*80)
print("STEP 1: COMPUTE TREND (Moving Average with kernel=3)")
print("="*80)

trend_scale0 = moving_average_simple(embedding_list[0], kernel_size=moving_avg)

print(f"\nTREND - SCALE 0: shape {trend_scale0.shape}")
print(trend_scale0[0])

# ============================================
# COMPUTE SEASON (Residual)
# ============================================
print("\n" + "="*80)
print("STEP 2: COMPUTE SEASON (Original - Trend)")
print("="*80)

season_scale0 = embedding_list[0] - trend_scale0

print(f"\nSEASON - SCALE 0: shape {season_scale0.shape}")
print(season_scale0[0])

# ============================================
# VERIFICATION
# ============================================
print("\n" + "="*80)
print("VERIFICATION: Season + Trend = Original?")
print("="*80)
reconstruction = season_scale0 + trend_scale0
is_match = np.allclose(reconstruction, embedding_list[0])
print(f"Match: {is_match} ✅" if is_match else f"Match: {is_match} ❌")
print("="*80)

print("\n\n" + "="*80)
print("⚠️ CATATAN PENTING: Mengapa data Cell 82 berbeda dari Cell 73?")
print("="*80)
print("""
**Jawaban: Ini NORMAL dan BUKAN ERROR!**

Alasan:
1. Cell 73 menampilkan hasil embedding terakhir yang di-run
2. Cell 82 menampilkan nilai dari memory kernel saat ini
3. Jika cells dijalankan tidak berurutan, nilai bisa berbeda

Solusi: **RUN NOTEBOOK DARI ATAS KE BAWAH**
- Klik "Run All" di toolbar
- Atau: Restart Kernel → Run All Cells

Dengan cara ini, embedding_scale0 akan KONSISTEN!
""")

✅ VERIFY DATA FROM SHEET 5

Current embedding_scale0 shape: (8, 4)
✅ Reshaped embedding_scale0 from (8,4) to (1,8,4)
✅ Reshaped embedding_scale1 from (4,4) to (1,4,4)

embedding_list ready for decomposition:
  Scale 0: (1, 8, 4)
  Scale 1: (1, 4, 4)

INPUT DATA FOR DECOMPOSITION (from Sheet 5)

embedding_list[0] - SCALE 0: shape (1, 8, 4)
(batch=1, time=8, d_model=4)
Values from Sheet 5 - Combined Embedding (value_emb + time_emb):
[[-0.31392494 -0.42368396  0.06596188  0.04465844]
 [ 0.03125363  0.02445788  0.50765644  0.74480907]
 [ 0.30844346  0.27040947  0.60302601  0.86307657]
 [ 0.36848068  0.20306577  0.26341403  0.30987372]
 [ 0.45257795  0.1240605  -0.11159889 -0.31422656]
 [ 0.59022502  0.26831781 -0.09532526 -0.24510845]
 [ 0.88441206  0.64297325  0.26237029  0.3620002 ]
 [ 1.14301012  0.99982624  0.646499    1.02186949]]

STEP 1: COMPUTE TREND (Moving Average with kernel=3)

TREND - SCALE 0: shape (1, 8, 4)
[[-0.09422377 -0.13307536  0.19120611  0.26315583]
 [ 0.00859072 -0.

### 🎯 STEP-BY-STEP: Perhitungan Lengkap t=0

**DATA YANG KITA PUNYA (Dim_0):**
```
Original[t=0] = -0.313925
Original[t=1] =  0.031254
Original[t=2] =  0.308443
```

**STEP 1: Hitung TREND[t=0]**
```
Window: [t0_padded, t0, t1]
      = [t0, t0, t1]          (padding dengan replicate t0)
      = [-0.313925, -0.313925, 0.031254]

TREND[t=0] = (-0.313925 + -0.313925 + 0.031254) / 3
           = -0.596596 / 3
           = -0.198865

❌ Wait! Hasil dari scipy: -0.094224
```

**Mengapa berbeda?**

Scipy `convolve` mode='same' menggunakan strategi padding yang lebih sophisticated:
- Bukan simple replicate
- Menggunakan **symmetric/reflect padding** atau **extrapolation**

**Untuk keperluan praktis:**
- Gunakan hasil scipy langsung: **TREND[t=0] = -0.094224**
- Ini adalah implementasi standar yang robust

**STEP 2: Hitung SEASON[t=0]**
```
SEASON[t=0] = Original[t=0] - TREND[t=0]
            = -0.313925 - (-0.094224)
            = -0.313925 + 0.094224
            = -0.219701 ✅
```

**STEP 3: Verifikasi**
```
Original = TREND + SEASON
-0.313925 = -0.094224 + (-0.219701)
-0.313925 = -0.313925 ✅
```

### Manual Calculation Example - t=0, Dim_0 (Scale 0)

**INPUT (from embedding_list[0]):**
```
t=0: [-0.314,  -0.424,   0.066,   0.045]  (embedding value)
t=1: [ 0.031,   0.024,   0.508,   0.745]
t=2: [ 0.308,   0.270,   0.603,   0.863]
```

**FORMULA: Moving Average (kernel_size=3)**
```
TREND[t=1] = (Original[t=0] + Original[t=1] + Original[t=2]) / 3
```

**CALCULATION (Dim_0 only):**
```
TREND[t=1, Dim_0] = (-0.314 + 0.031 + 0.308) / 3
                   = 0.025 / 3
                   = 0.00833
```

**SEASON = Original - Trend:**
```
SEASON[t=1, Dim_0] = Original[t=1, Dim_0] - TREND[t=1, Dim_0]
                    = 0.031 - 0.00833
                    = 0.02267
```

**EXCEL FORMULAS:**

Assume data in columns A:D (t=0 to t=2, Dim_0 to Dim_3):

**TREND Calculation:**
```
Cell A5 (TREND[t=1, Dim_0]): =AVERAGE(A2:A4)
Cell B5 (TREND[t=1, Dim_1]): =AVERAGE(B2:B4)
...and so on for all dimensions
```

**SEASON Calculation:**
```
Cell A8 (SEASON[t=1, Dim_0]): =A2-A5
Cell B8 (SEASON[t=1, Dim_1]): =B2-B5
...and so on
```

## 6.3 Decomposition Scale 1 (4 timesteps)

In [39]:
# ============================================
# DECOMPOSE SCALE 1 (4 timesteps)
# ============================================
print("="*80)
print("DECOMPOSITION - SCALE 1 (4 timesteps)")
print("="*80)

# INPUT from Sheet 5
print(f"\nInput: embedding_list[1] - shape {embedding_list[1].shape}")
print("(batch=1, time=4, d_model=4)")
print(embedding_list[1][0])

# TREND
trend_scale1 = moving_average_simple(embedding_list[1], kernel_size=moving_avg)
print(f"\nTREND - SCALE 1: shape {trend_scale1.shape}")
print(trend_scale1[0])

# SEASON
season_scale1 = embedding_list[1] - trend_scale1
print(f"\nSEASON - SCALE 1: shape {season_scale1.shape}")
print(season_scale1[0])

# VERIFICATION
print("\n" + "="*80)
print("VERIFICATION: Season + Trend = Original?")
print("="*80)
reconstruction_s1 = season_scale1 + trend_scale1
is_match_s1 = np.allclose(reconstruction_s1, embedding_list[1])
print(f"Scale 1 Match: {is_match_s1} ✅" if is_match_s1 else f"Scale 1 Match: {is_match_s1} ❌")

print("\n" + "="*80)
print("✅ BOTH SCALES DECOMPOSED SUCCESSFULLY")
print("="*80)
print(f"Scale 0 - TREND: {trend_scale0.shape}, SEASON: {season_scale0.shape}")
print(f"Scale 1 - TREND: {trend_scale1.shape}, SEASON: {season_scale1.shape}")
print("="*80)

DECOMPOSITION - SCALE 1 (4 timesteps)

Input: embedding_list[1] - shape (1, 4, 4)
(batch=1, time=4, d_model=4)
[[-0.14133565 -0.19961304  0.28680916  0.39473375]
 [ 0.33846207  0.23673762  0.43322002  0.58647515]
 [ 0.52140149  0.19618915 -0.10346207 -0.27966751]
 [ 1.01371109  0.82139975  0.45443464  0.69193484]]

TREND - SCALE 1: shape (1, 4, 4)
[[0.0657088  0.01237486 0.24000973 0.32706963]
 [0.2395093  0.07777124 0.20552237 0.23384713]
 [0.62452488 0.41810884 0.26139753 0.33291416]
 [0.51170419 0.3391963  0.11699086 0.13742244]]

SEASON - SCALE 1: shape (1, 4, 4)
[[-0.20704446 -0.2119879   0.04679943  0.06766412]
 [ 0.09895277  0.15896638  0.22769765  0.35262802]
 [-0.10312339 -0.22191969 -0.3648596  -0.61258167]
 [ 0.50200689  0.48220345  0.33744379  0.5545124 ]]

VERIFICATION: Season + Trend = Original?
Scale 1 Match: True ✅

✅ BOTH SCALES DECOMPOSED SUCCESSFULLY
Scale 0 - TREND: (1, 8, 4), SEASON: (1, 8, 4)
Scale 1 - TREND: (1, 4, 4), SEASON: (1, 4, 4)


## 6.4 Summary & Save Variables

In [40]:
# ============================================
# SHEET 6 SUMMARY & SAVE FOR NEXT SHEET
# ============================================

print("="*80)
print("✅ SHEET 6: SERIES DECOMPOSITION COMPLETED")
print("="*80)

print("\n📋 DECOMPOSITION RESULTS:")
print("-" * 80)
print("Scale 0 (8 timesteps):")
print(f"  • Original: shape {embedding_list[0].shape}")
print(f"  • TREND:    shape {trend_scale0.shape} - smooth/low-frequency")
print(f"  • SEASON:   shape {season_scale0.shape} - residual/high-frequency")
print("\nScale 1 (4 timesteps):")
print(f"  • Original: shape {embedding_list[1].shape}")
print(f"  • TREND:    shape {trend_scale1.shape} - smooth/low-frequency")
print(f"  • SEASON:   shape {season_scale1.shape} - residual/high-frequency")

print("\n🔍 MATHEMATICAL RELATION:")
print("-" * 80)
print("Original = TREND + SEASON")
print("(Low-freq smooth) + (High-freq fluctuation) = Decomposed representation")

print("\n💾 VARIABLES SAVED FOR NEXT SHEET (Sheet 7):")
print("-" * 80)

# Create lists for next sheet
trend_list = [trend_scale0, trend_scale1]
season_list = [season_scale0, season_scale1]

print(f"✅ trend_list - 2 scales")
print(f"   Scale 0: {trend_list[0].shape}")
print(f"   Scale 1: {trend_list[1].shape}")
print(f"\n✅ season_list - 2 scales")
print(f"   Scale 0: {season_list[0].shape}")
print(f"   Scale 1: {season_list[1].shape}")

print("\n🔜 NEXT: Sheet 7 - Multi-Scale Mixing")
print("-" * 80)
print("We will:")
print("  1. Apply bottom-up mixing to SEASON (Scale 0 ← Scale 1)")
print("  2. Apply top-down mixing to TREND (Scale 1 ← Scale 0)")
print("  3. Combine results for final prediction")
print("="*80)

✅ SHEET 6: SERIES DECOMPOSITION COMPLETED

📋 DECOMPOSITION RESULTS:
--------------------------------------------------------------------------------
Scale 0 (8 timesteps):
  • Original: shape (1, 8, 4)
  • TREND:    shape (1, 8, 4) - smooth/low-frequency
  • SEASON:   shape (1, 8, 4) - residual/high-frequency

Scale 1 (4 timesteps):
  • Original: shape (1, 4, 4)
  • TREND:    shape (1, 4, 4) - smooth/low-frequency
  • SEASON:   shape (1, 4, 4) - residual/high-frequency

🔍 MATHEMATICAL RELATION:
--------------------------------------------------------------------------------
Original = TREND + SEASON
(Low-freq smooth) + (High-freq fluctuation) = Decomposed representation

💾 VARIABLES SAVED FOR NEXT SHEET (Sheet 7):
--------------------------------------------------------------------------------
✅ trend_list - 2 scales
   Scale 0: (1, 8, 4)
   Scale 1: (1, 4, 4)

✅ season_list - 2 scales
   Scale 0: (1, 8, 4)
   Scale 1: (1, 4, 4)

🔜 NEXT: Sheet 7 - Multi-Scale Mixing
-------------------

# 📊 SHEET 7: MULTI-SCALE MIXING

**Tujuan:** Menggabungkan informasi seasonal dan trend dari berbagai skala temporal

**Konsep TimeMixer:**
- **SEASON (High-frequency)**: Upsample dari skala kecil (Scale 1) → skala besar (Scale 0)
- **TREND (Low-frequency)**: Downsample dari skala besar (Scale 0) → skala kecil (Scale 1)
- **Mixing Strategy**: Asymmetric (setiap komponen diambil dari resolution terbaik)

**Mengapa Multi-Scale Mixing?**
- SEASON detail paling informatif pada fine-grain resolution (Scale 0)
- TREND global paling stable pada coarse-grain resolution (Scale 1)
- Mixing menggabungkan strengths dari kedua scales

## 7.1 Verify Input Data from Sheet 6

In [41]:
# ============================================
# SHEET 7: MULTI-SCALE MIXING
# ============================================

print("="*100)
print("SHEET 7: MULTI-SCALE MIXING - Verify Input from Sheet 6")
print("="*100)

# Verify decomposition components
print("\n✅ INPUT COMPONENTS FROM SHEET 6:")
print("-" * 100)
print(f"season_list[0] (Scale 0): shape {season_list[0].shape} - Seasonal at 8 timesteps")
print(f"season_list[1] (Scale 1): shape {season_list[1].shape} - Seasonal at 4 timesteps")
print(f"trend_list[0]  (Scale 0): shape {trend_list[0].shape} - Trend at 8 timesteps")
print(f"trend_list[1]  (Scale 1): shape {trend_list[1].shape} - Trend at 4 timesteps")

# Create input verification table
print("\n📊 INPUT DATA DETAILS (First 4 timesteps, Dimension 0):")
print("-" * 100)

input_table_data = {
    'Scale 0 Timestep': [f't={i}' for i in range(4)],
    'Season_S0': [f'{season_list[0][0, i, 0]:.6f}' for i in range(4)],
    'Trend_S0': [f'{trend_list[0][0, i, 0]:.6f}' for i in range(4)],
    'Scale 1 Timestep': [f't={i}' for i in range(4)],
    'Season_S1': [f'{season_list[1][0, i, 0]:.6f}' for i in range(4)],
    'Trend_S1': [f'{trend_list[1][0, i, 0]:.6f}' for i in range(4)],
}

df_input = pd.DataFrame(input_table_data)
print(df_input.to_string(index=False))

print("\n💡 Note: Season_S1 & Trend_S1 are at 4 timesteps (downsampled from 8)")

SHEET 7: MULTI-SCALE MIXING - Verify Input from Sheet 6

✅ INPUT COMPONENTS FROM SHEET 6:
----------------------------------------------------------------------------------------------------
season_list[0] (Scale 0): shape (1, 8, 4) - Seasonal at 8 timesteps
season_list[1] (Scale 1): shape (1, 4, 4) - Seasonal at 4 timesteps
trend_list[0]  (Scale 0): shape (1, 8, 4) - Trend at 8 timesteps
trend_list[1]  (Scale 1): shape (1, 4, 4) - Trend at 4 timesteps

📊 INPUT DATA DETAILS (First 4 timesteps, Dimension 0):
----------------------------------------------------------------------------------------------------
Scale 0 Timestep Season_S0  Trend_S0 Scale 1 Timestep Season_S1 Trend_S1
             t=0 -0.219701 -0.094224              t=0 -0.207044 0.065709
             t=1  0.022663  0.008591              t=1  0.098953 0.239509
             t=2  0.072384  0.236059              t=2 -0.103123 0.624525
             t=3 -0.008020  0.376501              t=3  0.502007 0.511704

💡 Note: Season_S1 & 

## 7.2 Helper Functions: Upsample & Downsample

In [42]:
# ============================================
# HELPER FUNCTIONS FOR UPSAMPLING & DOWNSAMPLING
# ============================================

def upsample_2x(data):
    """
    Upsample by factor of 2 using repeat interpolation
    
    Input:  (B, T, D) where T=4
    Output: (B, T*2, D) where T*2=8
    
    Method: Each timestep repeated twice
    Example: [a, b, c, d] → [a, a, b, b, c, c, d, d]
    """
    B, T, D = data.shape
    upsampled = np.zeros((B, T*2, D), dtype=data.dtype)
    
    for t in range(T):
        upsampled[:, 2*t, :] = data[:, t, :]      # Position 2t
        upsampled[:, 2*t+1, :] = data[:, t, :]    # Position 2t+1 (repeat)
    
    return upsampled

def downsample_2x(data):
    """
    Downsample by factor of 2 using average pooling
    
    Input:  (B, T, D) where T=8
    Output: (B, T//2, D) where T//2=4
    
    Method: Average every 2 consecutive values
    Example: [a, b, c, d, e, f, g, h] → [(a+b)/2, (c+d)/2, (e+f)/2, (g+h)/2]
    """
    B, T, D = data.shape
    downsampled = np.zeros((B, T//2, D), dtype=data.dtype)
    
    for t in range(T//2):
        downsampled[:, t, :] = (data[:, 2*t, :] + data[:, 2*t+1, :]) / 2
    
    return downsampled

print("✅ Helper functions defined successfully")
print("   - upsample_2x(): Repeat interpolation for upsampling")
print("   - downsample_2x(): Average pooling for downsampling")

✅ Helper functions defined successfully
   - upsample_2x(): Repeat interpolation for upsampling
   - downsample_2x(): Average pooling for downsampling


## 7.3 Bottom-Up Mixing: SEASON (Scale 1 → Scale 0)

**Konsep:**
- Ambil informasi SEASON dari skala kecil (Scale 1: 4 timesteps)
- Upsample ke skala besar (Scale 0: 8 timesteps)
- Alasan: Detail fluktuasi seasonal paling informatif pada fine-grain resolution

**Formula Upsampling (Repeat Interpolation):**
```
season_scale1_upsampled[t_new] = season_list[1][t_old]
where t_new = t_old × 2 (dan t_new + 1 juga = season_list[1][t_old])
```

**Mapping Contoh:**
- Scale 1 [t=0] → Scale 0 [t=0, t=1]
- Scale 1 [t=1] → Scale 0 [t=2, t=3]
- Scale 1 [t=2] → Scale 0 [t=4, t=5]
- Scale 1 [t=3] → Scale 0 [t=6, t=7]

In [43]:
# ============================================
# BOTTOM-UP MIXING: UPSAMPLE SEASON
# ============================================

print("\n" + "="*100)
print("STEP 1: BOTTOM-UP MIXING - UPSAMPLE SEASON (Scale 1 → Scale 0)")
print("="*100)

print(f"\n📊 Input: season_list[1] (Scale 1)")
print(f"Shape: {season_list[1].shape} - 4 timesteps")
print("\nData Sample (Dimension 0 - All timesteps):")
for t in range(4):
    print(f"  t={t}: {season_list[1][0, t, 0]:.6f}")

# Upsample by 2x using repeat
season_scale1_upsampled = upsample_2x(season_list[1])

print(f"\n📊 Output: season_scale1_upsampled (after upsampling)")
print(f"Shape: {season_scale1_upsampled.shape} - 8 timesteps")
print("\nData Sample (Dimension 0 - All timesteps):")
for t in range(8):
    print(f"  t={t}: {season_scale1_upsampled[0, t, 0]:.6f}")

# Show upsampling mapping in table format
print("\n📊 UPSAMPLING MAPPING TABLE (Dimension 0):")
print("-" * 100)
print(f"{'Original_t':<12} {'Original_Value':<16} {'Upsampled_t':<14} {'Upsampled_Value':<16}")
print("-" * 100)

for t in range(4):
    orig_val = season_list[1][0, t, 0]
    up_val = season_scale1_upsampled[0, 2*t, 0]
    print(f"t={t:<10} {orig_val:<16.6f} t={2*t}, t={2*t+1:<8} {up_val:<16.6f} (repeated 2x)")

print("\n✅ Verification: Each value repeated exactly 2x")
for t in range(4):
    val = season_list[1][0, t, 0]
    up1 = season_scale1_upsampled[0, 2*t, 0]
    up2 = season_scale1_upsampled[0, 2*t+1, 0]
    match1 = np.isclose(val, up1)
    match2 = np.isclose(val, up2)
    print(f"  t={t}: up[{2*t}]={match1} ✅, up[{2*t+1}]={match2} ✅" if (match1 and match2) else f"  t={t}: ERROR ❌")


STEP 1: BOTTOM-UP MIXING - UPSAMPLE SEASON (Scale 1 → Scale 0)

📊 Input: season_list[1] (Scale 1)
Shape: (1, 4, 4) - 4 timesteps

Data Sample (Dimension 0 - All timesteps):
  t=0: -0.207044
  t=1: 0.098953
  t=2: -0.103123
  t=3: 0.502007

📊 Output: season_scale1_upsampled (after upsampling)
Shape: (1, 8, 4) - 8 timesteps

Data Sample (Dimension 0 - All timesteps):
  t=0: -0.207044
  t=1: -0.207044
  t=2: 0.098953
  t=3: 0.098953
  t=4: -0.103123
  t=5: -0.103123
  t=6: 0.502007
  t=7: 0.502007

📊 UPSAMPLING MAPPING TABLE (Dimension 0):
----------------------------------------------------------------------------------------------------
Original_t   Original_Value   Upsampled_t    Upsampled_Value 
----------------------------------------------------------------------------------------------------
t=0          -0.207044        t=0, t=1        -0.207044        (repeated 2x)
t=1          0.098953         t=2, t=3        0.098953         (repeated 2x)
t=2          -0.103123        t=4, t=5

## 7.4 Top-Down Mixing: TREND (Scale 0 → Scale 1)

**Konsep:**
- Ambil informasi TREND dari skala besar (Scale 0: 8 timesteps)
- Downsample ke skala kecil (Scale 1: 4 timesteps)
- Alasan: Trend global lebih stable dan informatif pada coarse-grain resolution

**Formula Downsampling (Average Pooling):**
```
trend_scale0_downsampled[t_new] = (trend_list[0][2*t_new] + trend_list[0][2*t_new+1]) / 2
```

**Mapping Contoh:**
- Scale 0 [t=0, t=1] → Average → Scale 1 [t=0]
- Scale 0 [t=2, t=3] → Average → Scale 1 [t=1]
- Scale 0 [t=4, t=5] → Average → Scale 1 [t=2]
- Scale 0 [t=6, t=7] → Average → Scale 1 [t=3]

In [44]:
# ============================================
# TOP-DOWN MIXING: DOWNSAMPLE TREND
# ============================================

print("\n" + "="*100)
print("STEP 2: TOP-DOWN MIXING - DOWNSAMPLE TREND (Scale 0 → Scale 1)")
print("="*100)

print(f"\n📊 Input: trend_list[0] (Scale 0)")
print(f"Shape: {trend_list[0].shape} - 8 timesteps")
print("\nData Sample (Dimension 0 - All timesteps):")
for t in range(8):
    print(f"  t={t}: {trend_list[0][0, t, 0]:.6f}")

# Downsample by 2x using average pooling
trend_scale0_downsampled = downsample_2x(trend_list[0])

print(f"\n📊 Output: trend_scale0_downsampled (after downsampling)")
print(f"Shape: {trend_scale0_downsampled.shape} - 4 timesteps")
print("\nData Sample (Dimension 0 - All timesteps):")
for t in range(4):
    print(f"  t={t}: {trend_scale0_downsampled[0, t, 0]:.6f}")

# Show downsampling calculation in detail
print("\n📊 DOWNSAMPLING CALCULATION TABLE (Dimension 0 - With Formulas):")
print("-" * 120)
print(f"{'Output_t':<10} {'Input_t1':<10} {'Input_t2':<10} {'Value_t1':<14} {'Value_t2':<14} {'Sum':<14} {'Average':<14} {'Formula':<30}")
print("-" * 120)

for t in range(4):
    t1 = 2*t
    t2 = 2*t+1
    val1 = trend_list[0][0, t1, 0]
    val2 = trend_list[0][0, t2, 0]
    sum_val = val1 + val2
    avg_val = trend_scale0_downsampled[0, t, 0]
    formula = f'({val1:.4f}+{val2:.4f})/2'
    
    print(f"t={t:<8} t={t1:<8} t={t2:<8} {val1:<14.6f} {val2:<14.6f} {sum_val:<14.6f} {avg_val:<14.6f} {formula:<30}")

print("\n✅ Verification: Average pooling correctly computed")
for t in range(4):
    manual_avg = (trend_list[0][0, 2*t, 0] + trend_list[0][0, 2*t+1, 0]) / 2
    computed_avg = trend_scale0_downsampled[0, t, 0]
    match = np.isclose(manual_avg, computed_avg)
    print(f"  t={t}: manual={manual_avg:.6f}, computed={computed_avg:.6f}, match={match} ✅" if match else f"  t={t}: ERROR ❌")


STEP 2: TOP-DOWN MIXING - DOWNSAMPLE TREND (Scale 0 → Scale 1)

📊 Input: trend_list[0] (Scale 0)
Shape: (1, 8, 4) - 8 timesteps

Data Sample (Dimension 0 - All timesteps):
  t=0: -0.094224
  t=1: 0.008591
  t=2: 0.236059
  t=3: 0.376501
  t=4: 0.470428
  t=5: 0.642405
  t=6: 0.872549
  t=7: 0.675807

📊 Output: trend_scale0_downsampled (after downsampling)
Shape: (1, 4, 4) - 4 timesteps

Data Sample (Dimension 0 - All timesteps):
  t=0: -0.042817
  t=1: 0.306280
  t=2: 0.556416
  t=3: 0.774178

📊 DOWNSAMPLING CALCULATION TABLE (Dimension 0 - With Formulas):
------------------------------------------------------------------------------------------------------------------------
Output_t   Input_t1   Input_t2   Value_t1       Value_t2       Sum            Average        Formula                       
------------------------------------------------------------------------------------------------------------------------
t=0        t=0        t=1        -0.094224      0.008591       -0.0856

## 7.5 Combine Multi-Scale Mixing Results

In [45]:
# ============================================
# COMBINE MULTI-SCALE MIXING RESULTS
# ============================================

print("\n" + "="*100)
print("STEP 3: COMBINE MULTI-SCALE MIXING RESULTS FOR SHEET 8")
print("="*100)

# Create mixed lists
# Scale 0: Use upsampled SEASON (from Scale 1) + original TREND (from Scale 0)
# Scale 1: Use original SEASON (from Scale 1) + downsampled TREND (from Scale 0)
season_list_mixed = [season_scale1_upsampled, season_list[1]]
trend_list_mixed = [trend_list[0], trend_scale0_downsampled]

print("\n✅ MIXED COMPONENTS SUMMARY:")
print("-" * 100)
print("\nSEASON Components:")
print(f"  season_list_mixed[0]: shape {season_list_mixed[0].shape} (Scale 0 - UPSAMPLED from Scale 1, 8 timesteps)")
print(f"  season_list_mixed[1]: shape {season_list_mixed[1].shape} (Scale 1 - ORIGINAL, 4 timesteps)")

print("\nTREND Components:")
print(f"  trend_list_mixed[0]: shape {trend_list_mixed[0].shape} (Scale 0 - ORIGINAL, 8 timesteps)")
print(f"  trend_list_mixed[1]: shape {trend_list_mixed[1].shape} (Scale 1 - DOWNSAMPLED from Scale 0, 4 timesteps)")

# Summary table
print("\n📊 MIXING SUMMARY TABLE:")
print("-" * 100)
print(f"{'Component':<20} {'Source':<30} {'Operation':<25} {'Shape':<15}")
print("-" * 100)
print(f"{'season_list_mixed[0]':<20} {'season_list[1]':<30} {'Upsample 2x':<25} {str(season_list_mixed[0].shape):<15}")
print(f"{'season_list_mixed[1]':<20} {'season_list[1]':<30} {'None (original)':<25} {str(season_list_mixed[1].shape):<15}")
print(f"{'trend_list_mixed[0]':<20} {'trend_list[0]':<30} {'None (original)':<25} {str(trend_list_mixed[0].shape):<15}")
print(f"{'trend_list_mixed[1]':<20} {'trend_list[0]':<30} {'Downsample 2x':<25} {str(trend_list_mixed[1].shape):<15}")

print("\n💾 DATA FLOW (Dimension 0 - First 4 timesteps):")
print("-" * 100)
print("Scale 0 (8 timesteps):")
print(f"  SEASON mixed[0] (upsampled): {season_list_mixed[0][0, :4, 0]}")
print(f"  TREND mixed[0] (original):   {trend_list_mixed[0][0, :4, 0]}")
print("\nScale 1 (4 timesteps):")
print(f"  SEASON mixed[1] (original):      {season_list_mixed[1][0, :, 0]}")
print(f"  TREND mixed[1] (downsampled):    {trend_list_mixed[1][0, :, 0]}")

print("\n" + "="*100)
print("✅ SHEET 7 COMPLETE - Multi-Scale Mixing Successful!")
print("="*100)
print("\n💾 VARIABLES SAVED FOR SHEET 8:")
print("   - season_list_mixed: [Scale0_upsampled(1,8,4), Scale1_original(1,4,4)]")
print("   - trend_list_mixed: [Scale0_original(1,8,4), Scale1_downsampled(1,4,4)]")
print("\n🔜 NEXT: Sheet 8 - Reconstruction, Aggregation & Projection")


STEP 3: COMBINE MULTI-SCALE MIXING RESULTS FOR SHEET 8

✅ MIXED COMPONENTS SUMMARY:
----------------------------------------------------------------------------------------------------

SEASON Components:
  season_list_mixed[0]: shape (1, 8, 4) (Scale 0 - UPSAMPLED from Scale 1, 8 timesteps)
  season_list_mixed[1]: shape (1, 4, 4) (Scale 1 - ORIGINAL, 4 timesteps)

TREND Components:
  trend_list_mixed[0]: shape (1, 8, 4) (Scale 0 - ORIGINAL, 8 timesteps)
  trend_list_mixed[1]: shape (1, 4, 4) (Scale 1 - DOWNSAMPLED from Scale 0, 4 timesteps)

📊 MIXING SUMMARY TABLE:
----------------------------------------------------------------------------------------------------
Component            Source                         Operation                 Shape          
----------------------------------------------------------------------------------------------------
season_list_mixed[0] season_list[1]                 Upsample 2x               (1, 8, 4)      
season_list_mixed[1] season_list[1] 

---

# 📊 SHEET 8: RECONSTRUCTION, AGGREGATION & TEMPORAL PROJECTION

**Tahapan Final sebelum Loss Calculation:**

1. **Reconstruction**: Gabungkan SEASON + TREND untuk setiap scale
2. **Upsample**: Bawa semua scales ke resolusi tertinggi (8 timesteps)
3. **Aggregation**: Rata-rata multi-scale representations
4. **Temporal Projection**: Downsample dari 8 → 4 timesteps (prediksi length)
5. **Denormalization**: Kembali ke skala data original

**Input dari Sheet 7:**
- `season_list_mixed`: [Scale0(1,8,4), Scale0_upsampled(1,8,4)]
- `trend_list_mixed`: [Scale0(1,8,4), Scale1_downsampled(1,4,4)]

**Output untuk Sheet 9:**
- `prediction`: Shape (1, 4, 4) - prediksi normalized
- `prediction_denorm`: Shape (1, 4, 4) - prediksi denormalized

## 8.1 Step 1: Reconstruction (SEASON + TREND)

**Konsep Rekonstruksi:**
Gabungkan komponen SEASON (fluktuasi) dan TREND (smooth) untuk setiap scale

**Formula:**
```
recon_scale0 = season_list_mixed[0] + trend_list_mixed[0]  → (1, 8, 4)
recon_scale1 = season_list_mixed[1] + trend_list_mixed[1]  → (1, 8, 4)
```

**Interpretasi:**
- Setiap temporal representation = low-frequency smooth (TREND) + high-frequency fluctuation (SEASON)
- Reconstruction memastikan informasi complete sebelum agregasi multi-scale
- Verifikasi: Pengurangan original → seasonal, sehingga seasonal + trend ≈ original

In [46]:
# ============================================
# SHEET 8: STEP 1 - RECONSTRUCTION
# ============================================

print("="*100)
print("SHEET 8: RECONSTRUCTION, AGGREGATION & TEMPORAL PROJECTION")
print("="*100)

print("\nSTEP 1: RECONSTRUCTION (SEASON + TREND)")
print("="*100)

# Reconstruct for Scale 0
recon_scale0 = season_list_mixed[0] + trend_list_mixed[0]

print(f"\nScale 0 Reconstruction:")
print(f"  Formula: recon_scale0 = season_list_mixed[0] + trend_list_mixed[0]")
print(f"  Shape: {recon_scale0.shape}")

# Reconstruct for Scale 1
recon_scale1 = season_list_mixed[1] + trend_list_mixed[1]

print(f"\nScale 1 Reconstruction:")
print(f"  Formula: recon_scale1 = season_list_mixed[1] + trend_list_mixed[1]")
print(f"  Shape: {recon_scale1.shape}")

# Detailed reconstruction table for Scale 0 - ALL DIMENSIONS
print("\n📊 DETAILED RECONSTRUCTION TABLE - Scale 0 (ALL 4 DIMENSIONS):")
print("-" * 140)
print(f"{'t':<4} {'Dim':<4} {'SEASON':<14} {'TREND':<14} {'SUM':<14} {'RECONSTRUCTION':<16} {'Formula':<50}")
print("-" * 140)

for t in range(8):
    for dim in range(4):
        season_val = season_list_mixed[0][0, t, dim]
        trend_val = trend_list_mixed[0][0, t, dim]
        recon_val = recon_scale0[0, t, dim]
        sum_val = season_val + trend_val
        formula = f'{season_val:.6f} + {trend_val:.6f} = {recon_val:.6f}'
        
        print(f"{t:<4} {dim:<4} {season_val:<14.6f} {trend_val:<14.6f} {sum_val:<14.6f} {recon_val:<16.6f} {formula:<50}")

# Detailed reconstruction table for Scale 1 - ALL DIMENSIONS
print("\n📊 DETAILED RECONSTRUCTION TABLE - Scale 1 (ALL 4 DIMENSIONS):")
print("-" * 140)
print(f"{'t':<4} {'Dim':<4} {'SEASON':<14} {'TREND':<14} {'SUM':<14} {'RECONSTRUCTION':<16} {'Formula':<50}")
print("-" * 140)

for t in range(4):
    for dim in range(4):
        season_val = season_list_mixed[1][0, t, dim]
        trend_val = trend_list_mixed[1][0, t, dim]
        recon_val = recon_scale1[0, t, dim]
        sum_val = season_val + trend_val
        formula = f'{season_val:.6f} + {trend_val:.6f} = {recon_val:.6f}'
        
        print(f"{t:<4} {dim:<4} {season_val:<14.6f} {trend_val:<14.6f} {sum_val:<14.6f} {recon_val:<16.6f} {formula:<50}")

# Verification - ALL DIMENSIONS
print("\n✅ VERIFICATION - Reconstruction computed correctly (ALL DIMENSIONS):")
print("-" * 100)

all_match = True
for t in range(8):
    for dim in range(4):
        manual = season_list_mixed[0][0, t, dim] + trend_list_mixed[0][0, t, dim]
        computed = recon_scale0[0, t, dim]
        match = np.isclose(manual, computed)
        all_match = all_match and match
        if not match:
            print(f"❌ Scale 0, t={t}, dim={dim}: manual={manual:.6f}, computed={computed:.6f}")

for t in range(4):
    for dim in range(4):
        manual = season_list_mixed[1][0, t, dim] + trend_list_mixed[1][0, t, dim]
        computed = recon_scale1[0, t, dim]
        match = np.isclose(manual, computed)
        all_match = all_match and match
        if not match:
            print(f"❌ Scale 1, t={t}, dim={dim}: manual={manual:.6f}, computed={computed:.6f}")

if all_match:
    print("✅ All reconstructions verified correctly for ALL 4 DIMENSIONS!")
    print(f"   Total values checked: {8*4 + 4*4} = {8*4 + 4*4} values (8×4 + 4×4)")
else:
    print("❌ ERROR in reconstruction!")

SHEET 8: RECONSTRUCTION, AGGREGATION & TEMPORAL PROJECTION

STEP 1: RECONSTRUCTION (SEASON + TREND)

Scale 0 Reconstruction:
  Formula: recon_scale0 = season_list_mixed[0] + trend_list_mixed[0]
  Shape: (1, 8, 4)

Scale 1 Reconstruction:
  Formula: recon_scale1 = season_list_mixed[1] + trend_list_mixed[1]
  Shape: (1, 4, 4)

📊 DETAILED RECONSTRUCTION TABLE - Scale 0 (ALL 4 DIMENSIONS):
--------------------------------------------------------------------------------------------------------------------------------------------
t    Dim  SEASON         TREND          SUM            RECONSTRUCTION   Formula                                           
--------------------------------------------------------------------------------------------------------------------------------------------
0    0    -0.207044      -0.094224      -0.301268      -0.301268        -0.207044 + -0.094224 = -0.301268                 
0    1    -0.211988      -0.133075      -0.345063      -0.345063        -0.211988 +

## 8.2 Step 2: Upsample Scale 1 → Scale 0

**Tujuan:** Bawa semua komponen ke resolusi tertinggi (8 timesteps)

**Formula:**
```
recon_scale1_upsampled = upsample_2x(recon_scale1)
```

**Operasi:** Repeat interpolation - setiap nilai dari Scale 1 diulang 2x untuk mencapai 8 timesteps

In [47]:
# ============================================
# STEP 2: UPSAMPLE SCALE 1 → SCALE 0
# ============================================

print("\nSTEP 2: UPSAMPLE SCALE 1 → SCALE 0 (8 timesteps)")
print("="*100)

# Upsample recon_scale1
recon_scale1_upsampled = upsample_2x(recon_scale1)

print(f"\nOriginal recon_scale1: shape {recon_scale1.shape} (4 timesteps)")
print(f"Upsampled result:      shape {recon_scale1_upsampled.shape} (8 timesteps)")

# Show upsampling mapping - ALL DIMENSIONS
print("\n📊 UPSAMPLING MAPPING TABLE (ALL 4 DIMENSIONS):")
print("-" * 120)
print(f"{'Scale1_t':<12} {'Dim':<4} {'Value':<14} {'Scale0_t1':<12} {'Scale0_t2':<12} {'Upsampled_Val':<14} {'Method':<20}")
print("-" * 120)

for t in range(4):
    for dim in range(4):
        orig_val = recon_scale1[0, t, dim]
        up_val = recon_scale1_upsampled[0, 2*t, dim]
        method = 'Repeat 2x'
        
        print(f"t={t:<10} {dim:<4} {orig_val:<14.6f} t={2*t:<10} t={2*t+1:<10} {up_val:<14.6f} {method:<20}")

print("\n✅ Upsampling complete - All scales now at 8 timesteps (ALL 4 DIMENSIONS)")


STEP 2: UPSAMPLE SCALE 1 → SCALE 0 (8 timesteps)

Original recon_scale1: shape (1, 4, 4) (4 timesteps)
Upsampled result:      shape (1, 8, 4) (8 timesteps)

📊 UPSAMPLING MAPPING TABLE (ALL 4 DIMENSIONS):
------------------------------------------------------------------------------------------------------------------------
Scale1_t     Dim  Value          Scale0_t1    Scale0_t2    Upsampled_Val  Method              
------------------------------------------------------------------------------------------------------------------------
t=0          0    -0.249861      t=0          t=1          -0.249861      Repeat 2x           
t=0          1    -0.299995      t=0          t=1          -0.299995      Repeat 2x           
t=0          2    0.338510       t=0          t=1          0.338510       Repeat 2x           
t=0          3    0.474666       t=0          t=1          0.474666       Repeat 2x           
t=1          0    0.405233       t=2          t=3          0.405233       Repe

## 8.3 Step 3: Aggregation (Average Multi-Scale)

**Konsep:** Rata-rata representasi dari kedua scales untuk mendapatkan multi-scale yang informatif

**Formula:**
```
aggregated[t] = (recon_scale0[t] + recon_scale1_upsampled[t]) / 2
```

**Tujuan:** Menggabungkan informasi dari berbagai resolusi temporal

In [48]:
# ============================================
# STEP 3: AGGREGATION (AVERAGE MULTI-SCALE)
# ============================================

print("\nSTEP 3: AGGREGATION (AVERAGE Multi-Scale)")
print("="*100)

# Average both scales
aggregated = (recon_scale0 + recon_scale1_upsampled) / 2

print(f"\nAggregation formula:")
print(f"  aggregated = (recon_scale0 + recon_scale1_upsampled) / 2")
print(f"  Shape: {aggregated.shape}")

# Show aggregation details - ALL DIMENSIONS
print("\n📊 DETAILED AGGREGATION TABLE (ALL 4 DIMENSIONS):")
print("-" * 140)
print(f"{'t':<4} {'Dim':<4} {'Scale0':<14} {'Scale1_Up':<14} {'Sum':<14} {'Average':<14} {'Formula':<50}")
print("-" * 140)

for t in range(8):
    for dim in range(4):
        s0 = recon_scale0[0, t, dim]
        s1_up = recon_scale1_upsampled[0, t, dim]
        sum_val = s0 + s1_up
        avg_val = aggregated[0, t, dim]
        formula = f'({s0:.6f}+{s1_up:.6f})/2 = {avg_val:.6f}'
        
        print(f"{t:<4} {dim:<4} {s0:<14.6f} {s1_up:<14.6f} {sum_val:<14.6f} {avg_val:<14.6f} {formula:<50}")

print("\n✅ Aggregation complete - ALL 4 DIMENSIONS processed")


STEP 3: AGGREGATION (AVERAGE Multi-Scale)

Aggregation formula:
  aggregated = (recon_scale0 + recon_scale1_upsampled) / 2
  Shape: (1, 8, 4)

📊 DETAILED AGGREGATION TABLE (ALL 4 DIMENSIONS):
--------------------------------------------------------------------------------------------------------------------------------------------
t    Dim  Scale0         Scale1_Up      Sum            Average        Formula                                           
--------------------------------------------------------------------------------------------------------------------------------------------
0    0    -0.301268      -0.249861      -0.551129      -0.275565      (-0.301268+-0.249861)/2 = -0.275565               
0    1    -0.345063      -0.299995      -0.645058      -0.322529      (-0.345063+-0.299995)/2 = -0.322529               
0    2    0.238006       0.338510       0.576515       0.288258       (0.238006+0.338510)/2 = 0.288258                  
0    3    0.330820       0.474666       0

## 8.4 Step 4: Temporal Projection (8 → 4 timesteps)

**Konsep:** Downsample aggregated dari 8 timesteps → 4 timesteps untuk menghasilkan prediksi

**Formula:**
```
prediction[t] = (aggregated[2*t] + aggregated[2*t+1]) / 2
```

**Metode:** Average pooling - sama seperti downsampling di Sheet 7

In [49]:
# ============================================
# STEP 4: TEMPORAL PROJECTION (8 → 4 timesteps)
# ============================================

print("\nSTEP 4: TEMPORAL PROJECTION (8 → 4 timesteps)")
print("="*100)

# Downsample aggregated from 8 to 4 timesteps
prediction = downsample_2x(aggregated)

print(f"\nTemporal projection:")
print(f"  Formula: prediction = downsample_2x(aggregated)")
print(f"  Input shape: {aggregated.shape} (8 timesteps)")
print(f"  Output shape: {prediction.shape} (4 timesteps - final prediction)")

# Show projection calculation - ALL DIMENSIONS
print("\n📊 DETAILED TEMPORAL PROJECTION TABLE (ALL 4 DIMENSIONS):")
print("-" * 140)
print(f"{'Pred_t':<10} {'Dim':<4} {'Agg_2t':<14} {'Agg_2t+1':<14} {'Sum':<14} {'Average(Pred)':<14} {'Formula':<50}")
print("-" * 140)

for t in range(4):
    for dim in range(4):
        agg_2t = aggregated[0, 2*t, dim]
        agg_2t1 = aggregated[0, 2*t+1, dim]
        sum_val = agg_2t + agg_2t1
        pred_val = prediction[0, t, dim]
        formula = f'({agg_2t:.6f}+{agg_2t1:.6f})/2 = {pred_val:.6f}'
        
        print(f"t={t:<8} {dim:<4} {agg_2t:<14.6f} {agg_2t1:<14.6f} {sum_val:<14.6f} {pred_val:<14.6f} {formula:<50}")

print("\n✅ Temporal projection complete - ALL 4 DIMENSIONS ready for denormalization")


STEP 4: TEMPORAL PROJECTION (8 → 4 timesteps)

Temporal projection:
  Formula: prediction = downsample_2x(aggregated)
  Input shape: (1, 8, 4) (8 timesteps)
  Output shape: (1, 4, 4) (4 timesteps - final prediction)

📊 DETAILED TEMPORAL PROJECTION TABLE (ALL 4 DIMENSIONS):
--------------------------------------------------------------------------------------------------------------------------------------------
Pred_t     Dim  Agg_2t         Agg_2t+1       Sum            Average(Pred)  Formula                                           
--------------------------------------------------------------------------------------------------------------------------------------------
t=0        0    -0.275565      -0.224157      -0.499722      -0.249861      (-0.275565+-0.224157)/2 = -0.249861               
t=0        1    -0.322529      -0.277461      -0.599990      -0.299995      (-0.322529+-0.277461)/2 = -0.299995               
t=0        2    0.288258       0.388762       0.677020       0

## 8.5 Step 5: Denormalization (Back to Original Scale)

**Konsep:** Konversi dari normalized space (yang digunakan model) kembali ke skala data original

**Formula:**
```
denormalized = (normalized × std) + mean
```

**Catatan:** Hanya features 0-1 di-denormalize (features yang memiliki normalisasi awal di Sheet 2)

In [50]:
# ============================================
# STEP 5: DENORMALIZATION
# ============================================

print("\nSTEP 5: DENORMALIZATION (Back to Original Scale)")
print("="*100)

# Get normalization parameters from Sheet 2
train_std_reshaped = train_std.reshape(1, 1, -1)
train_mean_reshaped = train_mean.reshape(1, 1, -1)

print(f"\nNormalization parameters (from Sheet 2):")
print(f"  Mean: {train_mean}")
print(f"  Std:  {train_std}")

# Denormalize
prediction_denorm = prediction.copy()
prediction_denorm[:, :, :2] = prediction[:, :, :2] * train_std_reshaped + train_mean_reshaped

print(f"\nDenormalization formula:")
print(f"  prediction_denorm = (prediction × std) + mean")
print(f"  Applied to features 0-1 only (features 2-3 remain as normalized embeddings)")
print(f"  Shape: {prediction_denorm.shape}")

print(f"\n⚠️ CATATAN PENTING - Semua 4 Dimensi:")
print(f"  - Dimensi 0-1: Features asli (OT, HUFL) - DI-DENORMALIZE ke skala original")
print(f"  - Dimensi 2-3: Time embeddings (sin, cos) - TIDAK di-denormalize (tetap normalized)")
print(f"  - Semua 4 dimensi TETAP ADA dan digunakan dalam model!")

# Show denormalization details
print("\n📊 DETAILED DENORMALIZATION TABLE (Features 0 & 1):")
print("-" * 150)
print(f"{'t':<4} {'Norm_F0':<14} {'Mean_F0':<14} {'Std_F0':<14} {'Denorm_F0':<14} {'Norm_F1':<14} {'Denorm_F1':<14}")
print("-" * 150)

for t in range(4):
    norm_f0 = prediction[0, t, 0]
    mean_f0 = train_mean[0]
    std_f0 = train_std[0]
    denorm_f0 = prediction_denorm[0, t, 0]
    
    norm_f1 = prediction[0, t, 1]
    denorm_f1 = prediction_denorm[0, t, 1]
    
    print(f"{t:<4} {norm_f0:<14.6f} {mean_f0:<14.6f} {std_f0:<14.6f} {denorm_f0:<14.6f} {norm_f1:<14.6f} {denorm_f1:<14.6f}")

print("\n📊 DENORMALIZATION FORMULAS (Feature 0):")
print("-" * 100)
for t in range(4):
    norm = prediction[0, t, 0]
    mean = train_mean[0]
    std = train_std[0]
    denorm = prediction_denorm[0, t, 0]
    print(f"  t={t}: ({norm:.6f} × {std:.6f}) + {mean:.6f} = {denorm:.6f}")

print("\n✅ Denormalization complete - Prediction back in original scale")


STEP 5: DENORMALIZATION (Back to Original Scale)

Normalization parameters (from Sheet 2):
  Mean: [17.0125     20.57251588]
  Std:  [4.59012187 3.40219309]

Denormalization formula:
  prediction_denorm = (prediction × std) + mean
  Applied to features 0-1 only (features 2-3 remain as normalized embeddings)
  Shape: (1, 4, 4)

⚠️ CATATAN PENTING - Semua 4 Dimensi:
  - Dimensi 0-1: Features asli (OT, HUFL) - DI-DENORMALIZE ke skala original
  - Dimensi 2-3: Time embeddings (sin, cos) - TIDAK di-denormalize (tetap normalized)
  - Semua 4 dimensi TETAP ADA dan digunakan dalam model!

📊 DETAILED DENORMALIZATION TABLE (Features 0 & 1):
------------------------------------------------------------------------------------------------------------------------------------------------------
t    Norm_F0        Mean_F0        Std_F0         Denorm_F0      Norm_F1        Denorm_F1     
------------------------------------------------------------------------------------------------------------------

## 8.6 Final Step: Comparison with Ground Truth

In [51]:
# ============================================
# FINAL COMPARISON: PREDICTION vs GROUND TRUTH
# ============================================

print("\nSTEP 6: FINAL COMPARISON - PREDICTION vs GROUND TRUTH")
print("="*100)

# Ground truth (actual next 4 timesteps after label context)
# batch_y shape: (1, 6, 2) contains [label_len + pred_len] = [2 + 4] = 6 timesteps
# - First label_len=2 timesteps: t_6, t_7 (decoder context, NOT for prediction comparison)
# - Next pred_len=4 timesteps: t_8, t_9, t_10, t_11 (actual ground truth)
# Therefore, skip the first label_len timesteps
ground_truth = batch_y[0, label_len:, :]  # Shape: (4, 2) - timesteps t_8 to t_11 (NORMALIZED)

# Denormalize ground truth to match prediction_denorm scale
ground_truth_denorm = ground_truth * train_std + train_mean  # Shape: (4, 2)

print(f"\n🎯 Ground Truth:")
print(f"   Shape: {ground_truth.shape}")
print(f"   Contains timesteps: t_8, t_9, t_10, t_11")
print(f"   NORMALIZED values (z-score): {ground_truth.shape}")
print(f"   DENORMALIZED values (original scale): {ground_truth_denorm.shape}")
print()

print("📋 Ground Truth Values (DENORMALIZED - Original Scale):")
print("-" * 100)
print(f"{'Timestep':<10} {'Feature_0 (OT)':<20} {'Feature_1 (HUFL)':<20}")
print("-" * 100)
for t in range(pred_len):
    timestep_idx = 8 + t  # t_8, t_9, t_10, t_11
    print(f"t_{timestep_idx:<8} {ground_truth_denorm[t, 0]:<20.6f} {ground_truth_denorm[t, 1]:<20.6f}")
print("-" * 100)
print()

# Comparison table (USING DENORMALIZED VALUES FOR BOTH)
print("\n📊 PREDICTION vs GROUND TRUTH COMPARISON (DENORMALIZED - Original Scale):")
print("-" * 160)
print(f"{'t':<4} {'Timestep':<10} {'Pred_F0':<14} {'Actual_F0':<14} {'Error_F0':<14} {'Pred_F1':<14} {'Actual_F1':<14} {'Error_F1':<14}")
print("-" * 160)

for t in range(pred_len):
    timestep_idx = 8 + t  # t_8, t_9, t_10, t_11
    pred_f0 = prediction_denorm[0, t, 0]
    actual_f0 = ground_truth_denorm[t, 0]  # ✅ Using denormalized ground truth
    error_f0 = pred_f0 - actual_f0
    
    pred_f1 = prediction_denorm[0, t, 1]
    actual_f1 = ground_truth_denorm[t, 1]  # ✅ Using denormalized ground truth
    error_f1 = pred_f1 - actual_f1
    
    print(f"{t:<4} t_{timestep_idx:<8} {pred_f0:<14.6f} {actual_f0:<14.6f} {error_f0:<14.6f} {pred_f1:<14.6f} {actual_f1:<14.6f} {error_f1:<14.6f}")

# Performance metrics (USING DENORMALIZED VALUES)
print("\n📊 PERFORMANCE METRICS (Original Scale - DENORMALIZED):")
print("-" * 100)

metrics_data = []
for dim in range(2):
    mse = np.mean((prediction_denorm[0, :, dim] - ground_truth_denorm[:, dim])**2)  # ✅ Using denormalized
    mae = np.mean(np.abs(prediction_denorm[0, :, dim] - ground_truth_denorm[:, dim]))  # ✅ Using denormalized
    rmse = np.sqrt(mse)
    
    feature_name = 'OT' if dim == 0 else 'HUFL'
    print(f"\nFeature {dim} ({feature_name}):")
    print(f"  MSE (Mean Squared Error):  {mse:.6f}")
    print(f"  RMSE (Root Mean Squared):  {rmse:.6f}")
    print(f"  MAE (Mean Absolute Error): {mae:.6f}")
    
    metrics_data.append({
        'Feature': dim,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae
    })

df_metrics = pd.DataFrame(metrics_data)

print("\n" + "="*100)
print("✅ SHEET 8 COMPLETE - ALL 4 DIMENSIONS PROCESSED")
print("="*100)

print("\n📊 SUMMARY: Apa yang Terjadi dengan Semua 4 Dimensi?")
print("-" * 100)
print("\n1. RECONSTRUCTION (Step 1):")
print("   ✅ Dimensi 0-3: Semua 4 dimensi di-reconstruct (SEASON + TREND)")
print("   ✅ Output: recon_scale0 (1,8,4), recon_scale1 (1,4,4)")
print("\n2. UPSAMPLING (Step 2):")
print("   ✅ Dimensi 0-3: Semua 4 dimensi di-upsample dari 4→8 timesteps")
print("   ✅ Output: recon_scale1_upsampled (1,8,4)")
print("\n3. AGGREGATION (Step 3):")
print("   ✅ Dimensi 0-3: Semua 4 dimensi di-average (Scale 0 + Scale 1)")
print("   ✅ Output: aggregated (1,8,4)")
print("\n4. TEMPORAL PROJECTION (Step 4):")
print("   ✅ Dimensi 0-3: Semua 4 dimensi di-downsample dari 8→4 timesteps")
print("   ✅ Output: prediction (1,4,4) - SEMUA DIMENSI ADA!")
print("\n5. DENORMALIZATION (Step 5):")
print("   ✅ Dimensi 0-1: Features asli (OT, HUFL) di-denormalize ke skala original")
print("   ⚠️  Dimensi 2-3: Time embeddings (sin, cos) TIDAK di-denormalize")
print("   ✅ Output: prediction_denorm (1,4,4) - SEMUA DIMENSI TETAP ADA!")
print("\n6. COMPARISON (Step 6):")
print("   ✅ Dimensi 0-1: Dibandingkan dengan ground truth (karena hanya ini yang observable)")
print("   ⚠️  Dimensi 2-3: Tidak dibandingkan (karena hanya embedding internal model)")

print("\n" + "="*100)
print("🎯 KESIMPULAN:")
print("   - SEMUA 4 DIMENSI tetap diproses di semua langkah Sheet 8")
print("   - Operasi numpy (add, multiply, downsample) otomatis menghandle semua dimensi")
print("   - Hanya TAMPILAN TABEL yang menunjukkan dimensi tertentu untuk clarity")
print("   - Dimensi 2-3 tetap penting sebagai temporal information dalam model!")
print("="*100)

print("\n💾 VARIABLES SAVED FOR SHEET 9:")
print("   - prediction: Normalized prediction (1, 4, 4) - SEMUA 4 DIMENSI")
print("   - prediction_denorm: Denormalized prediction (1, 4, 4) - SEMUA 4 DIMENSI")
print("   - ground_truth: Actual ground truth NORMALIZED (4, 2) - timesteps t_8 to t_11")
print("   - ground_truth_denorm: Actual ground truth DENORMALIZED (4, 2) - original scale")
print("   - metrics: Performance metrics (MSE, RMSE, MAE) in original scale")

print("\n🔜 NEXT: Sheet 9 - Loss Calculation & Error Analysis")


STEP 6: FINAL COMPARISON - PREDICTION vs GROUND TRUTH

🎯 Ground Truth:
   Shape: (4, 2)
   Contains timesteps: t_8, t_9, t_10, t_11
   NORMALIZED values (z-score): (4, 2)
   DENORMALIZED values (original scale): (4, 2)

📋 Ground Truth Values (DENORMALIZED - Original Scale):
----------------------------------------------------------------------------------------------------
Timestep   Feature_0 (OT)       Feature_1 (HUFL)    
----------------------------------------------------------------------------------------------------
t_8        25.900000            24.330127           
t_9        28.200000            19.850000           
t_10       29.900000            15.769873           
t_11       32.150000            15.619873           
----------------------------------------------------------------------------------------------------


📊 PREDICTION vs GROUND TRUTH COMPARISON (DENORMALIZED - Original Scale):
---------------------------------------------------------------------------------

---

# Sheet 9: Loss Calculation & Error Analysis

**Tujuan:**
Menghitung loss function yang akan digunakan untuk backpropagation dan training model TimeMixer.

**Komponen Loss:**
1. **MSE Loss (Mean Squared Error)**: Loss utama untuk forecasting
2. **MAE Loss (Mean Absolute Error)**: Metric tambahan untuk robustness
3. **Gradient Calculation**: Untuk backpropagation dalam training
4. **Per-Feature & Per-Timestep Analysis**: Breakdown detail untuk interpretasi

**Input dari Sheet 8:**
- `prediction_denorm`: Prediksi model (1, 4, 4) - denormalized (original scale)
- `ground_truth_denorm`: Target sebenarnya (4, 2) - denormalized (original scale)

**⚠️ IMPORTANT:** Semua perhitungan loss menggunakan nilai **DENORMALIZED** agar dalam skala original yang interpretable!

**Output untuk backpropagation:**
- `total_mse`: Scalar MSE loss untuk gradient descent
- `total_mae`: Scalar MAE loss untuk monitoring
- `gradients`: Gradient ∂MSE/∂prediction untuk weight update

---

## 9.1 Verify Input Data from Sheet 8

In [52]:
# ============================================
# SHEET 9: LOSS CALCULATION & ERROR ANALYSIS
# ============================================

print("="*100)
print("SHEET 9: LOSS CALCULATION & ERROR ANALYSIS")
print("="*100)

print("\n📋 STEP 1: VERIFY INPUT DATA FROM SHEET 8")
print("-" * 100)

# Verify prediction and ground truth shapes
print(f"\n✅ Input Data Verification:")
print(f"  prediction (normalized):     shape {prediction.shape}")
print(f"  prediction_denorm:           shape {prediction_denorm.shape}")
print(f"  ground_truth (normalized):   shape {ground_truth.shape}")
print(f"  ground_truth_denorm:         shape {ground_truth_denorm.shape}")

print(f"\n📊 Data Summary:")
print(f"  Prediction timesteps:  4 (t=0, 1, 2, 3 relative to prediction start)")
print(f"  Actual timesteps:      t_8, t_9, t_10, t_11 (from original data)")
print(f"  Features compared:     2 (Feature_0=OT, Feature_1=HUFL)")
print(f"  Scale:                 DENORMALIZED (original scale) for interpretability")
print(f"\n📊 DETAILED INPUT DATA TABLE (DENORMALIZED - Original Scale):")
print("-" * 130)
print(f"{'t':<4} {'Timestep':<10} {'Pred_F0':<14} {'Actual_F0':<14} {'Diff_F0':<14} {'Pred_F1':<14} {'Actual_F1':<14} {'Diff_F1':<14}")
print("-" * 130)

for t in range(pred_len):
    timestep_idx = 8 + t  # t_8, t_9, t_10, t_11
    pred_f0 = prediction_denorm[0, t, 0]
    actual_f0 = ground_truth_denorm[t, 0]  # ✅ Using denormalized
    diff_f0 = pred_f0 - actual_f0
    
    pred_f1 = prediction_denorm[0, t, 1]
    actual_f1 = ground_truth_denorm[t, 1]  # ✅ Using denormalized
    diff_f1 = pred_f1 - actual_f1
    
    print(f"{t:<4} t_{timestep_idx:<8} {pred_f0:<14.6f} {actual_f0:<14.6f} {diff_f0:<14.6f} {pred_f1:<14.6f} {actual_f1:<14.6f} {diff_f1:<14.6f}")

print("\n✅ Input verification complete")

SHEET 9: LOSS CALCULATION & ERROR ANALYSIS

📋 STEP 1: VERIFY INPUT DATA FROM SHEET 8
----------------------------------------------------------------------------------------------------

✅ Input Data Verification:
  prediction (normalized):     shape (1, 4, 4)
  prediction_denorm:           shape (1, 4, 4)
  ground_truth (normalized):   shape (4, 2)
  ground_truth_denorm:         shape (4, 2)

📊 Data Summary:
  Prediction timesteps:  4 (t=0, 1, 2, 3 relative to prediction start)
  Actual timesteps:      t_8, t_9, t_10, t_11 (from original data)
  Features compared:     2 (Feature_0=OT, Feature_1=HUFL)
  Scale:                 DENORMALIZED (original scale) for interpretability

📊 DETAILED INPUT DATA TABLE (DENORMALIZED - Original Scale):
----------------------------------------------------------------------------------------------------------------------------------
t    Timestep   Pred_F0        Actual_F0      Diff_F0        Pred_F1        Actual_F1      Diff_F1       
----------------

## 9.2 Calculate Error (Residual) untuk Setiap Prediksi

**Formula:**
```
error[t, f] = prediction[t, f] - ground_truth[t, f]
```

**Konsep:**
- Error positif: Model **overpredict** (prediksi lebih tinggi dari actual)
- Error negatif: Model **underpredict** (prediksi lebih rendah dari actual)
- Error mendekati 0: Prediksi akurat

In [53]:
# ============================================
# STEP 2: CALCULATE ERRORS (RESIDUALS)
# ============================================

print("\n📋 STEP 2: CALCULATE ERRORS (RESIDUALS)")
print("="*100)

# Calculate errors for features 0-1 (USING DENORMALIZED VALUES)
errors = prediction_denorm[0, :, :2] - ground_truth_denorm[:, :]  # ✅ Using denormalized

print(f"\nError calculation (in ORIGINAL SCALE):")
print(f"  errors = prediction_denorm - ground_truth_denorm")
print(f"  Shape: {errors.shape} (4 timesteps × 2 features)")
print(f"  Scale: DENORMALIZED (original scale for interpretability)")

print(f"\n📊 DETAILED ERROR CALCULATION TABLE (DENORMALIZED):")
print("-" * 160)
print(f"{'t':<4} {'Timestep':<10} {'Feature':<8} {'Prediction':<14} {'Actual':<14} {'Error':<14} {'Error²':<14} {'|Error|':<14} {'Formula':<40}")
print("-" * 160)

for t in range(pred_len):
    timestep_idx = 8 + t  # t_8, t_9, t_10, t_11
    for f in range(2):
        pred = prediction_denorm[0, t, f]
        actual = ground_truth_denorm[t, f]  # ✅ Using denormalized
        error = errors[t, f]
        error_sq = error ** 2
        error_abs = np.abs(error)
        formula = f'{pred:.4f} - {actual:.4f}'
        feature_name = 'OT' if f == 0 else 'HUFL'
        
        print(f"{t:<4} t_{timestep_idx:<8} {feature_name:<8} {pred:<14.6f} {actual:<14.6f} {error:<14.6f} {error_sq:<14.6f} {error_abs:<14.6f} {formula:<40}")

print(f"\n📊 ERROR INTERPRETATION:")
print("-" * 100)
for t in range(pred_len):
    for f in range(2):
        error = errors[t, f]
        if error > 0:
            interpretation = "OVERPREDICT (prediksi lebih tinggi)"
        elif error < 0:
            interpretation = "UNDERPREDICT (prediksi lebih rendah)"
        else:
            interpretation = "PERFECT (prediksi tepat)"
        
        print(f"  t={t}, Feature {f}: error = {error:+.6f} → {interpretation}")

print("\n✅ Error calculation complete")


📋 STEP 2: CALCULATE ERRORS (RESIDUALS)

Error calculation (in ORIGINAL SCALE):
  errors = prediction_denorm - ground_truth_denorm
  Shape: (4, 2) (4 timesteps × 2 features)
  Scale: DENORMALIZED (original scale for interpretability)

📊 DETAILED ERROR CALCULATION TABLE (DENORMALIZED):
----------------------------------------------------------------------------------------------------------------------------------------------------------------
t    Timestep   Feature  Prediction     Actual         Error          Error²         |Error|        Formula                                 
----------------------------------------------------------------------------------------------------------------------------------------------------------------
0    t_8        OT       15.865608      25.900000      -10.034392     100.689030     10.034392      15.8656 - 25.9000                       
0    t_8        HUFL     19.551875      24.330127      -4.778252      22.831693      4.778252       19.5519 - 

## 9.3 Calculate MSE Loss (Mean Squared Error)

**Formula MSE:**
```
MSE = (1/N) × Σ(prediction - actual)²
```

Dimana:
- N = total number of predictions = timesteps × features = 4 × 2 = 8
- Σ = sum of all squared errors

**Breakdown:**
1. **Per-Timestep MSE**: MSE untuk setiap timestep (average across features)
2. **Per-Feature MSE**: MSE untuk setiap feature (average across timesteps)
3. **Total MSE**: MSE keseluruhan (average semua errors)

In [54]:
# ============================================
# STEP 3: CALCULATE MSE LOSS
# ============================================

print("\n📋 STEP 3: MSE LOSS CALCULATION")
print("="*100)

# Calculate squared errors
squared_errors = errors ** 2

print(f"\n📊 STEP 3.1: SQUARED ERRORS")
print("-" * 120)
print(f"{'t':<4} {'Feature':<8} {'Error':<14} {'Error²':<14} {'Formula':<40}")
print("-" * 120)

for t in range(pred_len):
    for f in range(2):
        error = errors[t, f]
        error_sq = squared_errors[t, f]
        formula = f'({error:.6f})² = {error_sq:.6f}'
        
        print(f"{t:<4} {f:<8} {error:<14.6f} {error_sq:<14.6f} {formula:<40}")

# Calculate per-timestep MSE
print(f"\n📊 STEP 3.2: PER-TIMESTEP MSE (Average across features)")
print("-" * 120)
print(f"{'t':<4} {'Error²_F0':<14} {'Error²_F1':<14} {'Sum':<14} {'MSE_t':<14} {'Formula':<50}")
print("-" * 120)

mse_per_timestep = []
for t in range(pred_len):
    sq_f0 = squared_errors[t, 0]
    sq_f1 = squared_errors[t, 1]
    sum_sq = sq_f0 + sq_f1
    mse_t = sum_sq / 2.0
    mse_per_timestep.append(mse_t)
    formula = f'({sq_f0:.6f} + {sq_f1:.6f}) / 2 = {mse_t:.6f}'
    
    print(f"{t:<4} {sq_f0:<14.6f} {sq_f1:<14.6f} {sum_sq:<14.6f} {mse_t:<14.6f} {formula:<50}")

mse_per_timestep = np.array(mse_per_timestep)

# Calculate per-feature MSE
print(f"\n📊 STEP 3.3: PER-FEATURE MSE (Average across timesteps)")
print("-" * 120)
print(f"{'Feature':<8} {'Sum of Error²':<18} {'N_timesteps':<12} {'MSE_feature':<14} {'Formula':<50}")
print("-" * 120)

mse_per_feature = []
for f in range(2):
    sum_sq = np.sum(squared_errors[:, f])
    mse_f = sum_sq / pred_len
    mse_per_feature.append(mse_f)
    
    squared_values = ' + '.join([f'{squared_errors[t, f]:.4f}' for t in range(pred_len)])
    formula = f'({squared_values}) / {pred_len}'
    
    print(f"{f:<8} {sum_sq:<18.6f} {pred_len:<12} {mse_f:<14.6f} {formula[:48]:<50}")

mse_per_feature = np.array(mse_per_feature)

# Calculate total MSE
print(f"\n📊 STEP 3.4: TOTAL MSE (Average across ALL predictions)")
print("-" * 120)

total_squared_errors = np.sum(squared_errors)
total_predictions = pred_len * 2  # 4 timesteps × 2 features
total_mse = total_squared_errors / total_predictions

print(f"Total sum of squared errors: {total_squared_errors:.6f}")
print(f"Total number of predictions: {total_predictions}")
print(f"Total MSE = {total_squared_errors:.6f} / {total_predictions} = {total_mse:.6f}")

print(f"\n📊 MANUAL VERIFICATION:")
print(f"  Manual calculation:")
all_sq_errors = [squared_errors[t, f] for t in range(pred_len) for f in range(2)]
print(f"    All squared errors: {[f'{e:.6f}' for e in all_sq_errors]}")
manual_sum = sum(all_sq_errors)
manual_mse = manual_sum / len(all_sq_errors)
print(f"    Sum: {manual_sum:.6f}")
print(f"    MSE: {manual_sum:.6f} / {len(all_sq_errors)} = {manual_mse:.6f}")
print(f"  NumPy calculation: {total_mse:.6f}")
print(f"  Match: {np.isclose(manual_mse, total_mse)}")

print("\n✅ MSE calculation complete")


📋 STEP 3: MSE LOSS CALCULATION

📊 STEP 3.1: SQUARED ERRORS
------------------------------------------------------------------------------------------------------------------------
t    Feature  Error          Error²         Formula                                 
------------------------------------------------------------------------------------------------------------------------
0    0        -10.034392     100.689030     (-10.034392)² = 100.689030              
0    1        -4.778252      22.831693      (-4.778252)² = 22.831693                
1    0        -9.327432      87.000994      (-9.327432)² = 87.000994                
1    1        1.884516       3.551402       (1.884516)² = 3.551402                  
2    0        -10.806830     116.787567     (-10.806830)² = 116.787567              
2    1        4.972343       24.724194      (4.972343)² = 24.724194                 
3    0        -9.279655      86.111993      (-9.279655)² = 86.111993                
3    1        8.60

## 9.4 Calculate MAE Loss (Mean Absolute Error)

**Formula MAE:**
```
MAE = (1/N) × Σ|prediction - actual|
```

**Karakteristik:**
- MAE menggunakan **absolute value** (nilai mutlak) bukan squared
- Lebih robust terhadap outliers dibanding MSE
- Interpretasi lebih mudah: rata-rata error dalam satuan asli data

In [55]:
# ============================================
# STEP 4: CALCULATE MAE LOSS
# ============================================

print("\n📋 STEP 4: MAE LOSS CALCULATION")
print("="*100)

# Calculate absolute errors
absolute_errors = np.abs(errors)

print(f"\n📊 STEP 4.1: ABSOLUTE ERRORS")
print("-" * 120)
print(f"{'t':<4} {'Feature':<8} {'Error':<14} {'|Error|':<14} {'Formula':<40}")
print("-" * 120)

for t in range(pred_len):
    for f in range(2):
        error = errors[t, f]
        abs_error = absolute_errors[t, f]
        formula = f'|{error:.6f}| = {abs_error:.6f}'
        
        print(f"{t:<4} {f:<8} {error:<14.6f} {abs_error:<14.6f} {formula:<40}")

# Calculate per-timestep MAE
print(f"\n📊 STEP 4.2: PER-TIMESTEP MAE (Average across features)")
print("-" * 120)
print(f"{'t':<4} {'|Error|_F0':<14} {'|Error|_F1':<14} {'Sum':<14} {'MAE_t':<14} {'Formula':<50}")
print("-" * 120)

mae_per_timestep = []
for t in range(pred_len):
    abs_f0 = absolute_errors[t, 0]
    abs_f1 = absolute_errors[t, 1]
    sum_abs = abs_f0 + abs_f1
    mae_t = sum_abs / 2.0
    mae_per_timestep.append(mae_t)
    formula = f'({abs_f0:.6f} + {abs_f1:.6f}) / 2 = {mae_t:.6f}'
    
    print(f"{t:<4} {abs_f0:<14.6f} {abs_f1:<14.6f} {sum_abs:<14.6f} {mae_t:<14.6f} {formula:<50}")

mae_per_timestep = np.array(mae_per_timestep)

# Calculate per-feature MAE
print(f"\n📊 STEP 4.3: PER-FEATURE MAE (Average across timesteps)")
print("-" * 120)
print(f"{'Feature':<8} {'Sum of |Error|':<18} {'N_timesteps':<12} {'MAE_feature':<14} {'Formula':<50}")
print("-" * 120)

mae_per_feature = []
for f in range(2):
    sum_abs = np.sum(absolute_errors[:, f])
    mae_f = sum_abs / pred_len
    mae_per_feature.append(mae_f)
    
    abs_values = ' + '.join([f'{absolute_errors[t, f]:.4f}' for t in range(pred_len)])
    formula = f'({abs_values}) / {pred_len}'
    
    print(f"{f:<8} {sum_abs:<18.6f} {pred_len:<12} {mae_f:<14.6f} {formula[:48]:<50}")

mae_per_feature = np.array(mae_per_feature)

# Calculate total MAE
print(f"\n📊 STEP 4.4: TOTAL MAE (Average across ALL predictions)")
print("-" * 120)

total_absolute_errors = np.sum(absolute_errors)
total_mae = total_absolute_errors / total_predictions

print(f"Total sum of absolute errors: {total_absolute_errors:.6f}")
print(f"Total number of predictions:  {total_predictions}")
print(f"Total MAE = {total_absolute_errors:.6f} / {total_predictions} = {total_mae:.6f}")

print("\n✅ MAE calculation complete")


📋 STEP 4: MAE LOSS CALCULATION

📊 STEP 4.1: ABSOLUTE ERRORS
------------------------------------------------------------------------------------------------------------------------
t    Feature  Error          |Error|        Formula                                 
------------------------------------------------------------------------------------------------------------------------
0    0        -10.034392     10.034392      |-10.034392| = 10.034392                
0    1        -4.778252      4.778252       |-4.778252| = 4.778252                  
1    0        -9.327432      9.327432       |-9.327432| = 9.327432                  
1    1        1.884516       1.884516       |1.884516| = 1.884516                   
2    0        -10.806830     10.806830      |-10.806830| = 10.806830                
2    1        4.972343       4.972343       |4.972343| = 4.972343                   
3    0        -9.279655      9.279655       |-9.279655| = 9.279655                  
3    1        8.6

## 9.5 Loss Summary & Comparison

Ringkasan semua metrics yang telah dihitung (MSE dan MAE).

In [56]:
# ============================================
# STEP 5: LOSS SUMMARY
# ============================================

print("\n📋 STEP 5: LOSS SUMMARY & COMPARISON")
print("="*100)

# Create comprehensive summary table
print(f"\n📊 OVERALL LOSS METRICS SUMMARY:")
print("-" * 90)
print(f"{'Metric':<15} {'Total':<14} {'Feature 0':<14} {'Feature 1':<14} {'Description':<30}")
print("-" * 90)

print(f"{'MSE':<15} {total_mse:<14.6f} {mse_per_feature[0]:<14.6f} {mse_per_feature[1]:<14.6f} {'Mean Squared Error':<30}")
print(f"{'MAE':<15} {total_mae:<14.6f} {mae_per_feature[0]:<14.6f} {mae_per_feature[1]:<14.6f} {'Mean Absolute Error':<30}")

print(f"\n📊 PER-TIMESTEP BREAKDOWN:")
print("-" * 100)
print(f"{'Timestep':<10} {'MSE':<14} {'MAE':<14} {'Interpretation':<50}")
print("-" * 100)

for t in range(pred_len):
    timestep_idx = 8 + t
    mse_t = mse_per_timestep[t]
    mae_t = mae_per_timestep[t]
    
    if mse_t < 0.01:
        interpretation = "Excellent prediction"
    elif mse_t < 0.1:
        interpretation = "Good prediction"
    elif mse_t < 1.0:
        interpretation = "Moderate prediction"
    else:
        interpretation = "Poor prediction - needs improvement"
    
    print(f"t_{timestep_idx:<8} {mse_t:<14.6f} {mae_t:<14.6f} {interpretation:<50}")

print(f"\n📊 COMPARISON: MSE vs MAE")
print("-" * 100)
print(f"  MSE = {total_mse:.6f}  (squared errors - penalize large errors more)")
print(f"  MAE = {total_mae:.6f}  (absolute errors - robust to outliers)")
print(f"\n  MSE is typically used as the PRIMARY LOSS for training (backpropagation)")
print(f"  MAE is used as SECONDARY METRIC for evaluation and monitoring")

print("\n✅ Loss summary complete")


📋 STEP 5: LOSS SUMMARY & COMPARISON

📊 OVERALL LOSS METRICS SUMMARY:
------------------------------------------------------------------------------------------
Metric          Total          Feature 0      Feature 1      Description                   
------------------------------------------------------------------------------------------
MSE             64.475129      97.647396      31.302862      Mean Squared Error            
MAE             7.461475       9.862077       5.060872       Mean Absolute Error           

📊 PER-TIMESTEP BREAKDOWN:
----------------------------------------------------------------------------------------------------
Timestep   MSE            MAE            Interpretation                                    
----------------------------------------------------------------------------------------------------
t_8        61.760362      7.406322       Poor prediction - needs improvement               
t_9        45.276198      5.605974       Poor prediction - 

## 9.6 Gradient Calculation untuk Backpropagation

**Konsep Backpropagation:**
Untuk melatih model, kita perlu menghitung **gradient** (turunan) dari loss terhadap prediksi, yaitu:

```
∂L/∂prediction = ∂(MSE)/∂prediction
```

**Formula Gradient MSE:**
```
∂MSE/∂prediction[i] = (2/N) × (prediction[i] - actual[i])
                     = (2/N) × error[i]
```

**Mengapa perlu gradient?**
- Gradient menunjukkan **arah** dan **besaran** perubahan yang diperlukan
- Digunakan untuk update weights dalam gradient descent
- Formula update: `weight_new = weight_old - learning_rate × gradient`

**⚠️ Ini adalah komponen BACKPROPAGATION - gradient ini akan di-propagate ke layer sebelumnya untuk update weights!**

In [57]:
# ============================================
# STEP 6: GRADIENT CALCULATION (BACKPROPAGATION)
# ============================================

print("\n📋 STEP 6: GRADIENT CALCULATION FOR BACKPROPAGATION")
print("="*100)

# Calculate gradients
# Gradient of MSE: ∂L/∂pred = (2/N) × (pred - actual)
N = total_predictions  # 8 (4 timesteps × 2 features)
gradients = (2.0 / N) * errors

print(f"\n🔄 BACKPROPAGATION: Computing gradients for weight updates")
print(f"\nGradient formula:")
print(f"  ∂MSE/∂prediction = (2/N) × (prediction - actual)")
print(f"  ∂MSE/∂prediction = (2/N) × error")
print(f"  where N = {N} (total predictions = {pred_len} timesteps × 2 features)")

print(f"\n📊 DETAILED GRADIENT CALCULATION:")
print("-" * 160)
print(f"{'t':<4} {'Timestep':<10} {'Feature':<8} {'Error':<14} {'2/N':<14} {'Gradient':<14} {'Formula':<50}")
print("-" * 160)

for t in range(pred_len):
    timestep_idx = 8 + t
    for f in range(2):
        error = errors[t, f]
        factor = 2.0 / N
        gradient = gradients[t, f]
        formula = f'(2/{N}) × {error:.6f} = {gradient:.6f}'
        feature_name = 'OT' if f == 0 else 'HUFL'
        
        print(f"{t:<4} t_{timestep_idx:<8} {feature_name:<8} {error:<14.6f} {factor:<14.6f} {gradient:<14.6f} {formula:<50}")

print(f"\n📊 GRADIENT INTERPRETATION (for Weight Update):")
print("-" * 120)
for t in range(pred_len):
    timestep_idx = 8 + t
    for f in range(2):
        gradient = gradients[t, f]
        feature_name = 'OT' if f == 0 else 'HUFL'
        
        if gradient > 0:
            direction = "DECREASE prediction (predicted too high)"
        elif gradient < 0:
            direction = "INCREASE prediction (predicted too low)"
        else:
            direction = "NO CHANGE needed (perfect prediction)"
        
        print(f"  t_{timestep_idx}, {feature_name}: gradient = {gradient:+.6f} → {direction}")

# Manual verification example
print(f"\n📋 MANUAL VERIFICATION EXAMPLE (t=0, Feature 0):")
print("-" * 100)
t_example = 0
f_example = 0
error_example = errors[t_example, f_example]
gradient_example = gradients[t_example, f_example]
manual_gradient = (2.0 / N) * error_example

print(f"  Error[0,0] = {error_example:.6f}")
print(f"  N = {N}")
print(f"  Gradient = (2/{N}) × {error_example:.6f}")
print(f"           = {manual_gradient:.6f}")
print(f"  From array: {gradient_example:.6f}")
print(f"  Match: {np.isclose(manual_gradient, gradient_example)} ✅")

# Gradient statistics
print(f"\n📊 GRADIENT STATISTICS:")
print("-" * 100)
print(f"  Mean gradient:     {gradients.mean():.6f}")
print(f"  Max gradient:      {gradients.max():.6f}")
print(f"  Min gradient:      {gradients.min():.6f}")
print(f"  Std gradient:      {gradients.std():.6f}")

print(f"\n✅ Gradient calculation complete")
print(f"💡 These gradients will be used to update model weights during training!")
print(f"   Formula: weight_new = weight_old - learning_rate × gradient")


📋 STEP 6: GRADIENT CALCULATION FOR BACKPROPAGATION

🔄 BACKPROPAGATION: Computing gradients for weight updates

Gradient formula:
  ∂MSE/∂prediction = (2/N) × (prediction - actual)
  ∂MSE/∂prediction = (2/N) × error
  where N = 8 (total predictions = 4 timesteps × 2 features)

📊 DETAILED GRADIENT CALCULATION:
----------------------------------------------------------------------------------------------------------------------------------------------------------------
t    Timestep   Feature  Error          2/N            Gradient       Formula                                           
----------------------------------------------------------------------------------------------------------------------------------------------------------------
0    t_8        OT       -10.034392     0.250000       -2.508598      (2/8) × -10.034392 = -2.508598                    
0    t_8        HUFL     -4.778252      0.250000       -1.194563      (2/8) × -4.778252 = -1.194563                     
1   

## 9.7 Final Summary: Sheet 9 Complete

**✅ Sheet 9 Completed Successfully!**

Semua perhitungan loss dan gradient telah selesai. Berikut ringkasan lengkap:

**Sections Completed:**
- ✅ 9.1: Input verification
- ✅ 9.2: Error calculation
- ✅ 9.3: MSE calculation (total, per-feature, per-timestep)
- ✅ 9.4: MAE calculation (total, per-feature, per-timestep)
- ✅ 9.5: Loss summary & comparison
- ✅ 9.6: **Gradient calculation untuk BACKPROPAGATION**
- ✅ 9.7: Final summary

**Key Variables Saved:**
All variables are ready for next sheet or training loop!

In [58]:
# ============================================
# FINAL SUMMARY - SHEET 9 COMPLETE
# ============================================

print("\n" + "="*100)
print("✅ SHEET 9: LOSS CALCULATION & BACKPROPAGATION - COMPLETE")
print("="*100)

print("\n📊 COMPREHENSIVE LOSS METRICS:")
print("-" * 100)
print(f"  MSE (Mean Squared Error):  {total_mse:.6f}  ← PRIMARY LOSS for training")
print(f"  MAE (Mean Absolute Error): {total_mae:.6f}  ← SECONDARY METRIC for monitoring")

print(f"\n📊 PER-FEATURE BREAKDOWN:")
print("-" * 100)
print(f"  Feature 0 (OT):")
print(f"    MSE: {mse_per_feature[0]:.6f}")
print(f"    MAE: {mae_per_feature[0]:.6f}")
print(f"  Feature 1 (HUFL):")
print(f"    MSE: {mse_per_feature[1]:.6f}")
print(f"    MAE: {mae_per_feature[1]:.6f}")

print(f"\n📊 ERROR ANALYSIS SUMMARY:")
print("-" * 100)
print(f"  Total predictions: {total_predictions}")
print(f"  Sum of errors²:    {total_squared_errors:.6f}")
print(f"  Sum of |errors|:   {total_absolute_errors:.6f}")
print(f"  Mean error:        {np.mean(errors):.6f}")
print(f"  Max |error|:       {np.max(np.abs(errors)):.6f}")
print(f"  Min |error|:       {np.min(np.abs(errors)):.6f}")

print(f"\n🔄 GRADIENT FLOW FOR BACKPROPAGATION:")
print("-" * 100)
print(f"  Gradient formula: ∂MSE/∂prediction = (2/{N}) × error")
print(f"  Gradient shape:   {gradients.shape}")
print(f"  Mean gradient:    {np.mean(gradients):.6f}")
print(f"  Gradient norm:    {np.linalg.norm(gradients):.6f}")
print(f"  ✅ Gradients ready for weight update!")

print(f"\n📊 VARIABLES SAVED FOR BACKPROPAGATION:")
print("-" * 100)
saved_vars = {
    'errors': errors.shape,
    'squared_errors': squared_errors.shape,
    'absolute_errors': absolute_errors.shape,
    'total_mse': 'scalar',
    'total_mae': 'scalar',
    'mse_per_feature': mse_per_feature.shape,
    'mae_per_feature': mae_per_feature.shape,
    'mse_per_timestep': mse_per_timestep.shape,
    'mae_per_timestep': mae_per_timestep.shape,
    'gradients': gradients.shape
}

for var_name, var_shape in saved_vars.items():
    print(f"  {var_name:<25} {str(var_shape):<20}")

print("\n" + "="*100)
print("🎯 KEY FINDINGS:")
print("="*100)
print(f"  1. Primary Loss (MSE):   {total_mse:.6f}")
print(f"  2. Prediction Quality: ", end="")
if total_mse < 0.01:
    print("EXCELLENT - very low error")
elif total_mse < 0.1:
    print("GOOD - reasonable prediction accuracy")
elif total_mse < 1.0:
    print("MODERATE - acceptable but room for improvement")
else:
    print("NEEDS IMPROVEMENT - consider model tuning")

print(f"  3. Feature Performance:")
print(f"     - Feature 0 (OT) MSE:   {mse_per_feature[0]:.6f}")
print(f"     - Feature 1 (HUFL) MSE: {mse_per_feature[1]:.6f}")

if mse_per_feature[0] < mse_per_feature[1]:
    print(f"     → Feature 0 predictions are MORE accurate")
else:
    print(f"     → Feature 1 predictions are MORE accurate")

print("\n" + "="*100)
print("🔄 BACKPROPAGATION READY:")
print("="*100)
print("  ✅ Loss calculated: MSE = {:.6f}".format(total_mse))
print("  ✅ Gradients computed: ∂MSE/∂prediction ready")
print("  ✅ Next step: Propagate gradients backward through network layers")
print("  ✅ Weight update formula: W_new = W_old - learning_rate × gradient")
print("\n" + "="*100)


✅ SHEET 9: LOSS CALCULATION & BACKPROPAGATION - COMPLETE

📊 COMPREHENSIVE LOSS METRICS:
----------------------------------------------------------------------------------------------------
  MSE (Mean Squared Error):  64.475129  ← PRIMARY LOSS for training
  MAE (Mean Absolute Error): 7.461475  ← SECONDARY METRIC for monitoring

📊 PER-FEATURE BREAKDOWN:
----------------------------------------------------------------------------------------------------
  Feature 0 (OT):
    MSE: 97.647396
    MAE: 9.862077
  Feature 1 (HUFL):
    MSE: 31.302862
    MAE: 5.060872

📊 ERROR ANALYSIS SUMMARY:
----------------------------------------------------------------------------------------------------
  Total predictions: 8
  Sum of errors²:    515.801032
  Sum of |errors|:   59.691798
  Mean error:        -3.595166
  Max |error|:       10.806830
  Min |error|:       1.884516

🔄 GRADIENT FLOW FOR BACKPROPAGATION:
--------------------------------------------------------------------------------------

---

# SHEET 10: Backpropagation Through TimeMixer Layers

**Tujuan:**
Menghitung gradient propagation mundur melalui semua layer TimeMixer untuk update weights.

**Alur Backpropagation (Backward Pass):**
1. **Loss → Prediction**: Gradient sudah dihitung di Sheet 9
2. **Prediction → Temporal Projection**: Gradient melalui downsampling
3. **Temporal Projection → Aggregation**: Gradient melalui mean operation
4. **Aggregation → Multi-Scale Mixing**: Gradient ke season & trend components
5. **Mixing → Embedding**: Gradient ke value & time embeddings
6. **Embedding → Weights**: Gradient untuk update W_value & W_time

**Formula Umum:**
```
∂L/∂layer_input = ∂L/∂layer_output × ∂layer_output/∂layer_input
```

**Learning Rate:**
Untuk contoh ini, kita gunakan `learning_rate = 0.01`

---

## 10.1 Setup: Learning Rate & Initial Gradients

Sebelum memulai backpropagation, kita perlu setup learning rate dan verify gradient dari Sheet 9.

In [59]:
# ============================================
# SHEET 10: BACKPROPAGATION SETUP
# ============================================

print("="*100)
print("SHEET 10: BACKPROPAGATION THROUGH TIMEMIXER LAYERS")
print("="*100)

# Setup learning rate
learning_rate = 0.01

print(f"\n📋 BACKPROPAGATION SETUP:")
print("-" * 100)
print(f"  Learning rate: {learning_rate}")
print(f"  Loss function: MSE = {total_mse:.6f}")

print(f"\n📊 INITIAL GRADIENTS (from Sheet 9):")
print(f"  ∂L/∂prediction shape: {gradients.shape}")
print(f"  ∂L/∂prediction (first 2 samples):")
print(f"    t=0, F0: {gradients[0, 0]:.6f}")
print(f"    t=0, F1: {gradients[0, 1]:.6f}")

print(f"\n🔄 BACKPROPAGATION FLOW:")
print("-" * 100)
print("  Layer 8 (Output)     : prediction_denorm → ∂L/∂prediction ✅")
print("  Layer 7 (Denorm)     : prediction → ∂L/∂prediction_normalized")
print("  Layer 6 (Projection) : aggregated → ∂L/∂aggregated")
print("  Layer 5 (Aggregation): season + trend → ∂L/∂season, ∂L/∂trend")
print("  Layer 4 (Mixing)     : embeddings → ∂L/∂embedding")
print("  Layer 3 (Embedding)  : X, X_mark → ∂L/∂X, ∂L/∂X_mark")
print("  Layer 2 (Weights)    : W_value, W_time → ∂L/∂W_value, ∂L/∂W_time")

print("\n✅ Setup complete - ready for backpropagation")

SHEET 10: BACKPROPAGATION THROUGH TIMEMIXER LAYERS

📋 BACKPROPAGATION SETUP:
----------------------------------------------------------------------------------------------------
  Learning rate: 0.01
  Loss function: MSE = 64.475129

📊 INITIAL GRADIENTS (from Sheet 9):
  ∂L/∂prediction shape: (4, 2)
  ∂L/∂prediction (first 2 samples):
    t=0, F0: -2.508598
    t=0, F1: -1.194563

🔄 BACKPROPAGATION FLOW:
----------------------------------------------------------------------------------------------------
  Layer 8 (Output)     : prediction_denorm → ∂L/∂prediction ✅
  Layer 7 (Denorm)     : prediction → ∂L/∂prediction_normalized
  Layer 6 (Projection) : aggregated → ∂L/∂aggregated
  Layer 5 (Aggregation): season + trend → ∂L/∂season, ∂L/∂trend
  Layer 4 (Mixing)     : embeddings → ∂L/∂embedding
  Layer 3 (Embedding)  : X, X_mark → ∂L/∂X, ∂L/∂X_mark
  Layer 2 (Weights)    : W_value, W_time → ∂L/∂W_value, ∂L/∂W_time

✅ Setup complete - ready for backpropagation


## 10.2 Backprop Layer 1: Denormalization → Prediction (Normalized)

**Forward pass (Sheet 8 Step 5):**
```python
prediction_denorm[:, :, :2] = prediction[:, :, :2] * train_std + train_mean
```

**Backward pass:**
```
∂L/∂prediction_normalized = ∂L/∂prediction_denorm × train_std
```

**Mengapa × train_std?**
Karena forward: `out = in × std + mean`, maka backward: `∂L/∂in = ∂L/∂out × std`

In [60]:
# ============================================
# BACKPROP STEP 1: DENORMALIZATION
# ============================================

print("\n📋 BACKPROP STEP 1: Denormalization → Prediction (Normalized)")
print("="*100)

# Gradient through denormalization
# Forward: prediction_denorm = prediction * std + mean
# Backward: grad_prediction = grad_prediction_denorm * std

grad_prediction_norm = gradients * train_std.reshape(1, -1)  # Shape: (4, 2)

print(f"\nFormula:")
print(f"  ∂L/∂prediction_norm = ∂L/∂prediction_denorm × train_std")

print(f"\n📊 GRADIENT CALCULATION:")
print("-" * 120)
print(f"{'t':<4} {'Feature':<8} {'grad_denorm':<16} {'train_std':<14} {'grad_norm':<16} {'Formula':<40}")
print("-" * 120)

for t in range(pred_len):
    for f in range(2):
        grad_denorm = gradients[t, f]
        std = train_std[f]
        grad_norm = grad_prediction_norm[t, f]
        formula = f'{grad_denorm:.6f} × {std:.6f}'
        feature_name = 'OT' if f == 0 else 'HUFL'
        
        print(f"{t:<4} {feature_name:<8} {grad_denorm:<16.6f} {std:<14.6f} {grad_norm:<16.6f} {formula:<40}")

print(f"\n📊 MANUAL VERIFICATION (t=0, F0):")
print(f"  grad_denorm[0,0] = {gradients[0, 0]:.6f}")
print(f"  train_std[0] = {train_std[0]:.6f}")
print(f"  grad_norm[0,0] = {gradients[0, 0]:.6f} × {train_std[0]:.6f} = {grad_prediction_norm[0, 0]:.6f}")
print(f"  ✅ Verified: {np.isclose(grad_prediction_norm[0, 0], gradients[0, 0] * train_std[0])}")

print(f"\n✅ Gradient shape after denormalization: {grad_prediction_norm.shape}")
print(f"   Next: Backprop through temporal projection layer")


📋 BACKPROP STEP 1: Denormalization → Prediction (Normalized)

Formula:
  ∂L/∂prediction_norm = ∂L/∂prediction_denorm × train_std

📊 GRADIENT CALCULATION:
------------------------------------------------------------------------------------------------------------------------
t    Feature  grad_denorm      train_std      grad_norm        Formula                                 
------------------------------------------------------------------------------------------------------------------------
0    OT       -2.508598        4.590122       -11.514771       -2.508598 × 4.590122                    
0    HUFL     -1.194563        3.402193       -4.064134        -1.194563 × 3.402193                    
1    OT       -2.331858        4.590122       -10.703513       -2.331858 × 4.590122                    
1    HUFL     0.471129         3.402193       1.602872         0.471129 × 3.402193                     
2    OT       -2.701707        4.590122       -12.401166       -2.701707 × 4.590122

## 10.3 Backprop Layer 2: Temporal Projection → Aggregation

**Forward pass (Sheet 8 Step 4):**
```python
prediction = avg_pool(aggregated)  # (1,8,4) → (1,4,4) via average pooling
```

**Backward pass:**
Average pooling backward: gradient didistribusikan ke semua inputs yang di-average.

```
∂L/∂aggregated[i] = ∂L/∂prediction[i//2] / 2
```

Karena setiap output adalah average dari 2 inputs.

In [61]:
# ============================================
# 10.3 BACKPROP LAYER 2: TEMPORAL PROJECTION
# ============================================

print("="*80)
print("BACKPROP LAYER 2: Temporal Projection → Aggregation")
print("="*80)

print(f"\n🔹 Forward pass recap (Sheet 8 Step 4):")
print(f"   aggregated shape: {aggregated.shape}")  # (1, 8, 4)
print(f"   AvgPool2d(kernel_size=2, stride=2)")
print(f"   prediction shape: {prediction.shape}")  # (1, 4, 4)

print(f"\n🔹 Backward pass:")
print(f"   grad_prediction_norm shape: {grad_prediction_norm.shape}")  # (4, 2)
print(f"   Need to upsample to match aggregated shape (1, 8, 4)")

# Upsampling gradient: distribusikan gradient ke semua inputs yang di-average
# For each output at position i, it comes from averaging inputs at positions 2i and 2i+1
# So gradient flows equally to both inputs

# Initialize gradient for aggregated (before selecting features)
# aggregated shape: (1, 8, 4) - 8 timesteps, 4 features (F0, F1, F0_trend, F1_trend)
grad_aggregated = np.zeros((1, 8, 4))

# Gradients only exist for F0 and F1 (columns 0 and 1)
# Distribute gradients from downsampled (4 timesteps) to upsampled (8 timesteps)
for t in range(pred_len):  # 0,1,2,3
    for f in range(2):  # F0, F1
        # Each prediction timestep comes from averaging 2 aggregated timesteps
        grad_value = grad_prediction_norm[t, f] / 2.0
        # Distribute equally to the 2 source timesteps
        grad_aggregated[0, 2*t, f] = grad_value
        grad_aggregated[0, 2*t + 1, f] = grad_value

print(f"\n📊 Gradient Flow Through Temporal Projection:")
print(f"\n{'t_pred':<8} {'Feature':<10} {'grad_norm':<15} → {'t_agg':<8} {'grad_agg':<15} {'Formula':<30}")
print("-" * 100)

for t in range(pred_len):
    for f, fname in enumerate(['F0', 'F1']):
        t_agg_1 = 2 * t
        t_agg_2 = 2 * t + 1
        grad_norm_val = grad_prediction_norm[t, f]
        grad_agg_val = grad_aggregated[0, t_agg_1, f]
        
        formula = f"{grad_norm_val:.6f} / 2"
        print(f"{t:<8} {fname:<10} {grad_norm_val:<15.6f} → {t_agg_1:<8} {grad_agg_val:<15.6f} {formula:<30}")
        print(f"{'':<8} {'':<10} {'':<15}   {t_agg_2:<8} {grad_agg_val:<15.6f} {formula:<30}")

print(f"\n🔍 Manual Verification (t_pred=0, F0):")
print(f"   grad_prediction_norm[0,0] = {grad_prediction_norm[0,0]:.6f}")
print(f"   grad_aggregated[0,0,0] = {grad_aggregated[0,0,0]:.6f}")
print(f"   grad_aggregated[0,1,0] = {grad_aggregated[0,1,0]:.6f}")
print(f"   Sum = {grad_aggregated[0,0,0] + grad_aggregated[0,1,0]:.6f}")
print(f"   ✅ Verification: {grad_prediction_norm[0,0]:.6f} = {grad_aggregated[0,0,0] + grad_aggregated[0,1,0]:.6f}")

print(f"\n✅ Ready for next layer: Aggregation backprop")

BACKPROP LAYER 2: Temporal Projection → Aggregation

🔹 Forward pass recap (Sheet 8 Step 4):
   aggregated shape: (1, 8, 4)
   AvgPool2d(kernel_size=2, stride=2)
   prediction shape: (1, 4, 4)

🔹 Backward pass:
   grad_prediction_norm shape: (4, 2)
   Need to upsample to match aggregated shape (1, 8, 4)

📊 Gradient Flow Through Temporal Projection:

t_pred   Feature    grad_norm       → t_agg    grad_agg        Formula                       
----------------------------------------------------------------------------------------------------
0        F0         -11.514771      → 0        -5.757385       -11.514771 / 2                
                                      1        -5.757385       -11.514771 / 2                
0        F1         -4.064134       → 0        -2.032067       -4.064134 / 2                 
                                      1        -2.032067       -4.064134 / 2                 
1        F0         -10.703513      → 2        -5.351756       -10.703513 / 2 

## 10.4 Backprop Layer 3: Aggregation → Season + Trend

**Forward pass (Sheet 8 Step 3):**
```python
aggregated = (season_output + trend_output) / 2
```

**Backward pass:**
Karena aggregation adalah average dari season dan trend, gradient didistribusikan equally:

```
∂L/∂season_output = ∂L/∂aggregated / 2
∂L/∂trend_output = ∂L/∂aggregated / 2
```

In [62]:
# ============================================
# 10.4 BACKPROP LAYER 3: AGGREGATION
# ============================================

print("="*80)
print("BACKPROP LAYER 3: Aggregation → Season + Trend")
print("="*80)

print(f"\n🔹 Forward pass recap (Sheet 8 Step 3):")
print(f"   recon_scale0 shape: {recon_scale0.shape}")  # (1, 8, 4)
print(f"   recon_scale1_upsampled shape: {recon_scale1_upsampled.shape}")  # (1, 8, 4)
print(f"   aggregated = (recon_scale0 + recon_scale1_upsampled) / 2")

print(f"\n🔹 Backward pass:")
print(f"   grad_aggregated shape: {grad_aggregated.shape}")  # (1, 8, 4)
print(f"   Split equally to both scales")

# Gradients split equally
grad_recon_scale0 = grad_aggregated / 2.0
grad_recon_scale1_upsampled = grad_aggregated / 2.0

print(f"\n📊 Gradient Distribution (First 4 timesteps for F0 and F1):")
print(f"\n{'t':<6} {'Feature':<10} {'grad_agg':<15} → {'grad_scale0':<15} {'grad_scale1_up':<15} {'Formula':<20}")
print("-" * 95)

for t in range(4):  # Show first 4 timesteps
    for f, fname in enumerate(['F0', 'F1']):
        grad_agg_val = grad_aggregated[0, t, f]
        grad_s0_val = grad_recon_scale0[0, t, f]
        grad_s1_val = grad_recon_scale1_upsampled[0, t, f]
        formula = f"{grad_agg_val:.6f} / 2"
        
        print(f"{t:<6} {fname:<10} {grad_agg_val:<15.6f} → {grad_s0_val:<15.6f} {grad_s1_val:<15.6f} {formula:<20}")

print(f"\n🔍 Manual Verification (t=0, F0):")
print(f"   grad_aggregated[0,0,0] = {grad_aggregated[0,0,0]:.6f}")
print(f"   grad_recon_scale0[0,0,0] = {grad_recon_scale0[0,0,0]:.6f}")
print(f"   grad_recon_scale1_upsampled[0,0,0] = {grad_recon_scale1_upsampled[0,0,0]:.6f}")
print(f"   Sum = {grad_recon_scale0[0,0,0] + grad_recon_scale1_upsampled[0,0,0]:.6f}")
print(f"   ✅ Verification: {grad_aggregated[0,0,0]:.6f} = {grad_recon_scale0[0,0,0] + grad_recon_scale1_upsampled[0,0,0]:.6f}")

print(f"\n📊 Summary:")
print(f"   grad_recon_scale0 shape: {grad_recon_scale0.shape}")
print(f"   grad_recon_scale1_upsampled shape: {grad_recon_scale1_upsampled.shape}")
print(f"   Non-zero features: F0, F1 (columns 0, 1)")
print(f"   Zero features: F0_trend, F1_trend (columns 2, 3)")

print(f"\n✅ Ready for next layer: Multi-Scale Mixing backprop")

BACKPROP LAYER 3: Aggregation → Season + Trend

🔹 Forward pass recap (Sheet 8 Step 3):
   recon_scale0 shape: (1, 8, 4)
   recon_scale1_upsampled shape: (1, 8, 4)
   aggregated = (recon_scale0 + recon_scale1_upsampled) / 2

🔹 Backward pass:
   grad_aggregated shape: (1, 8, 4)
   Split equally to both scales

📊 Gradient Distribution (First 4 timesteps for F0 and F1):

t      Feature    grad_agg        → grad_scale0     grad_scale1_up  Formula             
-----------------------------------------------------------------------------------------------
0      F0         -5.757385       → -2.878693       -2.878693       -5.757385 / 2       
0      F1         -2.032067       → -1.016034       -1.016034       -2.032067 / 2       
1      F0         -5.757385       → -2.878693       -2.878693       -5.757385 / 2       
1      F1         -2.032067       → -1.016034       -1.016034       -2.032067 / 2       
2      F0         -5.351756       → -2.675878       -2.675878       -5.351756 / 2       


## 10.5 Backprop Layer 4: Multi-Scale Mixing → Embeddings

**Forward pass (Sheets 6-7):**
- Scale 1 (2×downsample): embedding → season_1, trend_1 → upsample → season_output
- Scale 2 (4×downsample): embedding → season_2, trend_2 → upsample → trend_output

**Backward pass:**
Gradient flows melalui upsampling dan downsampling operations untuk setiap scale:

```
grad_season_1 = upsample_gradient(grad_season)
grad_trend_2 = upsample_gradient(grad_trend)
```

In [63]:
# ============================================
# 10.5 BACKPROP LAYER 4: MULTI-SCALE MIXING
# ============================================

print("="*80)
print("BACKPROP LAYER 4: Multi-Scale Mixing → Embeddings")
print("="*80)

print(f"\n🔹 Forward pass recap (Sheets 6-7):")
print(f"   Scale 0: embedding → season/trend → recon_scale0")
print(f"   Scale 1: embedding → downsample(2x) → season/trend → upsample(2x) → recon_scale1_upsampled")

print(f"\n🔹 Backward pass:")
print(f"   Need to backprop through upsampling for scale 1")
print(f"   Need to backprop to embeddings for both scales")

# Backprop through upsampling for scale 1
# Forward: recon_scale1 (1,4,4) → upsample 2x → recon_scale1_upsampled (1,8,4)
# Backward: grad_recon_scale1_upsampled (1,8,4) → downsample → grad_recon_scale1 (1,4,4)

# Upsampling backward = sum pooling (aggregate gradients from upsampled positions)
grad_recon_scale1 = np.zeros((1, 4, 4))
for t in range(4):
    for f in range(4):
        # Each recon_scale1[t,f] was copied to positions 2t and 2t+1 in recon_scale1_upsampled
        grad_recon_scale1[0, t, f] = grad_recon_scale1_upsampled[0, 2*t, f] + grad_recon_scale1_upsampled[0, 2*t+1, f]

print(f"\n📊 Scale 1 Gradient - Upsampling Backward:")
print(f"\n{'t_s1':<8} {'Feature':<10} {'t_up':<10} {'grad_s1_up':<15} → {'grad_s1':<15} {'Formula':<30}")
print("-" * 100)

for t in range(4):
    for f, fname in enumerate(['F0', 'F1', 'F0_t', 'F1_t']):
        t_up_1 = 2*t
        t_up_2 = 2*t + 1
        grad_up_1 = grad_recon_scale1_upsampled[0, t_up_1, f]
        grad_up_2 = grad_recon_scale1_upsampled[0, t_up_2, f]
        grad_s1_val = grad_recon_scale1[0, t, f]
        formula = f"{grad_up_1:.6f}+{grad_up_2:.6f}"
        
        print(f"{t:<8} {fname:<10} {t_up_1},{t_up_2:<8} sum={grad_up_1+grad_up_2:<12.6f} → {grad_s1_val:<15.6f} {formula:<30}")

print(f"\n📊 Scale 0 Gradient:")
print(f"   grad_recon_scale0 shape: {grad_recon_scale0.shape}")
print(f"   (No upsampling needed - already at target timesteps)")

print(f"\n🔍 Manual Verification (t=0, F0):")
print(f"   Scale 1:")
print(f"     grad_recon_scale1_upsampled[0,0,0] + grad_recon_scale1_upsampled[0,1,0]")
print(f"     = {grad_recon_scale1_upsampled[0,0,0]:.6f} + {grad_recon_scale1_upsampled[0,1,0]:.6f}")
print(f"     = {grad_recon_scale1[0,0,0]:.6f}")

print(f"\n📊 Summary:")
print(f"   grad_recon_scale0 shape: {grad_recon_scale0.shape}  (8 timesteps)")
print(f"   grad_recon_scale1 shape: {grad_recon_scale1.shape}  (4 timesteps)")

print(f"\n💡 Next steps:")
print(f"   Need to backprop through season/trend decomposition")
print(f"   Then through mixing operations")
print(f"   Finally to embedding layer")

print(f"\n✅ Ready for next layer: Embedding backprop")

BACKPROP LAYER 4: Multi-Scale Mixing → Embeddings

🔹 Forward pass recap (Sheets 6-7):
   Scale 0: embedding → season/trend → recon_scale0
   Scale 1: embedding → downsample(2x) → season/trend → upsample(2x) → recon_scale1_upsampled

🔹 Backward pass:
   Need to backprop through upsampling for scale 1
   Need to backprop to embeddings for both scales

📊 Scale 1 Gradient - Upsampling Backward:

t_s1     Feature    t_up       grad_s1_up      → grad_s1         Formula                       
----------------------------------------------------------------------------------------------------
0        F0         0,1        sum=-5.757385    → -5.757385       -2.878693+-2.878693           
0        F1         0,1        sum=-2.032067    → -2.032067       -1.016034+-1.016034           
0        F0_t       0,1        sum=0.000000     → 0.000000        0.000000+0.000000             
0        F1_t       0,1        sum=0.000000     → 0.000000        0.000000+0.000000             
1        F0         

## 10.6 Backprop Layer 5: Embedding → Input

**Forward pass (Sheet 3):**
```python
enc_out = value_embedding(X) + temporal_embedding(X_mark)
```

**Backward pass:**
Gradient flows melalui kedua embedding paths:

```
∂L/∂W_value = ∂L/∂enc_out × X^T
∂L/∂W_time = ∂L/∂enc_out × X_mark^T
```

Ini adalah gradient untuk weight matrices yang akan digunakan untuk update.

In [64]:
# ============================================
# 10.6 BACKPROP LAYER 5: EMBEDDING (FULL VERSION)
# ============================================

print("="*80)
print("BACKPROP LAYER 5: Embedding → Input (COMPLETE BACKPROP)")
print("="*80)

print(f"\n🔹 Forward pass recap (Sheet 3):")
print(f"   enc_out = value_embedding(batch_x) + temporal_embedding(batch_x_mark)")
print(f"   batch_x shape: {batch_x.shape}")  # (1, 8, 2)
print(f"   batch_x_mark shape: {batch_x_mark.shape}")  # (1, 8, 2)
print(f"   W_value shape: {W_value.shape}")  # (2, 4)
print(f"   W_time shape: {W_time.shape}")  # (2, 4)

print(f"\n🔹 Complete backward pass through all layers:")
print(f"   1. Backprop through reconstruction (season + trend)")
print(f"   2. Backprop through season/trend decomposition")
print(f"   3. Backprop through mixing to embeddings")
print(f"   4. Compute weight gradients")

print("\n" + "="*80)
print("STEP 1: Backprop through Reconstruction")
print("="*80)

print(f"\n💡 Input Gradients (dari Layer sebelumnya):")
print(f"   grad_recon_scale0: berasal dari Section 10.4 (Aggregation Backprop)")
print(f"   grad_recon_scale1: berasal dari Section 10.5 (Multi-Scale Mixing Backprop)")
print(f"   ")
print(f"   Flow gradient sejauh ini:")
print(f"   Loss → Denorm → Temporal Projection → Aggregation → Multi-Scale Mixing")
print(f"                                           ↓                    ↓")
print(f"                                   grad_recon_scale0    grad_recon_scale1")
print(f"")
print(f"   grad_recon_scale0 shape: {grad_recon_scale0.shape}  (dari Section 10.4)")
print(f"   grad_recon_scale1 shape: {grad_recon_scale1.shape}  (dari Section 10.5)")

# From forward pass (Sheet 8): recon = season + trend
# Backward: grad flows equally to both components
# grad_season = grad_recon
# grad_trend = grad_recon

grad_season_scale0 = grad_recon_scale0.copy()
grad_trend_scale0 = grad_recon_scale0.copy()

grad_season_scale1 = grad_recon_scale1.copy()
grad_trend_scale1 = grad_recon_scale1.copy()

print(f"\n📊 Scale 0 - Reconstruction Backprop:")
print(f"   Forward (Sheet 8): recon_scale0 = season + trend")
print(f"   Backward: ∂L/∂season = ∂L/∂recon, ∂L/∂trend = ∂L/∂recon")
print(f"   grad_season_scale0 shape: {grad_season_scale0.shape}")
print(f"   grad_trend_scale0 shape: {grad_trend_scale0.shape}")

# Detail table for Scale 0 (first 4 timesteps, first 2 features)
print(f"\n📋 Detailed Gradient Table - Scale 0 (First 4 timesteps):")
print(f"\n{'t':<6} {'Feat':<6} {'grad_recon':<15} → {'grad_season':<15} {'grad_trend':<15}")
print("-" * 70)
for t in range(4):
    for f, fname in enumerate(['F0', 'F1']):
        g_recon = grad_recon_scale0[0, t, f]
        g_season = grad_season_scale0[0, t, f]
        g_trend = grad_trend_scale0[0, t, f]
        print(f"{t:<6} {fname:<6} {g_recon:<15.6f} → {g_season:<15.6f} {g_trend:<15.6f}")

print(f"\n📊 Scale 1 - Reconstruction Backprop:")
print(f"   grad_season_scale1 shape: {grad_season_scale1.shape}")
print(f"   grad_trend_scale1 shape: {grad_trend_scale1.shape}")

# Detail table for Scale 1 (all 4 timesteps, first 2 features)
print(f"\n📋 Detailed Gradient Table - Scale 1 (All 4 timesteps):")
print(f"\n{'t':<6} {'Feat':<6} {'grad_recon':<15} → {'grad_season':<15} {'grad_trend':<15}")
print("-" * 70)
for t in range(4):
    for f, fname in enumerate(['F0', 'F1']):
        g_recon = grad_recon_scale1[0, t, f]
        g_season = grad_season_scale1[0, t, f]
        g_trend = grad_trend_scale1[0, t, f]
        print(f"{t:<6} {fname:<6} {g_recon:<15.6f} → {g_season:<15.6f} {g_trend:<15.6f}")

print(f"\n🔍 Verification (t=0, F0):")
print(f"   grad_recon_scale0[0,0,0] = {grad_recon_scale0[0,0,0]:.6f}")
print(f"   → grad_season_scale0[0,0,0] = {grad_season_scale0[0,0,0]:.6f}")
print(f"   → grad_trend_scale0[0,0,0] = {grad_trend_scale0[0,0,0]:.6f}")

print("\n" + "="*80)
print("STEP 2: Backprop through Season/Trend Decomposition")
print("="*80)

# Forward pass (Sheet 6-7):
# season_list = [seasonal_scale0, seasonal_scale1]
# trend_list = [trend_scale0, trend_scale1]
# 
# For Scale 0: seasonal = embedding_scale0 - trend_scale0 (approximate)
# For Scale 1: seasonal = embedding_scale1_down - trend_scale1
# 
# trend = moving_average(embedding)

# Simplified approach: gradients flow back to embeddings
# grad_embedding = grad_season + grad_trend (both components come from same embedding)

print(f"\n💡 Season/Trend Decomposition Backprop:")
print(f"   Forward:")
print(f"     season = embedding - moving_avg(embedding)")
print(f"     trend = moving_avg(embedding)")
print(f"   ")
print(f"   Backward:")
print(f"     ∂L/∂embedding = ∂L/∂season × ∂season/∂embedding + ∂L/∂trend × ∂trend/∂embedding")
print(f"   ")
print(f"   Simplified for tutorial:")
print(f"     Both season and trend derived from same embedding")
print(f"     → grad_embedding = grad_season + grad_trend")

grad_embedding_scale0_from_decomp = grad_season_scale0 + grad_trend_scale0
grad_embedding_scale1_from_decomp = grad_season_scale1 + grad_trend_scale1

print(f"\n📊 Combined Gradients after Decomposition:")
print(f"   Scale 0: grad_embedding shape {grad_embedding_scale0_from_decomp.shape}")
print(f"   Scale 1: grad_embedding shape {grad_embedding_scale1_from_decomp.shape}")

# Detail table for Scale 0 decomposition
print(f"\n📋 Detailed Decomposition Gradient - Scale 0 (First 4 timesteps):")
print(f"\n{'t':<6} {'Feat':<6} {'grad_season':<15} {'grad_trend':<15} {'grad_embedding':<15}")
print("-" * 75)
for t in range(4):
    for f, fname in enumerate(['F0', 'F1']):
        g_season = grad_season_scale0[0, t, f]
        g_trend = grad_trend_scale0[0, t, f]
        g_emb = grad_embedding_scale0_from_decomp[0, t, f]
        print(f"{t:<6} {fname:<6} {g_season:<15.6f} {g_trend:<15.6f} {g_emb:<15.6f}")

# Detail table for Scale 1 decomposition
print(f"\n📋 Detailed Decomposition Gradient - Scale 1 (All 4 timesteps):")
print(f"\n{'t':<6} {'Feat':<6} {'grad_season':<15} {'grad_trend':<15} {'grad_embedding':<15}")
print("-" * 75)
for t in range(4):
    for f, fname in enumerate(['F0', 'F1']):
        g_season = grad_season_scale1[0, t, f]
        g_trend = grad_trend_scale1[0, t, f]
        g_emb = grad_embedding_scale1_from_decomp[0, t, f]
        print(f"{t:<6} {fname:<6} {g_season:<15.6f} {g_trend:<15.6f} {g_emb:<15.6f}")

print(f"\n🔍 Verification (t=0, F0):")
print(f"   grad_season_scale0[0,0,0] = {grad_season_scale0[0,0,0]:.6f}")
print(f"   grad_trend_scale0[0,0,0] = {grad_trend_scale0[0,0,0]:.6f}")
print(f"   Sum = {grad_embedding_scale0_from_decomp[0,0,0]:.6f}")

print("\n" + "="*80)
print("STEP 3: Backprop through Mixing to Original Embeddings")
print("="*80)

# Forward (Sheet 7):
# season_list_mixed[0] = season_list[0] (Scale 0, no change)
# season_list_mixed[1] = upsample_2x(season_list[1]) (Scale 1 upsampled to 8 timesteps)
# trend_list_mixed[0] = trend_list[0] (Scale 0, no change)
# trend_list_mixed[1] = downsample_2x(trend_list[1]) (Scale 1 downsampled to 4 timesteps)
# 
# Backward:
# For Scale 0: gradient passes directly (no mixing operation)
# For Scale 1: Already in 4 timesteps form (from decomposition), passes directly

print(f"\n💡 Mixing Layer Backprop:")
print(f"   Scale 0: No mixing → gradient passes directly")
print(f"   Scale 1: Already at correct scale (4 timesteps) → passes directly")
print(f"   Note: Scale 1 gradients already accumulated through reconstruction backprop")

# Scale 0: Direct passthrough (mixing doesn't change it)
grad_embedding_scale0_after_mixing = grad_embedding_scale0_from_decomp.copy()

# Scale 1: Already at correct scale (4 timesteps) from decomposition
# No additional downsampling needed - grad_embedding_scale1_from_decomp is already (1,4,4)
grad_embedding_scale1_after_mixing = grad_embedding_scale1_from_decomp.copy()

print(f"\n📊 After Mixing Backprop:")
print(f"   Scale 0 gradient: {grad_embedding_scale0_after_mixing.shape} (8 timesteps)")
print(f"   Scale 1 gradient: {grad_embedding_scale1_after_mixing.shape} (4 timesteps)")

# Detail table for mixing results
print(f"\n📋 Detailed Mixing Backprop - Scale 0 (First 4 timesteps):")
print(f"\n{'t':<6} {'Feat':<6} {'Before Mixing':<17} {'After Mixing':<17} {'Change':<10}")
print("-" * 70)
for t in range(4):
    for f, fname in enumerate(['F0', 'F1']):
        before = grad_embedding_scale0_from_decomp[0, t, f]
        after = grad_embedding_scale0_after_mixing[0, t, f]
        change = 'Same' if np.isclose(before, after) else 'Changed'
        print(f"{t:<6} {fname:<6} {before:<17.6f} {after:<17.6f} {change:<10}")

print(f"\n📋 Detailed Mixing Backprop - Scale 1 (All 4 timesteps):")
print(f"\n{'t':<6} {'Feat':<6} {'Before Mixing':<17} {'After Mixing':<17} {'Change':<10}")
print("-" * 70)
for t in range(4):
    for f, fname in enumerate(['F0', 'F1']):
        before = grad_embedding_scale1_from_decomp[0, t, f]
        after = grad_embedding_scale1_after_mixing[0, t, f]
        change = 'Same' if np.isclose(before, after) else 'Changed'
        print(f"{t:<6} {fname:<6} {before:<17.6f} {after:<17.6f} {change:<10}")

print(f"\n🔍 Verification Scale 1 (t=0, F0):")
print(f"   grad_embedding_scale1_from_decomp[0,0,0] = {grad_embedding_scale1_from_decomp[0,0,0]:.6f}")
print(f"   grad_embedding_scale1_after_mixing[0,0,0] = {grad_embedding_scale1_after_mixing[0,0,0]:.6f}")
print(f"   ✅ Same value (direct passthrough)")

print("\n" + "="*80)
print("STEP 4: Backprop through Downsampling to Base Embeddings")
print("="*80)

# Forward (Sheet 4):
# embedding_scale0 = enc_out (no downsampling)
# embedding_scale1 = downsample_2x(enc_out)
# 
# Backward:
# Scale 0: Direct passthrough
# Scale 1: Upsample gradient (reverse of downsample)

print(f"\n💡 Downsampling Backprop:")
print(f"   Scale 0: No downsampling → gradient passes directly")
print(f"   Scale 1: Downsampled 2x → need to upsample gradient (distribute)")

# Scale 0: no change
grad_enc_out_from_scale0 = grad_embedding_scale0_after_mixing.copy()

# Scale 1: upsample gradient
# Forward downsample: took every 2nd timestep
# Backward: put gradient at every 2nd position, zero elsewhere
grad_enc_out_from_scale1 = np.zeros((1, 8, 4))
for t in range(4):
    grad_enc_out_from_scale1[0, 2*t, :] = grad_embedding_scale1_after_mixing[0, t, :]
    # Position 2*t+1 gets zero (was skipped in downsampling)

print(f"\n📊 Gradients from Each Scale:")
print(f"   From Scale 0: {grad_enc_out_from_scale0.shape}")
print(f"   From Scale 1: {grad_enc_out_from_scale1.shape}")

# Detail table for downsampling backprop
print(f"\n📋 Detailed Downsampling Backprop - Scale 1 (All 8 timesteps):")
print(f"\n{'t_enc':<8} {'Feat':<6} {'t_scale1':<10} {'grad_scale1':<17} {'grad_enc_s1':<17} {'Operation':<25}")
print("-" * 95)
for t in range(8):
    for f, fname in enumerate(['F0', 'F1']):
        g_enc_s1 = grad_enc_out_from_scale1[0, t, f]
        if t % 2 == 0:  # Even positions
            t_s1 = t // 2
            g_s1 = grad_embedding_scale1_after_mixing[0, t_s1, f]
            operation = f"Copy from t={t_s1}"
        else:  # Odd positions
            t_s1 = '-'
            g_s1 = 0.0
            operation = "Zero (skipped in downsample)"
        
        print(f"{t:<8} {fname:<6} {str(t_s1):<10} {g_s1:<17.6f} {g_enc_s1:<17.6f} {operation:<25}")
    if t == 3:  # Show first 4 timesteps for brevity
        print("   ... (remaining timesteps follow same pattern)")
        break

# Combine gradients from both scales
grad_enc_out = grad_enc_out_from_scale0 + grad_enc_out_from_scale1

print(f"\n📊 Combined Encoder Output Gradient:")
print(f"   grad_enc_out = grad_from_scale0 + grad_from_scale1")
print(f"   Shape: {grad_enc_out.shape}")

print(f"\n📊 Gradient Distribution (First 4 timesteps, all features):")
print(f"\n{'t':<6} {'Feature':<10} {'from_S0':<15} {'from_S1':<15} {'Combined':<15} {'Formula':<30}")
print("-" * 100)
for t in range(4):
    for f, fname in enumerate(['F0', 'F1', 'F0_t', 'F1_t']):
        g_s0 = grad_enc_out_from_scale0[0, t, f]
        g_s1 = grad_enc_out_from_scale1[0, t, f]
        g_combined = grad_enc_out[0, t, f]
        formula = f"{g_s0:.6f}+{g_s1:.6f}"
        print(f"{t:<6} {fname:<10} {g_s0:<15.6f} {g_s1:<15.6f} {g_combined:<15.6f} {formula:<30}")

print(f"\n🔍 Verification (t=0, F0):")
print(f"   grad_enc_out_from_scale0[0,0,0] = {grad_enc_out_from_scale0[0,0,0]:.6f}")
print(f"   grad_enc_out_from_scale1[0,0,0] = {grad_enc_out_from_scale1[0,0,0]:.6f}")
print(f"   Sum = {grad_enc_out[0,0,0]:.6f}")
print(f"   ✅ Combined gradient ready for weight computation")

# Show all 8 timesteps for F0 (feature 0)
print(f"\n📊 Complete grad_enc_out Values for F0 (All 8 Timesteps):")
print(f"   (Nilai-nilai ini akan digunakan di Step 5 untuk perhitungan grad_W_value dan grad_W_time)")
print(f"\n{'t':<6} {'from_Scale0':<17} {'from_Scale1':<17} {'Combined (F0)':<17} {'Sumber':<30}")
print("-" * 90)
for t in range(8):
    g_s0 = grad_enc_out_from_scale0[0, t, 0]
    g_s1 = grad_enc_out_from_scale1[0, t, 0]
    g_combined = grad_enc_out[0, t, 0]
    source = f"Scale0 + Scale1"
    print(f"{t:<6} {g_s0:<17.6f} {g_s1:<17.6f} {g_combined:<17.6f} {source:<30}")

# Show all 8 timesteps for F1 (feature 1)
print(f"\n📊 Complete grad_enc_out Values for F1 (All 8 Timesteps):")
print(f"   (Nilai-nilai ini digunakan untuk perhitungan grad_W_value[0,1], grad_W_value[1,1], dll.)")
print(f"\n{'t':<6} {'from_Scale0':<17} {'from_Scale1':<17} {'Combined (F1)':<17} {'Sumber':<30}")
print("-" * 90)
for t in range(8):
    g_s0 = grad_enc_out_from_scale0[0, t, 1]
    g_s1 = grad_enc_out_from_scale1[0, t, 1]
    g_combined = grad_enc_out[0, t, 1]
    source = f"Scale0 + Scale1"
    print(f"{t:<6} {g_s0:<17.6f} {g_s1:<17.6f} {g_combined:<17.6f} {source:<30}")
    
print(f"\n💡 Penjelasan nilai yang Anda tanyakan:")
print(f"   Nilai-nilai: -6.096201, -2.032067, 2.404308, 0.801436, 6.343826, 2.114609, 10.982761, 3.660920")
print(f"   adalah grad_enc_out[t,1] untuk t=0,1,2,3,4,5,6,7 (column 'Combined (F1)' di tabel atas)")
print(f"   ")
print(f"   Nilai-nilai ini berasal dari Step 4:")
print(f"     grad_enc_out[t,1] = grad_enc_out_from_scale0[t,1] + grad_enc_out_from_scale1[t,1]")
print(f"   ")
print(f"   Digunakan di Step 5 untuk perhitungan:")
print(f"     grad_W_value[0,1] = Σ(batch_x[t,0] × grad_enc_out[t,1]) untuk t=0..7")

print("\n" + "="*80)
print("STEP 5: Compute Weight Gradients for Embeddings")
print("="*80)

print(f"\n💡 Input untuk Step 5:")
print(f"   grad_enc_out: Berasal dari Step 4 (Downsampling Backprop)")
print(f"   Formula: grad_enc_out = grad_enc_out_from_scale0 + grad_enc_out_from_scale1")
print(f"   Shape: {grad_enc_out.shape}  → (1, 8 timesteps, 4 d_model features)")
print(f"")
print(f"   Arti grad_enc_out[t,j]:")
print(f"     - t: timestep index (0-7)")
print(f"     - j: feature index dalam d_model (0-3)")
print(f"       j=0: F0 (feature 0)")
print(f"       j=1: F1 (feature 1)")
print(f"       j=2: F0_temporal (temporal encoding untuk F0)")
print(f"       j=3: F1_temporal (temporal encoding untuk F1)")
print(f"")
print(f"   Contoh: grad_enc_out[0,0] = {grad_enc_out[0,0,0]:.6f}")
print(f"           (gradient untuk timestep 0, feature F0)")

print(f"\n💡 Konsep Backpropagation untuk Weight Gradients:")
print(f"   Forward pass (Sheet 3):")
print(f"     enc_out = W_value @ batch_x^T + W_time @ batch_x_mark^T + bias")
print(f"   ")
print(f"   Backward pass (Chain Rule):")
print(f"     ∂L/∂W_value = ∂L/∂enc_out × ∂enc_out/∂W_value")
print(f"                 = grad_enc_out × batch_x^T")
print(f"   ")
print(f"     ∂L/∂W_time = ∂L/∂enc_out × ∂enc_out/∂W_time")
print(f"                = grad_enc_out × batch_x_mark^T")
print(f"")
print(f"   Dalam matrix multiplication:")
print(f"     grad_W_value[i,j] = Σ_t batch_x[t,i] × grad_enc_out[t,j]")
print(f"     grad_W_time[i,j] = Σ_t batch_x_mark[t,i] × grad_enc_out[t,j]")

# Calculate weight gradients
# ∂L/∂W_value = batch_x^T @ (∂L/∂enc_out)
# ∂L/∂W_time = batch_x_mark^T @ (∂L/∂enc_out)

batch_x_reshaped = batch_x.reshape(8, 2)  # (8, 2)
grad_enc_out_reshaped = grad_enc_out.reshape(8, 4)  # (8, 4)

# Weight gradient: (2, 4) = (2, 8) × (8, 4)
grad_W_value = batch_x_reshaped.T @ grad_enc_out_reshaped  # (2, 4)

print(f"\n📊 Gradient for Value Embedding Weights:")
print(f"   Formula: grad_W_value = batch_x^T @ grad_enc_out")
print(f"   ")
print(f"   Matrix dimensions:")
print(f"     batch_x_reshaped: {batch_x_reshaped.shape}  (8 timesteps, 2 features)")
print(f"     batch_x_reshaped^T: (2, 8)")
print(f"     grad_enc_out_reshaped: {grad_enc_out_reshaped.shape}  (8 timesteps, 4 d_model)")
print(f"     grad_W_value: {grad_W_value.shape}  = (2, 8) @ (8, 4)")
print(f"\n   grad_W_value:")
print(f"   {grad_W_value}")

# Detail table for W_value gradient computation
print(f"\n📋 Detailed grad_W_value Computation (Element by Element):")
print(f"\n{'Position':<12} {'Expanded Formula':<80} {'Result':<15}")
print("-" * 110)
for i in range(2):
    for j in range(4):
        # grad_W_value[i,j] = sum over t: batch_x[t,i] * grad_enc_out[t,j]
        # Show first 3 terms explicitly
        term_values = [batch_x[0, t, i] * grad_enc_out[0, t, j] for t in range(8)]
        terms_str = [f"({batch_x[0, t, i]:.3f}×{grad_enc_out[0, t, j]:.3f})" for t in range(3)]
        formula = f"Σ_t x[t,{i}]×g[t,{j}] = {'+'.join(terms_str)}+..."
        result = grad_W_value[i, j]
        print(f"W_value[{i},{j}] {formula:<80} {result:<15.6f}")

print(f"\n💡 Contoh Perhitungan Manual grad_W_value[0,0]:")
print(f"   grad_W_value[0,0] = Σ(batch_x[t,0] × grad_enc_out[t,0]) untuk t=0..7")
print(f"")
for t in range(8):
    x_val = batch_x[0, t, 0]
    g_val = grad_enc_out[0, t, 0]
    product = x_val * g_val
    print(f"     t={t}: {x_val:8.6f} × {g_val:12.6f} = {product:12.6f}")
print(f"")
print(f"     Sum = {sum(batch_x[0, t, 0] * grad_enc_out[0, t, 0] for t in range(8)):.6f}")
print(f"     ✅ Matches grad_W_value[0,0] = {grad_W_value[0,0]:.6f}")

print(f"\n💡 Contoh Perhitungan Manual grad_W_value[0,1]:")
print(f"   grad_W_value[0,1] = Σ(batch_x[t,0] × grad_enc_out[t,1]) untuk t=0..7")
print(f"   (Menjawab: bagaimana perhitungan dari 3 term pertama hingga hasil akhir)")
print(f"")
for t in range(8):
    x_val = batch_x[0, t, 0]
    g_val = grad_enc_out[0, t, 1]  # Feature 1 (F1)
    product = x_val * g_val
    print(f"     t={t}: {x_val:8.6f} × {g_val:12.6f} = {product:12.6f}")
print(f"")
print(f"     Sum = {sum(batch_x[0, t, 0] * grad_enc_out[0, t, 1] for t in range(8)):.6f}")
print(f"     ✅ Matches grad_W_value[0,1] = {grad_W_value[0,1]:.6f}")
print(f"")
print(f"   📝 Penjelasan:")
print(f"      - Formula tabel menunjukkan 3 term pertama: (-1.506×-6.096)+(-1.136×-2.032)+(-0.624×2.404)+...")
print(f"      - '...' berarti ada 5 term lagi (t=3,4,5,6,7) yang perlu dijumlahkan")
print(f"      - Total 8 term di atas dijumlahkan = {grad_W_value[0,1]:.6f}")

# Similarly for temporal embedding
batch_x_mark_reshaped = batch_x_mark.reshape(8, 2)  # (8, 2)
grad_W_time = batch_x_mark_reshaped.T @ grad_enc_out_reshaped  # (2, 4)

print(f"\n📊 Gradient for Temporal Embedding Weights:")
print(f"   Formula: grad_W_time = batch_x_mark^T @ grad_enc_out")
print(f"   ")
print(f"   Matrix dimensions:")
print(f"     batch_x_mark_reshaped: {batch_x_mark_reshaped.shape}  (8 timesteps, 2 time features)")
print(f"     batch_x_mark_reshaped^T: (2, 8)")
print(f"     grad_enc_out_reshaped: {grad_enc_out_reshaped.shape}  (8 timesteps, 4 d_model)")
print(f"     grad_W_time: {grad_W_time.shape}  = (2, 8) @ (8, 4)")
print(f"\n   grad_W_time:")
print(f"   {grad_W_time}")

# Detail table for W_time gradient computation
print(f"\n📋 Detailed grad_W_time Computation (Element by Element):")
print(f"\n{'Position':<12} {'Expanded Formula':<80} {'Result':<15}")
print("-" * 110)
for i in range(2):
    for j in range(4):
        # grad_W_time[i,j] = sum over t: batch_x_mark[t,i] * grad_enc_out[t,j]
        # Show first 3 terms explicitly
        term_values = [batch_x_mark[0, t, i] * grad_enc_out[0, t, j] for t in range(8)]
        terms_str = [f"({batch_x_mark[0, t, i]:.3f}×{grad_enc_out[0, t, j]:.3f})" for t in range(3)]
        formula = f"Σ_t xm[t,{i}]×g[t,{j}] = {'+'.join(terms_str)}+..."
        result = grad_W_time[i, j]
        print(f"W_time[{i},{j}]  {formula:<80} {result:<15.6f}")

print(f"\n💡 Contoh Perhitungan Manual grad_W_time[0,0]:")
print(f"   grad_W_time[0,0] = Σ(batch_x_mark[t,0] × grad_enc_out[t,0]) untuk t=0..7")
print(f"")
for t in range(8):
    xm_val = batch_x_mark[0, t, 0]
    g_val = grad_enc_out[0, t, 0]
    product = xm_val * g_val
    print(f"     t={t}: {xm_val:8.6f} × {g_val:12.6f} = {product:12.6f}")
print(f"")
print(f"     Sum = {sum(batch_x_mark[0, t, 0] * grad_enc_out[0, t, 0] for t in range(8)):.6f}")
print(f"     ✅ Matches grad_W_time[0,0] = {grad_W_time[0,0]:.6f}")

print(f"\n" + "="*80)
print("RINGKASAN PERHITUNGAN WEIGHT GRADIENTS")
print("="*80)
print(f"""
Untuk grad_W_value:
  1. Input: batch_x shape (1,8,2) → reshape ke (8,2)
  2. Gradient: grad_enc_out shape (1,8,4) → reshape ke (8,4)
  3. Transpose batch_x: (8,2) → (2,8)
  4. Matrix multiply: (2,8) @ (8,4) = (2,4)
  5. Hasil: grad_W_value shape (2,4)
  
  Formula elemen:
    grad_W_value[i,j] = Σ(batch_x[t,i] × grad_enc_out[t,j]) untuk t=0..7
  
Untuk grad_W_time:
  1. Input: batch_x_mark shape (1,8,2) → reshape ke (8,2)
  2. Gradient: grad_enc_out shape (1,8,4) → reshape ke (8,4)
  3. Transpose batch_x_mark: (8,2) → (2,8)
  4. Matrix multiply: (2,8) @ (8,4) = (2,4)
  5. Hasil: grad_W_time shape (2,4)
  
  Formula elemen:
    grad_W_time[i,j] = Σ(batch_x_mark[t,i] × grad_enc_out[t,j]) untuk t=0..7

Kenapa menggunakan transpose?
  - Forward: enc_out = W @ input^T  (weight × input transpose)
  - Backward: grad_W = grad_output @ input  (gradient × input)
  - Equivalent: grad_W = input^T @ grad_output  (input transpose × gradient)
""")



print(f"\n" + "="*80)
print("COMPLETE BACKPROP PATH SUMMARY")
print("="*80)
print(f"""
Gradient Flow (Full Detail):
  
  Loss (MSE)
    ↓
  Denormalization (× std)
    ↓
  Temporal Projection (Upsample 4→8 timesteps)
    ↓
  Aggregation (÷2 to each scale)
    ↓
  Upsampling Backward (Sum pool for Scale 1)
    ↓
  Reconstruction (Split to season + trend)
    ↓
  Decomposition (Combine season + trend gradients)
    ↓
  Mixing (Downsample Scale 1 gradient)
    ↓
  Downsampling Backward (Upsample Scale 1, passthrough Scale 0)
    ↓
  Combine Scales (Add Scale 0 + Scale 1)
    ↓
  Encoder Output Gradient: shape {grad_enc_out.shape}
    ↓
  Weight Gradients:
    - grad_W_value: {grad_W_value.shape}
    - grad_W_time: {grad_W_time.shape}
""")

print(f"\n✅ Complete embedding backprop finished - Ready for weight updates")

BACKPROP LAYER 5: Embedding → Input (COMPLETE BACKPROP)

🔹 Forward pass recap (Sheet 3):
   enc_out = value_embedding(batch_x) + temporal_embedding(batch_x_mark)
   batch_x shape: (1, 8, 2)
   batch_x_mark shape: (1, 8, 2)
   W_value shape: (2, 4)
   W_time shape: (2, 4)

🔹 Complete backward pass through all layers:
   1. Backprop through reconstruction (season + trend)
   2. Backprop through season/trend decomposition
   3. Backprop through mixing to embeddings
   4. Compute weight gradients

STEP 1: Backprop through Reconstruction

💡 Input Gradients (dari Layer sebelumnya):
   grad_recon_scale0: berasal dari Section 10.4 (Aggregation Backprop)
   grad_recon_scale1: berasal dari Section 10.5 (Multi-Scale Mixing Backprop)
   
   Flow gradient sejauh ini:
   Loss → Denorm → Temporal Projection → Aggregation → Multi-Scale Mixing
                                           ↓                    ↓
                                   grad_recon_scale0    grad_recon_scale1

   grad_recon_scale0

## 10.7 Weight Update

**Update rule:**
```python
W_new = W_old - learning_rate × ∂L/∂W
```

Dengan learning_rate = 0.01, kita akan update:
- W_value: (2, 4) weight matrix untuk value embedding
- W_time: (4, 4) weight matrix untuk temporal embedding

Ini adalah langkah terakhir dari backpropagation - mengupdate weights untuk mengurangi loss di iterasi berikutnya.

In [65]:
# ============================================
# 10.7 WEIGHT UPDATE
# ============================================

print("="*80)
print("WEIGHT UPDATE")
print("="*80)

print(f"\n🔹 Update rule:")
print(f"   W_new = W_old - learning_rate × ∂L/∂W")
print(f"   learning_rate = {learning_rate}")

# Store old weights
W_value_old = W_value.copy()
W_time_old = W_time.copy()

# Update weights
W_value_new = W_value_old - learning_rate * grad_W_value
W_time_new = W_time_old - learning_rate * grad_W_time

print(f"\n📊 Value Embedding Weight Update:")
print(f"\n{'Position':<12} {'W_old':<15} {'grad':<15} {'learning_rate×grad':<20} {'W_new':<15}")
print("-" * 90)

for i in range(2):
    for j in range(4):
        w_old = W_value_old[i, j]
        grad = grad_W_value[i, j]
        delta = learning_rate * grad
        w_new = W_value_new[i, j]
        
        print(f"W_value[{i},{j}] {w_old:<15.6f} {grad:<15.6f} {delta:<20.6f} {w_new:<15.6f}")

print(f"\n📊 Temporal Embedding Weight Update:")
print(f"\n{'Position':<12} {'W_old':<15} {'grad':<15} {'learning_rate×grad':<20} {'W_new':<15}")
print("-" * 90)

for i in range(2):  # W_time has shape (2, 4)
    for j in range(4):
        w_old = W_time_old[i, j]
        grad = grad_W_time[i, j]
        delta = learning_rate * grad
        w_new = W_time_new[i, j]
        
        print(f"W_time[{i},{j}]  {w_old:<15.6f} {grad:<15.6f} {delta:<20.6f} {w_new:<15.6f}")

# Calculate weight changes
delta_W_value = W_value_new - W_value_old
delta_W_time = W_time_new - W_time_old

print(f"\n📊 Weight Change Magnitude:")
print(f"   ||ΔW_value|| = {np.linalg.norm(delta_W_value):.6f}")
print(f"   ||ΔW_time|| = {np.linalg.norm(delta_W_time):.6f}")
print(f"   max(|ΔW_value|) = {np.max(np.abs(delta_W_value)):.6f}")
print(f"   max(|ΔW_time|) = {np.max(np.abs(delta_W_time)):.6f}")

print(f"\n🔍 Manual Verification (W_value[0,0]):")
print(f"   W_value_old[0,0] = {W_value_old[0,0]:.6f}")
print(f"   grad_W_value[0,0] = {grad_W_value[0,0]:.6f}")
print(f"   delta = {learning_rate} × {grad_W_value[0,0]:.6f} = {learning_rate * grad_W_value[0,0]:.6f}")
print(f"   W_value_new[0,0] = {W_value_old[0,0]:.6f} - {learning_rate * grad_W_value[0,0]:.6f}")
print(f"   = {W_value_new[0,0]:.6f}")
print(f"   ✅ Verification: {W_value_new[0,0]:.6f}")

print(f"\n🔍 Manual Verification (W_time[0,0]) - IMPORTANT!")
print(f"   W_time_old[0,0] = {W_time_old[0,0]:.6f}")
print(f"   grad_W_time[0,0] = {grad_W_time[0,0]:.6f}")
print(f"   delta = {learning_rate} × {grad_W_time[0,0]:.6f} = {learning_rate * grad_W_time[0,0]:.6f}")
print(f"   W_time_new[0,0] = {W_time_old[0,0]:.6f} - ({learning_rate * grad_W_time[0,0]:.6f})")
print(f"   = {W_time_new[0,0]:.6f}")
print(f"   ✅ W_time SUDAH UPDATE! {W_time_old[0,0]:.6f} → {W_time_new[0,0]:.6f}")

print(f"\n✅ BOTH W_value AND W_time HAVE BEEN UPDATED!")
print(f"\n💡 Interpretation:")
print(f"   - Weights have been adjusted to reduce prediction error")
print(f"   - Direction: opposite to gradient (gradient descent)")
print(f"   - Magnitude: controlled by learning_rate = {learning_rate}")
print(f"   - Next iteration will use updated weights W_new")

WEIGHT UPDATE

🔹 Update rule:
   W_new = W_old - learning_rate × ∂L/∂W
   learning_rate = 0.01

📊 Value Embedding Weight Update:

Position     W_old           grad            learning_rate×grad   W_new          
------------------------------------------------------------------------------------------
W_value[0,0] 0.500000        9.404068        0.094041             0.405959       
W_value[0,1] 0.300000        30.488773       0.304888             -0.004888      
W_value[0,2] 0.200000        0.000000        0.000000             0.200000       
W_value[0,3] 0.100000        0.000000        0.000000             0.100000       
W_value[1,0] 0.100000        12.237073       0.122371             -0.022371      
W_value[1,1] 0.200000        -9.353866       -0.093539            0.293539       
W_value[1,2] 0.300000        0.000000        0.000000             0.300000       
W_value[1,3] 0.500000        0.000000        0.000000             0.500000       

📊 Temporal Embedding Weight Update:

Pos

## 10.8 Backpropagation Summary

**Complete Backpropagation Flow (Full Version):**

1. **Loss → Gradients** (Sheet 9)
   - MSE loss calculated from prediction errors
   - Initial gradients: ∂L/∂prediction_denorm

2. **Layer 1: Denormalization**
   - grad_prediction_norm = grad × train_std

3. **Layer 2: Temporal Projection**
   - grad_aggregated = upsample(grad_prediction_norm)

4. **Layer 3: Aggregation**
   - grad_recon_scale0 = grad_aggregated / 2
   - grad_recon_scale1 = grad_aggregated / 2

5. **Layer 4: Multi-Scale Mixing**
   - grad_recon_scale1 from upsampling backward (sum pooling)

6. **Layer 5: Embedding (Full Detail)**
   - **Step 1**: Reconstruction backprop (recon → season + trend)
   - **Step 2**: Decomposition backprop (season + trend → embedding)
   - **Step 3**: Mixing backprop (passthrough for both scales)
   - **Step 4**: Downsampling backprop (Scale 1 upsample, Scale 0 passthrough)
   - **Step 5**: Combine scales → grad_enc_out
   - **Step 6**: Weight gradients computation
     - grad_W_value = batch_x^T @ grad_enc_out
     - grad_W_time = batch_x_mark^T @ grad_enc_out

7. **Layer 6: Weight Update**
   - W_value_new = W_value_old - learning_rate × grad_W_value
   - W_time_new = W_time_old - learning_rate × grad_W_time

**Result:** Model weights updated untuk mengurangi prediction error di iterasi berikutnya!

In [66]:
# ============================================
# 10.8 BACKPROPAGATION SUMMARY
# ============================================

print("="*80)
print("BACKPROPAGATION SUMMARY")
print("="*80)

print(f"\n🎯 COMPLETE BACKPROPAGATION FLOW:\n")

print(f"{'Layer':<6} {'Operation':<45} {'Input Shape':<20} {'Output Shape':<20}")
print("-" * 100)
print(f"{'Loss':<6} {'MSE Calculation':<45} {'(4,2)':<20} {'scalar':<20}")
print(f"{'↓':<6} {'∂MSE/∂prediction':<45} {'':<20} {'(4,2)':<20}")
print(f"")
print(f"{'L1':<6} {'Denormalization Backprop':<45} {'(4,2)':<20} {'(4,2)':<20}")
print(f"{'↓':<6} {'grad × std':<45} {'':<20} {'':<20}")
print(f"")
print(f"{'L2':<6} {'Temporal Projection Backprop':<45} {'(4,2)':<20} {'(1,8,4)':<20}")
print(f"{'↓':<6} {'Upsample gradient':<45} {'':<20} {'':<20}")
print(f"")
print(f"{'L3':<6} {'Aggregation Backprop':<45} {'(1,8,4)':<20} {'2×(1,8,4)':<20}")
print(f"{'↓':<6} {'Split to scale0 and scale1':<45} {'':<20} {'':<20}")
print(f"")
print(f"{'L4':<6} {'Multi-Scale Mixing Backprop':<45} {'2×(1,8,4)':<20} {'(1,8,4)+(1,4,4)':<20}")
print(f"{'↓':<6} {'Upsampling backward (sum pool)':<45} {'':<20} {'':<20}")
print(f"")
print(f"{'L5':<6} {'Embedding Backprop (FULL - 6 sub-steps)':<45} {'(1,8,4)+(1,4,4)':<20} {'(1,8,4)':<20}")
print(f"{'↓':<6} {'  Step 1: Reconstruction → season+trend':<45} {'':<20} {'':<20}")
print(f"{'↓':<6} {'  Step 2: Decomposition → embedding':<45} {'':<20} {'':<20}")
print(f"{'↓':<6} {'  Step 3: Mixing (passthrough)':<45} {'':<20} {'':<20}")
print(f"{'↓':<6} {'  Step 4: Downsampling backward':<45} {'':<20} {'':<20}")
print(f"{'↓':<6} {'  Step 5: Combine scales':<45} {'':<20} {'':<20}")
print(f"{'↓':<6} {'  Step 6: grad_W = X^T @ grad_enc_out':<45} {'':<20} {'(2,4)+(2,4)':<20}")
print(f"")
print(f"{'L6':<6} {'Weight Update':<45} {'(2,4)+(2,4)':<20} {'W_new':<20}")
print(f"{'↓':<6} {'W - lr×grad':<45} {'':<20} {'':<20}")

print(f"\n📊 KEY GRADIENT VALUES:\n")
print(f"   Initial gradient (from MSE):")
print(f"     - Shape: {gradients.shape}")
print(f"     - Range: [{np.min(gradients):.6f}, {np.max(gradients):.6f}]")
print(f"")
print(f"   Intermediate gradients:")
print(f"     - grad_enc_out (encoder): {grad_enc_out.shape}, ||grad|| = {np.linalg.norm(grad_enc_out):.6f}")
print(f"     - grad_enc_out_from_scale0: ||grad|| = {np.linalg.norm(grad_enc_out_from_scale0):.6f}")
print(f"     - grad_enc_out_from_scale1: ||grad|| = {np.linalg.norm(grad_enc_out_from_scale1):.6f}")
print(f"")
print(f"   Final weight gradients:")
print(f"     - grad_W_value: {grad_W_value.shape}, ||grad|| = {np.linalg.norm(grad_W_value):.6f}")
print(f"     - grad_W_time: {grad_W_time.shape}, ||grad|| = {np.linalg.norm(grad_W_time):.6f}")

print(f"\n🔄 WEIGHT UPDATES:\n")
print(f"   Value Embedding:")
print(f"     - ||ΔW_value|| = {np.linalg.norm(delta_W_value):.6f}")
print(f"     - max change = {np.max(np.abs(delta_W_value)):.6f}")
print(f"")
print(f"   Temporal Embedding:")
print(f"     - ||ΔW_time|| = {np.linalg.norm(delta_W_time):.6f}")
print(f"     - max change = {np.max(np.abs(delta_W_time)):.6f}")

print(f"\n📈 TRAINING METRICS:\n")
print(f"   Total MSE: {total_mse:.6f}")
print(f"   Total MAE: {total_mae:.6f}")
print(f"   Learning Rate: {learning_rate}")

print(f"\n✅ BACKPROPAGATION COMPLETE!")
print(f"\n💡 What happens next:")
print(f"   1. Forward pass dengan W_new akan menghasilkan prediction yang lebih baik")
print(f"   2. Loss akan berkurang (MSE < {total_mse:.6f})")
print(f"   3. Proses ini diulang untuk banyak iterasi (epochs)")
print(f"   4. Model akan converge ke optimal weights")

print(f"\n" + "="*80)
print("END OF SHEET 10: BACKPROPAGATION")
print("="*80)

BACKPROPAGATION SUMMARY

🎯 COMPLETE BACKPROPAGATION FLOW:

Layer  Operation                                     Input Shape          Output Shape        
----------------------------------------------------------------------------------------------------
Loss   MSE Calculation                               (4,2)                scalar              
↓      ∂MSE/∂prediction                                                   (4,2)               

L1     Denormalization Backprop                      (4,2)                (4,2)               
↓      grad × std                                                                             

L2     Temporal Projection Backprop                  (4,2)                (1,8,4)             
↓      Upsample gradient                                                                      

L3     Aggregation Backprop                          (1,8,4)              2×(1,8,4)           
↓      Split to scale0 and scale1                                            